# ML-5 V10.2 — единый сквозной исследовательский ноутбук

**ОДИН файл, ОДИН запуск:** `Runtime → Restart session and run all` (Google Colab). Все этапы выполняются строго последовательно в текущем сеансе, без ручного запуска второй части и без подстановки старого GitHub Release.

**Шаги:** источники Stage I → предметные расчёты семи модулей → технический/научный QA → Excel, Parquet и полный Evidence → публикация в новую приватную ветку и GitHub Release → проверка опубликованных файлов → чтение научных Parquet именно из релиза текущего запуска → независимые повторные расчёты → научный архив для статьи и его скачивание.

Отсутствующие подтверждённые данные обозначаются `NOT_FEASIBLE`, не реконструируются. `SMOKE` служит для тестирования без научной публикации; для полного процесса задан `FULL`. Внешняя научная рецензия отдельно от внутреннего QA.

In [ ]:
SELF_BUNDLE_B64 = '...'


## 0. Единая конфигурация: сквозной запуск без ручных действий между этапами
Меняется только следующая ячейка. `FULL` использует все соответствующие записи без произвольного ограничения выборки; `SMOKE` — диагностика на реальных источниках, без публикации и без научного статуса. Источники читаются только из приватного GitHub с проверкой SHA-256. Публикация по умолчанию отключена.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class Config:
    MODE: str = 'FULL'                     # FULL | SMOKE
    REPO: str = 'AntonDozhdikov/Demograf_sovetnik'
    SOURCE_BRANCH: str = 'main'
    INDEX_PATH: str = 'handoff/07_v2/dataset_index.json'
    INDEX_BLOB_SHA: str = 'd6625ccddc24f24a1ee451e33c48b736d3a57043'
    COLAB_SECRET: str = 'TRIAD-RU-Colab'
    ROOT: str = '/content/ML5_V10_2_FULL_RESEARCH'
    CACHE: str = '/content/ML5_STAGE1_VERIFIED_CACHE'
    MAX_MEMORY_GB: float = 4.0
    WORKERS: int = 2
    SEED: int = 20261123
    TEST_YEARS: tuple = (2024, 2025)
    MAX_GRAPH_REGIONS: int = 32
    TOP_N: int = 30
    VERIFY_ALL_52_BYTES: bool = False       # все 52 проверены индексом; скачиваются только фактически используемые
    RUN_FORECAST: bool = True               # ограниченный вспомогательный прогноз
    RUN_50_SEED_CHECK: bool = True  # ранее выполненные 50 сидов V8.1 обязательно сохраняются
    INDEPENDENT_SCIENCE_APPROVED: bool = False # внешняя экспертиза пока не выполнена; локальный QA не подменяет ее
    PUBLISH_PRIVATE_PRERELEASE: bool = True  # строго ПОСЛЕ локальной полной технической и предметной приемки
    RELEASE_MAX_PART_MIB: int = 140
    VERIFY_PUBLISHED_DOWNLOADS: bool = True
    REVIEW_MODEL_BUDGET_MIB: int = 0  # Модели хранятся в приватном GitHub, а не в научном ZIP
    SCIENTIFIC_ARCHIVE_MAX_MIB: int = 120
    SAME_RUN_ONLY: bool = True   # Научные данные читаются только из релиза текущего Run all
CFG=Config()
assert CFG.MODE in ('FULL','SMOKE')
if CFG.MODE!='FULL':
    CFG.PUBLISH_PRIVATE_PRERELEASE=False
assert CFG.SAME_RUN_ONLY and (CFG.MODE!='FULL' or CFG.PUBLISH_PRIVATE_PRERELEASE), 'В FULL недопустимо пропускать GitHub round-trip'
print('ML-5 · V10.2 ·',CFG.MODE,'· единый Run all · публикация:',CFG.PUBLISH_PRIVATE_PRERELEASE)

## 1. Среда, неизменяемые источники и проверка каталогов
Скачивание работает по настоящим GitHub release `asset_id` и Git blobs из index. GitHub Drive не используется. Уже проверенные локальные копии повторно не загружаются. Данные и модели соседних ML-направлений не используются.

In [ ]:
import sys,subprocess,importlib.util
required={'duckdb':'duckdb>=1.1,<2','pyarrow':'pyarrow>=17,<25','networkx':'networkx>=3,<4',
          'xlsxwriter':'xlsxwriter>=3,<4','requests':'requests>=2,<3','scipy':'scipy>=1.12,<2',
          'sklearn':'scikit-learn>=1.4,<2','matplotlib':'matplotlib>=3.8,<4','joblib':'joblib>=1.3,<2','nbformat':'nbformat>=5,<6'}
missing=[pin for mod,pin in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','-q','install',*missing])
import os,re,io,json,time,math,hashlib,zipfile,tempfile,shutil,base64,platform,traceback
from datetime import datetime,timezone
from urllib.parse import quote
import numpy as np,pandas as pd,duckdb,pyarrow.parquet as pq,requests,networkx as nx
import matplotlib.pyplot as plt
from scipy.stats import spearmanr,kendalltau
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
from IPython.display import display,FileLink

ROOT=Path(CFG.ROOT);CACHE=Path(CFG.CACHE)
for p in [ROOT, CACHE, ROOT/'derived', ROOT/'figures', ROOT/'exports',ROOT/'models', ROOT/'tmp']:
    p.mkdir(parents=True,exist_ok=True)
for target,pattern in [('derived','*.parquet'),('models','*.joblib'),('figures','*.png')]:
    for stale in (ROOT/target).glob(pattern):stale.unlink()
for pattern in ('*.xlsx','*.zip','*.json','*.ipynb'):
    for stale in (ROOT/'exports').glob(pattern):stale.unlink()
TABLES={}; QA=[]; FINDINGS=[]; MODULES={}; SOURCE_USAGE=[]; QUARANTINE=[]; FORECAST_ROWS=[]
RUN_START=datetime.now(timezone.utc)
RUN_ID=RUN_START.strftime('%Y%m%dT%H%M%S%fZ')+'-'+hashlib.sha256((str(CFG.SEED)+CFG.REPO+RUN_START.isoformat()).encode()).hexdigest()[:12]
ROUNDTRIP_PHASE='PENDING'

from tqdm.auto import tqdm
# Одна шкала; состав этапов определён до начала вычислений.
TASK_STAGES=['Источники','OD СПО','Регионы','Изменения OD','Реестры','Контингент',
 'Специальности','Профили программ','Выпускники','Занятость','Сети','Чувствительность сетей',
 'Прогноз сети','Муниципалитеты','Сопоставление БДМО','Рынок труда','Территории','Региональный контроль',
 'Прогноз OD','Факторы потоков','Гравитационный baseline','Статистика','Графики','Выводы','Excel и Parquet','QA и архив','GitHub публикация','Данные из GitHub','Научный синтез','Скачиваемый архив','Финал']
MAIN_PROGRESS=tqdm(total=len(TASK_STAGES),desc='ML-5 V10.2',unit='этап',leave=True)
DONE_STAGES=set()
def mark_stage(name):
    if name not in TASK_STAGES:raise RuntimeError('Неизвестный этап прогресса: '+str(name))
    if name not in DONE_STAGES:
        DONE_STAGES.add(name);MAIN_PROGRESS.update(1);MAIN_PROGRESS.set_postfix_str(name)


def put(name,df,detail='',required=False):
    if not isinstance(df,pd.DataFrame):raise TypeError(name)
    TABLES[name]=df.reset_index(drop=True)
    if required and df.empty:raise RuntimeError(f'Пустая обязательная таблица {name}')
    QA.append({'check':'table_'+name,'status':'PASS' if len(df) else 'EMPTY','rows':int(len(df)),'detail':detail})
    return TABLES[name]

def issue(module,status,reason):
    MODULES.setdefault(module,[]).append({'status':status,'reason':reason})

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**22),b''):h.update(b)
    return h.hexdigest()

def clean_code(s):
    out=s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
    return out.mask(out.str.lower().isin(['','nan','none','null','<na>']))

def known_region_code(series):
    # Историческая V7.5.1: 0, 000..., unknown и «неизвестно» — НЕ территориальные коды.
    x=clean_code(series)
    return (x.notna() & x.ne('') & ~x.str.fullmatch(r'0+').fillna(False)
            & ~x.str.lower().isin(['nan','none','null','неизвестно','неизвестный','неизвестная','unknown','other','прочее','не указан','не указано','неизвестен','нет данных','без кода']).fillna(False)).fillna(False)

def assert_balanced(observed, label, atol=1e-7):
    # Тождество потоков В ПРЕДЕЛАХ ОДНОЙ набл. совокупности; не полнота миграции.
    if observed.empty:return
    x=observed.groupby('year').agg(вход=('прибывшие_в_наблюдаемом_показателе','sum'),
                                   выход=('выбывшие_в_наблюдаемом_показателе','sum'),
                                   баланс=('наблюдаемый_баланс','sum'))
    errors=((x['вход']-x['выход']).abs()>atol) | (x['баланс'].abs()>atol)
    if errors.any():raise AssertionError(f'Нарушено тождество OD: {label}, {x.loc[errors].to_dict()}')

def clean_name(s):
    return (s.astype('string').str.strip().str.lower().str.replace('ё','е',regex=False)
            .str.replace(r'\s+',' ',regex=True))

def sqlquote(value):return "'"+str(value).replace("'","''")+"'"

def db():
    con=duckdb.connect(database=':memory:')
    con.execute(f"SET memory_limit='{CFG.MAX_MEMORY_GB}GB'")
    con.execute(f'SET threads={int(CFG.WORKERS)}')
    con.execute('SET temp_directory='+sqlquote(ROOT/'tmp'))
    return con

def qry(query,params=None):
    with db() as con:return con.execute(query,params or []).df()

def safe_div(n,d):
    return np.divide(np.asarray(n,dtype=float),np.asarray(d,dtype=float),out=np.full(np.broadcast_shapes(np.shape(n),np.shape(d)),np.nan),where=np.asarray(d,dtype=float)!=0)

def register_critical(label,ok,message):
    QA.append({'check':label,'status':'PASS' if ok else 'FAIL','rows':0,'detail':message})
    if not ok:raise RuntimeError(f'{label}: {message}')

def fixed_region_universe(edges):
    codes=sorted(set(edges['origin_code'].dropna().astype(str)) | set(edges['destination_code'].dropna().astype(str)))
    return codes

print('Python',sys.version.split()[0],'DuckDB',duckdb.__version__,'RAM/cache:',shutil.disk_usage(ROOT).free//2**30,'GiB free')

## 0А. Дымный тест согласованности до обращения к крупным данным

Локальный транспортный тест не подменяет научные результаты. Ошибки останавливают `Run all` до дорогих вычислений.

In [ ]:
# 0A. Программный дымный тест согласованности (НЕ научный прогон).
# Проверяются Parquet -> SHA -> контроль строк -> архив -> распаковка, имена этапов и локальная имитация удалённого чтения.
import tempfile
SMOKE_CHECKS=[]
def smoke_assert(label,condition):
    SMOKE_CHECKS.append({'test':label,'result':'PASS' if bool(condition) else 'FAIL'})
    if not condition:raise AssertionError('PRE-FULL SMOKE: '+label)
with tempfile.TemporaryDirectory(prefix='ml5_v102_smoke_') as _s:
    _d=Path(_s);_mock=_d/'remote';_mock.mkdir();_download=_d/'download';_download.mkdir()
    _tiny=pd.DataFrame({'year':[2024,2024,2025,2025],'origin_code':['01','02','01','02'],
       'destination_code':['03','03','03','03'],'value':[10.,20.,11.,25.]})
    _source=_d/'od.parquet';_tiny.to_parquet(_source,index=False)
    _remote=_mock/'table.parquet';shutil.copy2(_source,_remote)
    _received=_download/'table.parquet';shutil.copy2(_remote,_received)
    smoke_assert('PARQUET_SHA_ROUNDTRIP',sha256(_source)==sha256(_remote)==sha256(_received))
    _re=pd.read_parquet(_received)
    smoke_assert('PARQUET_SCHEMA_ROWS',len(_re)==4 and list(_re.columns)==list(_tiny.columns))
    # SQL с экранированными годом и именем: именно этот класс ошибок был в V8.
    with duckdb.connect(':memory:') as _conn:
        _conn.register('od_fixture',_re.assign(name=['a','b','c','d']))
        _sql=_conn.execute('SELECT CAST("year" AS INTEGER) AS "year", CAST("name" AS VARCHAR) AS "name" FROM od_fixture').df()
    smoke_assert('DUCKDB_RESERVED_IDENTIFIERS',len(_sql)==4 and list(_sql.columns)==['year','name'])
    # Mock GitHub index binds asset by exact SHA and current run, never selects an older release.
    _mock_index={'run_id':RUN_ID,'tag':'mock-'+RUN_ID,'tables':[{'name':'OD','file':_remote.name,'sha256':sha256(_source),'rows':len(_tiny)}]}
    _idx=_mock/'index.json';_idx.write_text(json.dumps(_mock_index),encoding='utf-8')
    _idx_copy=_download/'index.json';shutil.copy2(_idx,_idx_copy)
    _check=json.loads(_idx_copy.read_text('utf-8'))
    smoke_assert('SAME_RUN_BINDING',_check['run_id']==RUN_ID and _check['tag']=='mock-'+RUN_ID)
    smoke_assert('MOCK_REMOTE_TABLE_HASH',sha256(_received)==_check['tables'][0]['sha256'])
    _summary=_re.groupby('year').value.sum()
    smoke_assert('OBSERVED_YEAR_SUMS',_summary.loc[2024]==30 and _summary.loc[2025]==36)
    _z=_d/'archive.zip'
    with zipfile.ZipFile(_z,'w',zipfile.ZIP_DEFLATED) as _zf:_zf.write(_received,'data/od.parquet')
    with zipfile.ZipFile(_z) as _zf:
        smoke_assert('ARCHIVE_READBACK',_zf.testzip() is None and hashlib.sha256(_zf.read('data/od.parquet')).hexdigest()==sha256(_source))
smoke_assert('STAGE_PLAN_UNIQUE',len(TASK_STAGES)==len(set(TASK_STAGES)))
smoke_assert('SINGLE_RUN_PUBLISH_ENABLED',CFG.MODE!='FULL' or CFG.PUBLISH_PRIVATE_PRERELEASE)
SMOKE_PROTOCOL=put('Дымный_тест_согласованности',pd.DataFrame(SMOKE_CHECKS),required=True)
print('Дымный тест согласованности:',len(SMOKE_CHECKS),'из',len(SMOKE_CHECKS),'PASS. Данные теста не используются в научных выводах.')


In [ ]:
# Исходный предметный контракт ML-5: семь модулей, только наблюдаемые поля Stage I.
# Запрещённые реконструкции: постоянный переезд по OD, география работы без поля,
# кампусы по адресу головного вуза, потоки по специальностям без связки OD×код,
# будущие значения источников как исторически доступные признаки.
SCIENTIFIC_EXCLUSIONS=('NO_POPULATION_MIGRATION_FROM_OD','NO_RETENTION_FROM_EMPLOYMENT',
 'NO_UNVERIFIED_CAMPUS_GEOGRAPHY','NO_OD_SPECIALTY_WITHOUT_LINK',
 'NO_UNVERIFIED_TOTAL_ADMISSIONS','NO_HISTORICAL_EX_ANTE_CLAIM')
# Первоначальные вопросы ML-5, без преждевременного объявления их выполненными.
QUESTIONS={
 'ML-5.1':'География фактически наблюдаемых межрегиональных потоков СПО и причины различий',
 'ML-5.2':'Структура и доступные показатели привлекательности организаций ВО/СПО',
 'ML-5.3':'Региональные доноры и получатели образовательных потоков, неравенство возможностей',
 'ML-5.4':'География специальностей ВО и СПО: реальные группы приёма и охват',
 'ML-5.5':'Выпуск, занятость и заработная плата по источникам E04/E05',
 'ML-5.6':'Центры, коридоры и устойчивость сети образовательной мобильности',
 'ML-5.7':'Муниципальная образовательная статистика: реально наблюдаемые показатели, без выдуманных кампусов'}

In [ ]:
# Только Colab Secret. Для обычной локальной диагностики допускается переменная окружения,
# но значение токена никогда не печатается и не включается в результаты.
def get_token():
    val=os.environ.get('GITHUB_TOKEN','')
    if not val:
        try:
            from google.colab import userdata
            val=userdata.get(CFG.COLAB_SECRET) or userdata.get('GITHUB_TOKEN')
        except Exception:pass
    if not val:raise RuntimeError(f'Нет доступа к приватному GitHub. В Colab Secrets включите {CFG.COLAB_SECRET}')
    return val

SESSION=requests.Session();SESSION.headers.update({'Authorization':'Bearer '+get_token(),
 'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
API='https://api.github.com/repos/'+CFG.REPO

def request(method,url,**kw):
    last=None
    for retry in range(4):
        try:
            r=SESSION.request(method,url,timeout=(25,900),**kw)
            if r.status_code in (429,500,502,503,504):
                r.close();time.sleep(min(2**retry,8));continue
            if not r.ok:
                status=r.status_code;r.close()
                raise RuntimeError(f'GitHub HTTP {status} (адрес и секрет скрыты)')
            return r
        except requests.RequestException as e:
            last=e;time.sleep(min(2**retry,8))
    raise RuntimeError('GitHub временно недоступен; завершён повтор запросов') from last

def github_json(path,**kw):
    with request('GET',API+path,**kw) as r:return r.json()

REPO_METADATA=github_json('')
if not REPO_METADATA.get('private'):raise RuntimeError('Исходный GitHub должен быть приватным')
GITHUB_WRITE_PREFLIGHT=bool((REPO_METADATA.get('permissions') or {}).get('push'))
SOURCE_COMMIT=github_json('/git/ref/heads/'+quote(CFG.SOURCE_BRANCH,safe='/'))['object']['sha']
# Verify complete Stage I handoff contract before starting a calculation (read-only).
HANDOFF_PREFLIGHT=[]
for handoff_path in ('handoff/07_v2/07_handoff_bundle.zip','handoff/07_v2/stage2_loader.py'):
    metadata=github_json('/contents/'+quote(handoff_path,safe='/'),params={'ref':SOURCE_COMMIT})
    if metadata.get('type')!='file':raise RuntimeError('Stage-I handoff is incomplete: '+handoff_path)
    HANDOFF_PREFLIGHT.append({'path':handoff_path,'git_blob_sha':metadata['sha'],'size':metadata.get('size')})

def git_bytes(path):
    meta=github_json('/contents/'+quote(path,safe='/'),params={'ref':SOURCE_COMMIT})
    if meta.get('type')!='file':raise RuntimeError('GitHub: вместо файла получен каталог')
    if meta.get('encoding')=='base64' and meta.get('content'):
        return base64.b64decode(meta['content']),meta['sha']
    blob=github_json('/git/blobs/'+meta['sha'])
    if blob.get('encoding')!='base64':raise RuntimeError('GitHub: нераспознанный формат blob')
    return base64.b64decode(blob['content']),meta['sha']

index_data,index_sha=git_bytes(CFG.INDEX_PATH)
if index_sha!=CFG.INDEX_BLOB_SHA:raise RuntimeError('Каталог Stage I изменён: требуется повторная проверка источников')
INDEX=json.loads(index_data);ASSETS=INDEX['assets']
if INDEX.get('version')!='7.2' or INDEX.get('repo')!=CFG.REPO or len(ASSETS)!=52:
    raise RuntimeError('Неверная версия или состав каталога Stage I')
if len({a['asset_id'] for a in ASSETS})!=52:raise RuntimeError('Повторный asset_id в каталоге')
for a in ASSETS:
    if not a.get('sha256_verified') or len(str(a.get('sha256','')))!=64:raise RuntimeError('Непроверенный источник '+a['asset_id'])
INDEX_CATALOG=put('Каталог_52',pd.DataFrame([{
    'asset_id':a['asset_id'],'источник':a.get('source_id'),'сегмент':a.get('segment'),
    'роль':a.get('role'),'строк_в_каталоге':a.get('rows_actual'),
    'столбцов_в_каталоге':len(a.get('columns_manifest') or []),
    'SHA256':a.get('sha256'),'метод_хранения':a['locator']['type'] } for a in ASSETS]),required=True)
SOURCE_USED={}

def asset(*,prefix=None,role=None,source=None,segment=None):
    matches=[a for a in ASSETS if (prefix is None or a['asset_id'].startswith(prefix))
          and (role is None or a.get('role')==role)
          and (source is None or a.get('source_id')==source)
          and (segment is None or a.get('segment')==segment)]
    if len(matches)!=1:raise RuntimeError(f'Нужен ровно один источник {prefix or role or source}: {len(matches)}')
    return matches[0]

def get_source(a):
    key=a['sha256'];p=CACHE/(key+'.parquet')
    if p.exists() and sha256(p)==key:
        SOURCE_USED[a['asset_id']]={'sha256':key,'rows':int(pq.ParquetFile(p).metadata.num_rows),'verified':True}
        return p
    if p.exists():p.unlink()
    tmp=Path(str(p)+'.partial');tmp.unlink(missing_ok=True)
    loc=a['locator'];size_free=shutil.disk_usage(CACHE).free
    if size_free<2**30:raise RuntimeError('Свободно менее 1 ГиБ: источники не загружать')
    try:
        if loc['type']=='git':
            raw,_=git_bytes(loc['path'])
            tmp.write_bytes(raw);del raw
        elif loc['type']=='release':
            url=API+'/releases/assets/'+str(loc['asset_id'])
            with request('GET',url,headers={'Accept':'application/octet-stream'},stream=True) as r:
                if 'json' in r.headers.get('Content-Type','').lower():
                    meta=r.json();download_url=meta.get('browser_download_url')
                    if not download_url:raise RuntimeError('Release asset: не получена бинарная ссылка')
                    r.close()
                    with request('GET',download_url,stream=True) as blob:
                        if 'json' in blob.headers.get('Content-Type','').lower():raise RuntimeError('Release asset вернул JSON')
                        with tmp.open('wb') as f:
                            for part in blob.iter_content(4*2**20):
                                if part:f.write(part)
                else:
                    with tmp.open('wb') as f:
                        for part in r.iter_content(4*2**20):
                            if part:f.write(part)
        else:raise RuntimeError('Неизвестный тип Stage I locator')
        if sha256(tmp)!=key:raise RuntimeError('SHA-256 источника не совпадает с каталогом: '+a['asset_id'])
        pf=pq.ParquetFile(tmp)
        if a.get('rows_actual') is not None and pf.metadata.num_rows!=int(a['rows_actual']):
            raise RuntimeError('Расхождение количества строк источника: '+a['asset_id'])
        expected=set(a.get('columns_manifest') or ())
        actual=set(pf.schema_arrow.names)
        if expected and not expected.issubset(actual):raise RuntimeError('Отсутствуют столбцы: '+a['asset_id'])
        tmp.replace(p)
    finally:tmp.unlink(missing_ok=True)
    SOURCE_USED[a['asset_id']]={'sha256':key,'rows':int(pq.ParquetFile(p).metadata.num_rows),'verified':True}
    print('Подтверждён источник:',a['asset_id'],'строк:',SOURCE_USED[a['asset_id']]['rows'])
    return p

# Дополнительный аудит всех 52 исходных файлов — отдельный явно включаемый режим:
# он не является обязательным скачиванием ~сотен миллионов строк для содержательного анализа.
if CFG.VERIFY_ALL_52_BYTES:
    for i,a in enumerate(ASSETS,1):
        get_source(a)
        print(f'Аудит SHA: {i}/52')

print('Источники Stage I:',len(ASSETS),'| исходный commit:',SOURCE_COMMIT[:12],
      '| index blob:',index_sha[:12])
mark_stage('Источники')

## 2. ML-5.1 / ML-5.3. Действительные OD СПО и изменение покрытия
Происхождение — регион предыдущего общего образования, назначение — регион текущего профессионального образования по E01. Это не подтверждённая постоянная миграция и не все абитуриенты России. Анализируются **реально зарегистрированные** пары. Внутрирегиональные и неизвестные записи отделены. Уровень покрытия измеряется ежегодно; постоянная панель включает только пары, зарегистрированные каждый год. Нулевые и незарегистрированные значения не отождествляются.

In [ ]:
E01=asset(role='education_od_flows');E02=asset(prefix='04::E02');VPO=asset(prefix='04::VPO_2_1_1')
SPO_ORG=asset(role='education_organizations');SPO_REG=asset(role='regional_spo')
E04=asset(prefix='05::E04::organization_total');E05=asset(prefix='05::E05::field_specialty')
BDMO_EDU=asset(prefix='02::education');D01=asset(source='D01')
D05_LABOR=asset(prefix='02::labor')
# DATA_GATE: проверка реальной схемы каждого используемого первоисточника ДО расчётов.
REQUIRED_SOURCE_FIELDS={
 'E01':(E01,['year','origin_region_code','origin_region_name','destination_region_code','destination_region_name','indicator_value']),
 'E02':(E02,['id','name','year','region','parent_id','institution','indicator_name','area','indicator_value']),
 'VPO_2_1_1':(VPO,['year','object_level','object_name','specialty_code','specialty_name','education_level','education_form','indicator_name','indicator_value','admission_type','university_type']),
 'SPO_ORG':(SPO_ORG,['year','id','name','region_name','region_oktmo','head','branches']),
 'SPO_REG':(SPO_REG,['year','region_name','indicator_name','indicator_unit','specialty_code','specialty_name','region_value']),
 'E04':(E04,['object_level','object_name','university','gender','education_level','year','count_graduate','percent_employed','average_salary_fact_med']),
 'E05':(E05,['object_level','object_name','study_area','specialty_code','specialty','gender','education_level','year','count_graduate','percent_employed','average_salary_fact_med']),
 'BDMO':(BDMO_EDU,['year','indicator_name','indicator_code','indicator_unit','indicator_value','region_name','oktmo']),
 'D01':(D01,['year','oktmo','region','population','births','deaths','migration_rate']),
 'D05':(D05_LABOR,['year','object_level','indicator_code','indicator_name','indicator_unit'])}
GATES=[]
for name,(src,fields) in REQUIRED_SOURCE_FIELDS.items():
    miss=sorted(set(fields)-set(src.get('columns_manifest') or []))
    GATES.append({'source':name,'asset_id':src['asset_id'],'rows_in_catalog':src.get('rows_actual'),
                  'required_columns':len(fields),'missing_columns':','.join(miss),
                  'status':'PASS_SCHEMA' if not miss else 'FAIL_SCHEMA'})
    if miss:raise RuntimeError(f'DATA_GATE {name}: отсутствуют {miss}')
put('DATA_GATE_источников',pd.DataFrame(GATES),required=True)


q="""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("origin_region_code" AS VARCHAR) AS "origin_code", CAST("origin_region_name" AS VARCHAR) AS "origin_name", CAST("destination_region_code" AS VARCHAR) AS "destination_code", CAST("destination_region_name" AS VARCHAR) AS "destination_name", TRY_CAST("indicator_value" AS DOUBLE) AS "value" FROM read_parquet(?)"""
OD=qry(q,[str(get_source(E01))]);OD=OD.loc[OD['year'].notna() & OD['value'].notna() & OD['value'].ge(0)].copy(); OD['year']=OD['year'].astype(int)
OD['origin_code']=clean_code(OD.origin_code);OD['destination_code']=clean_code(OD.destination_code)
OD['origin_name']=OD.origin_name.astype('string').str.strip();OD['destination_name']=OD.destination_name.astype('string').str.strip()
known_o=known_region_code(OD['origin_code'])
known_d=known_region_code(OD['destination_code'])
OD['category']=np.select([(~known_o|~known_d).to_numpy(dtype=bool),
    OD.origin_code.eq(OD.destination_code).fillna(False).to_numpy(dtype=bool)],
    ['НЕИЗВЕСТНО','ВНУТРИРЕГИОНАЛЬНО'],default='МЕЖРЕГИОНАЛЬНО')
key=['year','origin_code','destination_code']
dups=OD.duplicated(key,keep=False)&OD.origin_code.notna()&OD.destination_code.notna()
if dups.any():
    QUARANTINE.append({'table':'E01','problem':'Повторный OD-ключ','count':int(dups.sum())})
    # Нельзя ни суммировать, ни произвольно выбирать: конфликтующие ключи удаляются все.
    OD=OD.loc[~dups].copy()
put('OD_исходные',OD,required=True)
OD_PARTS=put('OD_состав',OD.groupby(['year','category'],dropna=False).agg(
 записей=('value','size'),наблюдаемая_величина=('value','sum'),положительных=('value',lambda s:int(s.gt(0).sum()))).reset_index())
INTER=OD.loc[OD.category=='МЕЖРЕГИОНАЛЬНО'].copy()
INTER['edge']=INTER.origin_code+'|'+INTER.destination_code
if INTER.empty:raise RuntimeError('E01: нет достоверных межрегиональных OD-пар')
YEARS=sorted(INTER.year.dropna().astype(int).unique());BY_YEAR={y:set(g.edge) for y,g in INTER.groupby('year')}
fixed=set.intersection(*(BY_YEAR[y] for y in YEARS))
COVERAGE=[]
for y in YEARS:
    current=BY_YEAR[y];previous=BY_YEAR.get(y-1,set())
    g=INTER.loc[INTER.year.eq(y)]
    gf=g.loc[g.edge.isin(fixed)]
    COVERAGE.append({'год':int(y),'наблюдаемых_пар':len(current),
        'положительных_пар':int(g.value.gt(0).sum()),'неизменных_пар':len(gf),
        'все_наблюдаемые_значения':float(g.value.sum()),'значения_общих_пар':float(gf.value.sum()),
        'новых_к_предыдущему':len(current-previous) if previous else np.nan,
        'исчезнувших_к_предыдущему':len(previous-current) if previous else np.nan,
        'пересечение_к_предыдущему':len(previous&current) if previous else np.nan})
COVERAGE=put('OD_охват_по_годам',pd.DataFrame(COVERAGE),required=True)
FIXED=put('OD_постоянные_пары',INTER[INTER.edge.isin(fixed)].copy())
# Контроль только при том же закреплённом E01 Stage I: проверенные V7.5.1 итоги.
expected_fixed={2021:64189,2022:68567,2023:86367,2024:88844,2025:96487}
expected_unknown={2023:170,2024:170,2025:170}
if set(YEARS)==set(expected_fixed):
    for y,val in expected_fixed.items():
        actual=float(FIXED.loc[FIXED.year.eq(y),'value'].sum())
        if not np.isclose(actual,val,atol=0.001):
            raise RuntimeError(f'E01, общие пары, {y}: {actual} вместо контрольного {val}')
    if len(fixed)!=2480:raise RuntimeError('E01: изменена доказанная совокупность из 2480 пар')
    actual_unknown=OD.loc[OD.category.eq('НЕИЗВЕСТНО')].groupby('year').size().to_dict()
    for y,n in expected_unknown.items():
        if actual_unknown.get(y)!=n:raise RuntimeError(f'E01: UNKNOWN_CODE {y}: {actual_unknown.get(y)} вместо {n}')
    QA.append({'check':'E01_CONTROL_2480_AND_UNKNOWN','status':'PASS','rows':len(FIXED),
               'detail':'Восстановлены 170 UNKNOWN_CODE за 2023–2025 и 2480 постоянных пар'})

if 2022 in YEARS and 2023 in YEARS:
    a=COVERAGE.set_index('год')
    FINDINGS.append({'модуль':'ML-5.1','вывод':'Изменение охвата OD в 2023 году',
      'расчёт':f"зарегистрированные пары {int(a.loc[2022,'наблюдаемых_пар'])} -> {int(a.loc[2023,'наблюдаемых_пар'])}",
      'ограничение':'Рост записей не является автоматически ростом реальных потоков'})
issue('ML-5.1','OBSERVED','Проверены OD-пары СПО, годовое покрытие и постоянная совокупность. Постоянная смена места жительства не наблюдается')
print('OD годы:',YEARS,'межрегиональных записей:',len(INTER),'пар на постоянном покрытии:',len(fixed))
mark_stage('OD СПО')

## 3. Кто притягивает и откуда уезжают учиться: региональные профили
Для каждого региона: наблюдаемые отправления и поступления, баланс **только в E01**, партнёры, доля крупнейших направлений и изменение ранга. Основные межгодовые сравнения строятся на неизменных парах; полные наблюдаемые величины приводятся отдельно. Нельзя называть баланс показателем общей миграции населения.

In [ ]:
ORIGIN_NAMES=(INTER[['origin_code','origin_name']].drop_duplicates().rename(columns={'origin_code':'code','origin_name':'name'}))
DEST_NAMES=(INTER[['destination_code','destination_name']].drop_duplicates().rename(columns={'destination_code':'code','destination_name':'name'}))
REG_NAMES=pd.concat([ORIGIN_NAMES,DEST_NAMES],ignore_index=True)
name_freq=(REG_NAMES.groupby(['code','name'],dropna=False).size().reset_index(name='frequency')
           .sort_values(['code','frequency','name'],ascending=[True,False,True]))
REG_MAP=name_freq.drop_duplicates('code').set_index('code')['name'].to_dict()
conflict=name_freq.groupby('code').name.nunique()
if conflict.gt(1).any():
    QUARANTINE.append({'table':'E01_region_names','problem':'Несколько вариантов названия для кода региона','count':int(conflict.gt(1).sum())})
put('E01_названия_регионов',name_freq)

def regional_balances(edges,support):
    e=edges.copy()
    out=(e.groupby(['year','origin_code'],as_index=False).agg(
      выбывшие_в_наблюдаемом_показателе=('value','sum'),направлений=('destination_code','nunique'),
      положительных_направлений=('value',lambda a:int(a.gt(0).sum())))
      .rename(columns={'origin_code':'region_code'}))
    inc=(e.groupby(['year','destination_code'],as_index=False).agg(
      прибывшие_в_наблюдаемом_показателе=('value','sum'),регионов_источников=('origin_code','nunique'))
      .rename(columns={'destination_code':'region_code'}))
    g=out.merge(inc,on=['year','region_code'],how='outer',validate='one_to_one')
    # Нет зарегистрированной входящей стороны => её сумма в выбранной OD-совокупности 0,
    # но НЕ утверждается отсутствие реальной миграции или неучтённых направлений.
    g['есть_наблюдаемые_исходящие']=g['выбывшие_в_наблюдаемом_показателе'].notna()
    g['есть_наблюдаемые_входящие']=g['прибывшие_в_наблюдаемом_показателе'].notna()
    for col in ['прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе',
                'направлений','положительных_направлений','регионов_источников']:
        g[col]=g[col].fillna(0)
    g['наблюдаемый_баланс']=g['прибывшие_в_наблюдаемом_показателе']-g['выбывшие_в_наблюдаемом_показателе']
    g['наблюдаемое_отношение_вход_выход']=g['прибывшие_в_наблюдаемом_показателе']/g['выбывшие_в_наблюдаемом_показателе'].replace(0,np.nan)
    g['название_региона']=g.region_code.map(REG_MAP)
    g['совокупность']=support
    return g.sort_values(['year','наблюдаемый_баланс'],ascending=[True,False])

BAL_FULL=put('Регионы_все_пары',regional_balances(INTER,'ВСЕ_ЗАРЕГИСТРИРОВАННЫЕ'))
assert_balanced(BAL_FULL,'полное наблюдаемое OD')
BAL_FIXED=put('Регионы_постоянный_охват',regional_balances(FIXED,'ОБЩИЕ_ПАРЫ_ВСЕ_ГОДЫ')) if len(FIXED) else pd.DataFrame()
assert_balanced(BAL_FIXED,'постоянная совокупность')
LATEST=max(YEARS)
BASE_LATEST=BAL_FIXED if len(BAL_FIXED) else BAL_FULL
REGION_LATEST=BASE_LATEST.loc[BASE_LATEST.year.eq(LATEST)].copy()
REGION_LATEST['место_по_входящим']=REGION_LATEST['прибывшие_в_наблюдаемом_показателе'].rank(method='min',ascending=False)
REGION_LATEST['место_по_исходящим']=REGION_LATEST['выбывшие_в_наблюдаемом_показателе'].rank(method='min',ascending=False)
REGION_LATEST['тип_наблюдаемого_потока']=np.select([
 REGION_LATEST.наблюдаемый_баланс.gt(0),REGION_LATEST.наблюдаемый_баланс.lt(0)],
 ['ПРЕОБЛАДАЕТ_ВХОДЯЩИЙ','ПРЕОБЛАДАЕТ_ИСХОДЯЩИЙ'],default='ПАРИТЕТ_НАБЛЮДАЕМЫХ_ПОТОКОВ')
REGION_LATEST=put('Региональные_профили',REGION_LATEST)

# Концентрация направлений: отдельно распределение по каждой стороне OD.
def concentration(edges,region_col,other_col,kind):
    res=[]
    for (year,region),g in edges.groupby(['year',region_col],dropna=False):
        w=g.value.to_numpy(float);w=w[w>0];total=w.sum()
        if total<=0:continue
        shares=w/total;ordered=np.sort(w)[::-1]
        res.append({'year':int(year),'region_code':region,'сторона':kind,
             'положительных_связей':len(w),'наблюдаемая_сумма':float(total),
             'HHI':float(np.sum(shares**2)),'эффективное_число_партнёров':float(1/np.sum(shares**2)),
             'доля_крупнейшего':float(ordered[0]/total),
             'доля_первых_5':float(ordered[:5].sum()/total),
             'энтропия':float(-np.sum(shares*np.log(shares)))})
    r=pd.DataFrame(res)
    if len(r):r['название_региона']=r.region_code.map(REG_MAP)
    return r

CONCENTRATION=put('Концентрация_направлений',pd.concat([
 concentration(FIXED if len(FIXED) else INTER,'origin_code','destination_code','ИСХОДЯЩИЕ'),
 concentration(FIXED if len(FIXED) else INTER,'destination_code','origin_code','ВХОДЯЩИЕ')],ignore_index=True))
CORRIDORS=INTER[INTER.year.eq(LATEST)].sort_values('value',ascending=False).head(CFG.TOP_N).copy()
CORRIDORS=put('Крупнейшие_OD_направления',CORRIDORS[['year','origin_code','origin_name','destination_code','destination_name','value']],required=True)

# Динамика на идентичной совокупности: наблюдаемые значения одинаковых OD-пар.
if len(FIXED)>=1 and len(YEARS)>=2:
    fst=FIXED[FIXED.year.eq(min(YEARS))][['edge','value']].rename(columns={'value':'начало'})
    lst=FIXED[FIXED.year.eq(max(YEARS))][['edge','value']].rename(columns={'value':'конец'})
    growth=fst.merge(lst,on='edge',validate='one_to_one')
    growth['абсолютное_изменение']=growth.конец-growth.начало
    growth['процент_от_начала']=np.where(growth.начало>0,100*growth.абсолютное_изменение/growth.начало,np.nan)
    growth['origin_code']=growth.edge.str.split('|',regex=False).str[0]
    growth['destination_code']=growth.edge.str.split('|',regex=False).str[1]
    growth['origin_name']=growth.origin_code.map(REG_MAP);growth['destination_name']=growth.destination_code.map(REG_MAP)
    put('Изменение_общих_маршрутов',growth.sort_values('абсолютное_изменение',ascending=False))

FINDINGS.append({'модуль':'ML-5.3','вывод':'Отток измеряется только в наблюдаемой E01-совокупности',
    'расчёт':'Ранжирование регионов, концентрация направлений и баланс на общих парах',
    'ограничение':'Нет полного знаменателя всех поступивших по регионам; нельзя вычислять общую долю уехавших'})
issue('ML-5.3','OBSERVED','Региональные профили, наблюдаемые отток/приток, направления и концентрация; без доли всех абитуриентов')
print('Регионов:',len(REGION_LATEST),'коридоров:',len(CORRIDORS))
mark_stage('Регионы')

## 3А. Сопоставимость OD и устойчивость положения регионов

Изменение зарегистрированных направлений нельзя автоматически считать миграционным ростом. Отдельно показываем **появление/исчезновение наблюдаемых пар** и изменение значений только на пересечении сравниваемых лет. Для каждого региона проверяем устойчивость его положения на постоянной совокупности (входящий и исходящий ранги, балансы). Истинные отсутствующие OD-пары не заменяются нулём.

In [ ]:
COVER_MOVES=[]; REG_SHIFT=[]; RANK_STABILITY=[]
for y0,y1 in zip(YEARS[:-1],YEARS[1:]):
    a=INTER.loc[INTER['year'].eq(y0),['edge','origin_code','destination_code','value']]
    b=INTER.loc[INTER['year'].eq(y1),['edge','origin_code','destination_code','value']]
    ab=a.merge(b,on='edge',suffixes=('_old','_new'),validate='one_to_one')
    new=b.loc[~b.edge.isin(a.edge)]; gone=a.loc[~a.edge.isin(b.edge)]
    prev=a['value'].sum(); nxt=b['value'].sum()
    COVER_MOVES.append({'период':f'{y0}–{y1}','год_от':int(y0),'год_до':int(y1),
      'общих_маршрутов':len(ab),'впервые_зарегистрированных':len(new),'выбывших_из_реестра':len(gone),
      'сумма_сопоставимых_раньше':float(ab.value_old.sum()),'сумма_сопоставимых_позже':float(ab.value_new.sum()),
      'изменение_только_на_общих_маршрутах':float((ab.value_new-ab.value_old).sum()),
      'сумма_новых_записей_в_текущем_году':float(new.value.sum()),
      'сумма_исчезнувших_записей_в_предыдущем_году':float(gone.value.sum()),
      'разница_всей_наблюдаемой_суммы':float(nxt-prev),
      'проверка_арифметики':bool(np.isclose((ab.value_new-ab.value_old).sum()+new.value.sum()-gone.value.sum(),nxt-prev,atol=1e-7)),
      'предупреждение':'Новые/исчезнувшие записи означают изменение наблюдаемого охвата, а не доказанную миграционную динамику'})
if COVER_MOVES and not all(x['проверка_арифметики'] for x in COVER_MOVES):raise AssertionError('Разложение изменений OD не согласовано с первичным реестром')
put('OD_переходы_покрытия',pd.DataFrame(COVER_MOVES),required=True)
if len(BAL_FIXED) and len(YEARS)>=2:
    y0,y1=int(YEARS[0]),int(YEARS[-1]);d0=BAL_FIXED.loc[BAL_FIXED.year.eq(y0)];d1=BAL_FIXED.loc[BAL_FIXED.year.eq(y1)]
    cols=['region_code','наблюдаемый_баланс','прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе']
    t=d0[cols].merge(d1[cols],on='region_code',suffixes=('_начало','_конец'),validate='one_to_one')
    t['регион']=t.region_code.map(REG_MAP)
    for kind in ['наблюдаемый_баланс','прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе']:
        t[kind+'_изменение']=t[kind+'_конец']-t[kind+'_начало']
        t[kind+'_место_начало']=t[kind+'_начало'].rank(ascending=False,method='min')
        t[kind+'_место_конец']=t[kind+'_конец'].rank(ascending=False,method='min')
        t[kind+'_изменение_места']=t[kind+'_место_начало']-t[kind+'_место_конец']
        rho,p=spearmanr(t[kind+'_начало'],t[kind+'_конец'])
        RANK_STABILITY.append({'показатель':kind,'первый_год':y0,'последний_год':y1,
          'число_регионов':len(t),'rho_рангов':float(rho),'p_описательная':float(p),
          'совокупность':'Только неизменные OD-пары за все годы'})
    REG_SHIFT=t.sort_values('наблюдаемый_баланс_изменение',ascending=False,kind='mergesort')
    put('OD_региональные_сдвиги',REG_SHIFT,required=True)
put('OD_устойчивость_регионов',pd.DataFrame(RANK_STABILITY))
FINDINGS.append({'модуль':'ML-5.1 / ML-5.3','вывод':'Разделены изменение интенсивности наблюдаемых маршрутов и изменение охвата E01',
    'расчёт':f'{len(COVER_MOVES)} переходов между годами; сравнение {len(fixed)} фиксированных маршрутов',
    'ограничение':'Записи вне пересечения лет не получают нулевых значений в отсутствующие годы'})
mark_stage('Изменения OD')

## 4. ML-5.2. Реестр учреждений ВО и СПО, отдельная статистика организаций
Источники E02 и E01 используются **в исходной статистической единице**. Реестр не определяет автоматически фактическую муниципальную географию кампусов и филиалов. Приведённый контингент E02 — не общий приём и не доля приезжих. Сравнение возможно по точным стратам, без сложения перекрывающихся категорий.

In [ ]:
# E02: DISTINCT организаций по стабильному id и году; конфликтующие ключи отправляются в карантин.
p=get_source(E02)
vo=qry("""SELECT DISTINCT CAST("id" AS VARCHAR) AS "id", CAST("name" AS VARCHAR) AS "name", CAST("region" AS VARCHAR) AS "region", CAST("parent_id" AS VARCHAR) AS "parent_id", CAST("parent_name" AS VARCHAR) AS "parent_name", TRY_CAST("year" AS INTEGER) AS "year", CAST("institution" AS VARCHAR) AS "institution" FROM read_parquet(?) WHERE "id" IS NOT NULL AND "year" IS NOT NULL""",[str(p)])
vo['id']=clean_code(vo.id);vo['parent_id']=clean_code(vo.parent_id)
vokey=['year','id'];dups=vo.duplicated(vokey,keep=False)
if dups.any():QUARANTINE.append({'table':'E02_organizations','problem':'Несколько вариантов записи одной организации в году','count':int(dups.sum())})
vo=vo[~dups].copy();vo['sector']='ВО';vo['region_key']=clean_name(vo.region)
vo['parent_group']=vo.parent_id.fillna(vo.id)
put('Реестр_ВО',vo)

sp=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("id" AS VARCHAR) AS "id", CAST("name" AS VARCHAR) AS "name", CAST("region_name" AS VARCHAR) AS "region", CAST("region_oktmo" AS VARCHAR) AS "region_oktmo", CAST("city" AS VARCHAR) AS "city", CAST("head" AS VARCHAR) AS "head", CAST("address" AS VARCHAR) AS "address", CAST("branches" AS VARCHAR) AS "branches", CAST("type" AS VARCHAR) AS "organization_type" FROM read_parquet(?) WHERE "id" IS NOT NULL AND "year" IS NOT NULL""",[str(get_source(SPO_ORG))])
sp['id']=clean_code(sp.id);sp['region_key']=clean_name(sp.region)
spdups=sp.duplicated(['year','id'],keep=False)
if spdups.any():QUARANTINE.append({'table':'SPO_organizations','problem':'Неоднозначный id×год','count':int(spdups.sum())})
sp=sp[~spdups].copy();sp['sector']='СПО'
put('Реестр_СПО',sp)

regcounts=pd.concat([vo.groupby(['year','region_key','region'],dropna=False).agg(число_учреждений=('id','nunique')).reset_index().assign(сектор='ВО'),
 sp.groupby(['year','region_key','region'],dropna=False).agg(число_учреждений=('id','nunique')).reset_index().assign(сектор='СПО')],ignore_index=True)
REGISTRY_COUNTS=put('Сеть_организаций_регион',regcounts)

# Ровно один известный показатель E02. Любой разрез area сохраняется дословно:
# суммирование разных area и нескольких строк одной организации запрещено.
CONTINGENT_NAME='Приведенный контингент студентов'
q="""SELECT CAST("id" AS VARCHAR) AS "id", CAST("name" AS VARCHAR) AS "name", CAST("region" AS VARCHAR) AS "region", TRY_CAST("year" AS INTEGER) AS "year", CAST("area" AS VARCHAR) AS "area", CAST("indicator_unit" AS VARCHAR) AS "unit", COUNT(*)::BIGINT AS "n_rows", MIN(TRY_CAST("indicator_value" AS DOUBLE)) AS "value_min", MAX(TRY_CAST("indicator_value" AS DOUBLE)) AS "value_max" FROM read_parquet(?) WHERE "indicator_name" = ? AND "id" IS NOT NULL AND "year" IS NOT NULL GROUP BY 1,2,3,4,5,6"""
cc=qry(q,[str(p),CONTINGENT_NAME]);cc['id']=clean_code(cc.id)
# Иные единицы и совпадения по двум разным stratum не могут быть интерпретированы как total.
valid=cc.n_rows.eq(1)&cc.value_min.notna()&cc.value_min.eq(cc.value_max)&cc.value_min.ge(0)
CONT=put('ВО_наблюдаемый_контингент',cc.assign(подтвержденная_строка=valid))
if len(CONT):
    eligible=CONT[CONT.подтвержденная_строка].copy()
    eligible['region_key']=clean_name(eligible.region)
    # Если имя area неизвестно, публикуем только исходный разрез с явным названием.
    eligible=eligible.sort_values(['year','area','value_min','region_key','id'],ascending=[False,True,False,True,True],kind='mergesort')
    put('ВО_контингент_страты',eligible)

# Сопоставление регионов ОД и регистров: по ТОЧНО совпадающим нормализованным именам.
# Совпадение здесь — только для анализа географии уровня региона, НЕ привязка кампуса к ОКТМО.
orglatest=REGISTRY_COUNTS[(REGISTRY_COUNTS.year==LATEST)&(REGISTRY_COUNTS.сектор=='СПО')]
orgsafe=orglatest.groupby('region_key',as_index=False).agg(организаций_СПО=('число_учреждений','sum'))
joined=REGION_LATEST.copy();joined['region_key']=clean_name(joined.название_региона)
joined=joined.merge(orgsafe,on='region_key',how='left',validate='many_to_one',indicator=True)
joined['регион_сопоставлен']=joined._merge.eq('both');joined=joined.drop(columns='_merge')
put('Регион_OD_и_СПО_сеть',joined)
print('Точное совпадение географии E01 и реестра СПО:',int(joined.регион_сопоставлен.sum()),'из',len(joined))
issue('ML-5.2','PARTIALLY_OBSERVED','Реестры ВО/СПО и отдельные E02-страты; общий приём и доля иногородних по организациям недоступны')
mark_stage('Реестры')

## 4А. Масштаб и изменение наблюдаемого контингента вузов

Анализируем **ровно один утверждённый индикатор E02**, сохраняя его `area` и единицу измерения. Строим панель совпадающих организаций и разрезов, изучаем изменение и концентрацию внутри сопоставимого регионального среза; не называем контингент приёмом и не смешиваем разные `area`.

In [ ]:
VO_COV=[];VO_CHANGE=[];VO_REGION=[]
raw=CONT.loc[CONT['подтвержденная_строка']].copy() if len(CONT) else pd.DataFrame()
if len(raw):
    raw=raw.loc[raw.year.notna() & raw.id.notna() & raw.value_min.ge(0)].copy()
    raw['region_key']=clean_name(raw['region']);raw['id']=clean_code(raw['id'])
    # Только полностью определённые разрезы. Повторы ключа x год в карантин, без усреднения.
    keys=['id','area','unit','year']
    amb=raw.duplicated(keys,keep=False)
    if amb.any():QUARANTINE.append({'table':'E02_contingent','problem':'Повторные id-area-unit-year','count':int(amb.sum())})
    raw=raw.loc[~amb].copy()
    regkeys=['year','region_key','area','unit']
    if len(raw):
        grp=raw.groupby(regkeys,dropna=False)
        VO_REGION=grp.agg(организаций_с_измерением=('id','nunique'),сумма_наблюдаемых_значений=('value_min','sum'),
                          медиана_по_организациям=('value_min','median'),p90_по_организациям=('value_min',lambda s:s.quantile(.9))).reset_index()
        # Внутри точно одинаковой единицы/разреза по организациям проверяем концентрацию.
        vals=[]
        for k,g in grp:
            pos=g.value_min.loc[g.value_min.gt(0)].to_numpy(float)
            if len(pos)<3:continue
            vals.append(dict(zip(regkeys,k)) | {'организаций_положительных':len(pos),'HHI_контингента_внутри_разреза':float(((pos/pos.sum())**2).sum()),
                'доля_крупнейших_3':float(np.sort(pos)[-3:].sum()/pos.sum()),
                'оговорка':'Сумма только наблюдаемых организаций внутри одного area; не тотал региона'})
        VO_COV=pd.DataFrame(vals)
        panel=raw[['id','area','unit','year','region_key','value_min']].sort_values(['id','area','unit','year'],kind='mergesort')
        prev=panel[['id','area','unit','year','region_key','value_min']].copy();prev['year']+=1;prev=prev.rename(columns={'value_min':'значение_предыдущий_год','region_key':'region_key_prior'})
        matched=panel.merge(prev,on=['id','area','unit','year'],how='inner',validate='one_to_one')
        relocation=matched['region_key'].fillna('_MISSING_').ne(matched['region_key_prior'].fillna('_MISSING_'))
        if relocation.any():QUARANTINE.append({'table':'E02_contingent','problem':'Одинаковый id-area-unit сменил регион между годами; межгодовой расчёт исключён','count':int(relocation.sum())})
        matched=matched.loc[~relocation].copy()
        matched['разница']=matched['value_min']-matched['значение_предыдущий_год']
        matched['темп_изменения_процентов']=100*safe_div(matched['разница'],matched['значение_предыдущий_год'])
        VO_CHANGE=matched.sort_values(['year','id','area','unit'],kind='mergesort')
put('ВО_динамика_контингента',VO_CHANGE if isinstance(VO_CHANGE,pd.DataFrame) else pd.DataFrame())
put('ВО_структура_регионов',VO_REGION if isinstance(VO_REGION,pd.DataFrame) else pd.DataFrame())
put('ВО_структура_охвата',VO_COV if isinstance(VO_COV,pd.DataFrame) else pd.DataFrame())
FINDINGS.append({'модуль':'ML-5.2','вывод':'Панель сопоставимых наблюдений контингента учреждений по годам и регионам',
 'расчёт':f'{len(VO_CHANGE) if isinstance(VO_CHANGE,pd.DataFrame) else 0} межгодовых пар по одному id-area-unit; отдельно уникальные учебные организации',
 'ограничение':'Это не число зачисленных и не доля поступивших из других регионов'})
mark_stage('Контингент')

## 5. ML-5.4. Специальности и география реального приёма
ВПО-1: только показатель с **точным именем** `Принято, всего` и региональные записи. Статистическая единица — регион × код × уровень × форма × вид конкурса × тип вуза × год. Строки с кодом `Всего` отделяются как итоги и **никогда не складываются** с детальными специальностями. СПО: показатели региональных программ из собственного каталога E01, без утверждений об OD по специальности.

In [ ]:
q="""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("object_name" AS VARCHAR) AS "region", CAST("object_level" AS VARCHAR) AS "object_level", CAST("specialty_code" AS VARCHAR) AS "code", CAST("specialty_name" AS VARCHAR) AS "specialty_name", CAST("education_level" AS VARCHAR) AS "level", CAST("education_form" AS VARCHAR) AS "form", CAST("admission_type" AS VARCHAR) AS "admission_type", CAST("university_type" AS VARCHAR) AS "university_type", COUNT(*)::BIGINT AS "n_records", MIN(TRY_CAST("indicator_value" AS DOUBLE)) AS "low", MAX(TRY_CAST("indicator_value" AS DOUBLE)) AS "high" FROM read_parquet(?) WHERE TRIM(CAST("indicator_name" AS VARCHAR)) = 'Принято, всего' AND regexp_matches(lower(CAST("object_level" AS VARCHAR)),'регион|субъект') AND "year" IS NOT NULL AND "specialty_code" IS NOT NULL GROUP BY 1,2,3,4,5,6,7,8,9"""
vp=qry(q,[str(get_source(VPO))])
vp['code']=clean_code(vp.code)
vp['aggregated_code']=vp.code.astype('string').str.lower().str.contains(r'^всего$|^итого$|^all$',regex=True,na=False)
vp['valid']=vp.n_records.eq(1)&vp.low.notna()&vp.low.eq(vp.high)&vp.low.ge(0)&(vp.low.round()==vp.low)
vp['region_key']=clean_name(vp.region)
PROGRAM_GATE=put('ВПО_контроль_страт',vp.groupby(['year','valid','aggregated_code'],dropna=False).agg(
 строк=('code','size'),наблюдаемая_сумма_показателя=('low','sum')).reset_index())
PROGRAM=vp[vp.valid & ~vp.aggregated_code & vp.code.notna()].copy()
if PROGRAM.empty:
    issue('ML-5.4','NOT_FEASIBLE','ВПО-1: нет однозначных региональных записей по специальности')
else:
    PROGRAM=put('ВПО_прием_точные_страты',PROGRAM,required=True)
    family=['year','region_key','level','form','admission_type','university_type']
    # Доли рассчитываются только ВНУТРИ одного и того же подробного разреза, никакой суммы итогов.
    g=PROGRAM.groupby(family,dropna=False).agg(наблюдаемых_кодов=('code','nunique'),
       сумма_точных_строк=('low','sum')).reset_index()
    g=g[g.наблюдаемых_кодов>=2]
    pr=PROGRAM.merge(g,on=family,how='inner',validate='many_to_one')
    pr['доля_в_наблюдаемых_кодах']=pr.low/pr.сумма_точных_строк.where(pr.сумма_точных_строк>0)
    pr['рейтинг_кода_в_разрезе']=pr.groupby(family,dropna=False).low.rank(ascending=False,method='min')
    top=pr.loc[pr.рейтинг_кода_в_разрезе<=5].copy()
    put('ВПО_ведущие_направления',top.sort_values(['year','region_key','level','form','admission_type','university_type','рейтинг_кода_в_разрезе','code'],ascending=[False,True,True,True,True,True,True,True],kind='mergesort'))
    # Межрегиональная география КОНКРЕТНОЙ программы, в идентичных уровнях/формах.
    country_family=['year','code','level','form','admission_type','university_type']
    cov=PROGRAM.groupby(country_family,dropna=False).agg(наблюдаемых_регионов=('region_key','nunique'),
       сумма_наблюдаемых_строк=('low','sum')).reset_index()
    geo=PROGRAM.merge(cov,on=country_family,how='left',validate='many_to_one')
    geo['доля_среди_наблюдаемых_регионов']=geo.low/geo.сумма_наблюдаемых_строк.where(geo.сумма_наблюдаемых_строк>0)
    put('ВПО_география_специальностей',geo.sort_values(['доля_среди_наблюдаемых_регионов','year','region_key','code'],ascending=[False,False,True,True],kind='mergesort'))
    issue('ML-5.4','OBSERVED','Разрезы регионального приёма ВПО-1 с подтверждённым кодом и одинаковыми классификаторами; OD по программе не наблюдается')

# СПО: реальные программные коды + оригинальные названия индикаторов без семантического угадывания.
spo=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("region_name" AS VARCHAR) AS "region", CAST("specialty_code" AS VARCHAR) AS "code", CAST("specialty_name" AS VARCHAR) AS "specialty", CAST("indicator_name" AS VARCHAR) AS "indicator", CAST("indicator_unit" AS VARCHAR) AS "unit", COUNT(*)::BIGINT AS "n_rows", MIN(TRY_CAST("region_value" AS DOUBLE)) AS "low", MAX(TRY_CAST("region_value" AS DOUBLE)) AS "high" FROM read_parquet(?) WHERE "specialty_code" IS NOT NULL AND "year" IS NOT NULL GROUP BY 1,2,3,4,5,6""",[str(get_source(SPO_REG))])
spo['code']=clean_code(spo.code)
spo['valid']=spo.n_rows.eq(1)&spo.low.notna()&spo.low.eq(spo.high)
put('СПО_региональные_спец_индикаторы',spo)
put('СПО_словарь_программ',spo.groupby(['indicator','unit'],dropna=False).agg(
 наблюдений=('code','size'),число_кодов=('code','nunique'),лет=('year','nunique')).reset_index()
    .sort_values('наблюдений',ascending=False))
FINDINGS.append({'модуль':'ML-5.4','вывод':'Сравнение географии специальностей возможно по наблюдаемым точным разрезам',
 'расчёт':f'ВПО-1: {len(PROGRAM):,} точных региональных страт без кодов Всего; СПО: {len(spo):,} регистрируемых ячеек',
 'ограничение':'Специальность в регионе назначения не характеризует прежний регион обучения конкретного студента'})
mark_stage('Специальности')

## 5А. Специализация территорий и устойчивость структуры приёма

Для ВО изучаем разнообразие и концентрацию **внутри идентичных уровней образования, форм, типов и категорий приёма**. Для СПО каждый исходный `indicator_name` и `indicator_unit` анализируется отдельно. Сложение разных показателей и создание географии OD по специальности запрещены.

In [ ]:
VPO_DIV=[];VPO_JACC=[];SPO_DIV=[]
if len(PROGRAM):
    family=['year','region_key','level','form','admission_type','university_type']
    src=PROGRAM.loc[PROGRAM.valid & PROGRAM.low.gt(0)].copy()
    for k,g in src.groupby(family,dropna=False):
        codes=g.groupby('code',dropna=False)['low'].sum()
        if len(codes)<3 or codes.sum()<=0:continue
        sh=codes/codes.sum();VPO_DIV.append(dict(zip(family,k)) | {
          'кодовых_значений':len(codes),'значение_только_наблюдаемых_кодов':float(codes.sum()),
          'HHI':float((sh**2).sum()),'эффективное_число_кодов':float(1/(sh**2).sum()),
          'доля_первых_5':float(codes.nlargest(5).sum()/codes.sum()),'энтропия':float(-(sh*np.log(sh)).sum()),
          'ограничение':'Доля только внутри идентичной отчетной страты; межгодовые классификаторы не реконструируются'})
    timekey=family[1:]
    by_year={}
    for (year,*dims),g in src.groupby(family,dropna=False):
        key=tuple(str(v) for v in dims)
        by_year.setdefault(key,{})[int(year)]=set(g.loc[g.low.gt(0),'code'].dropna().astype(str))
    for k,d in by_year.items():
        for y0,y1 in zip(sorted(d)[:-1],sorted(d)[1:]):
            a,b=d[y0],d[y1]
            VPO_JACC.append(dict(zip(timekey,k)) | {'год_от':y0,'год_до':y1,
             'кодов_раньше':len(a),'кодов_позже':len(b),
             'jaccard_собственных_кодов':len(a&b)/len(a|b) if a|b else np.nan,
             'сравнимость':'Совпадение исходных кодов, без недоказанного исторического crosswalk'})
put('ВПО_разнообразие_профилей',pd.DataFrame(VPO_DIV))
put('ВПО_динамика_кодов',pd.DataFrame(VPO_JACC))
if len(spo):
    u=spo.loc[spo.valid & spo.low.gt(0) & spo.code.notna()].copy()
    for k,g in u.groupby(['year','region','indicator','unit'],dropna=False):
        code_sum=g.groupby('code')['low'].sum()
        if len(code_sum)<3 or code_sum.sum()<=0:continue
        shares=code_sum/code_sum.sum()
        SPO_DIV.append(dict(zip(['year','region','indicator','unit'],k)) | {
         'кодов_наблюдаемых':len(code_sum),'HHI':float(np.sum(shares**2)),
         'эффективное_число_кодов':float(1/np.sum(shares**2)),
         'сумма_наблюдаемого_показателя':float(code_sum.sum()),
         'ограничение':'Один и тот же показатель, единица и год; не сопоставлять с OD без связки'})
put('СПО_разнообразие_программ',pd.DataFrame(SPO_DIV))
FINDINGS.append({'модуль':'ML-5.4','вывод':'Измерено территориальное разнообразие программ и устойчивость кодов в ВО/СПО',
 'расчёт':f'{len(VPO_DIV)} однородных разрезов ВО, {len(SPO_DIV)} разрезов СПО, {len(VPO_JACC)} межгодовых сравнений кодов',
 'ограничение':'Изменения показателей в разных классификаторах не объединены без источника соответствий'})
mark_stage('Профили программ')

## 6. ML-5.5. Занятость, зарплата, выпуск по E04/E05
Показатели относятся к реально наблюдаемым *отчётным стратам* и не дают географии трудоустройства. Приводятся распределения и сравнения внутри одинаковых периодов/групп, без реконструкции индивидуальных траекторий и коэффициента закрепления. Среднее по отчётным строкам явно помечается как среднее **ячеек**, не доля всех выпускников.

In [ ]:
G4=get_source(E04);G5=get_source(E05)
query4="""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("object_name" AS VARCHAR) AS "region", CAST("object_level" AS VARCHAR) AS "object_level", CAST("university" AS VARCHAR) AS "university", CAST("gender" AS VARCHAR) AS "gender", CAST("education_level" AS VARCHAR) AS "level", TRY_CAST("count_graduate" AS DOUBLE) AS "graduates", TRY_CAST("percent_employed" AS DOUBLE) AS "employed_pct", TRY_CAST("average_salary_fact_med" AS DOUBLE) AS "salary_median", TRY_CAST("average_salary_fact_avg" AS DOUBLE) AS "salary_mean" FROM read_parquet(?)"""
q4=qry(query4,[str(G4)])
q4=q4[q4.year.notna() & q4.graduates.ge(0)].copy()
q4['region_key']=clean_name(q4.region)
q4['rate_valid']=q4.employed_pct.between(0,100)
q4['salary_valid']=q4.salary_median.ge(0) & q4.salary_median.notna()
key4=['year','region_key','university','gender','level']
amb=q4.duplicated(key4,keep=False)
if amb.any():QUARANTINE.append({'table':'E04','problem':'Повторная стратa занятости (не суммировать)','count':int(amb.sum())})
q4=q4[~amb].copy()
E04_CELLS=put('Выпускники_E04_страты',q4)

q5=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("object_name" AS VARCHAR) AS "region", CAST("object_level" AS VARCHAR) AS "object_level", CAST("study_area" AS VARCHAR) AS "study_area", CAST("specialty_code" AS VARCHAR) AS "code", CAST("specialty" AS VARCHAR) AS "specialty", CAST("education_level" AS VARCHAR) AS "level", CAST("gender" AS VARCHAR) AS "gender", TRY_CAST("count_graduate" AS DOUBLE) AS "graduates", TRY_CAST("percent_employed" AS DOUBLE) AS "employed_pct", TRY_CAST("average_salary_fact_med" AS DOUBLE) AS "salary_median" FROM read_parquet(?)""",[str(G5)])
q5=q5[q5.year.notna()&q5.graduates.ge(0)].copy()
q5['region_key']=clean_name(q5.region);q5['code']=clean_code(q5.code)
q5['rate_valid']=q5.employed_pct.between(0,100)
q5['salary_valid']=q5.salary_median.ge(0)&q5.salary_median.notna()
key5=['year','region_key','study_area','code','level','gender']
amb5=q5.duplicated(key5,keep=False)
if amb5.any():QUARANTINE.append({'table':'E05','problem':'Повторная специальность×год×уровень×пол','count':int(amb5.sum())})
q5=q5[~amb5].copy()
E05_CELLS=put('Выпускники_E05_страты',q5)

# Только статистика распределения ячеек, не общий процент трудоустроенных среди всех выпускников.
def outcome_profile(f,group,origin):
    f=f.loc[f.rate_valid | f.salary_valid].copy()
    if not len(f):return pd.DataFrame()
    z=f.groupby(group,dropna=False).agg(строк_отчета=('graduates','size'),
        наблюдаемая_медиана_доли_занятых=('employed_pct','median'),
        наблюдаемая_медиана_зарплаты=('salary_median','median'),
        q25_зарплаты=('salary_median',lambda x:x.quantile(.25)),
        q75_зарплаты=('salary_median',lambda x:x.quantile(.75)),
        валидных_занятость=('rate_valid','sum'),валидных_зарплата=('salary_valid','sum')).reset_index()
    z['источник']=origin;z['определение']='Медиана отчётных ячеек; не взвешенная доля всех выпускников'
    return z

if len(E04_CELLS):
    put('E04_регионы_и_годы',outcome_profile(
        E04_CELLS[E04_CELLS.object_level.astype(str).str.contains('регион|субъект',case=False,regex=True,na=False)],
        ['year','region_key','level','gender'],'E04'))
if len(E05_CELLS):
    put('E05_программы_итоги',outcome_profile(
        E05_CELLS[E05_CELLS.object_level.astype(str).str.contains('регион|субъект',case=False,regex=True,na=False)],
        ['year','study_area','level','gender'],'E05'))
    topout=E05_CELLS.loc[E05_CELLS.rate_valid&E05_CELLS.salary_valid]
    topout=topout.sort_values(['year','salary_median','region_key','code','study_area','level','gender'],ascending=[False,False,True,True,True,True,True],kind='mergesort')
    put('E05_детальные_зарплаты',topout)
issue('ML-5.5','OBSERVED','Проанализированы реальные доли занятости и зарплаты E04/E05; география работы и закрепление не наблюдаются')
FINDINGS.append({'модуль':'ML-5.5','вывод':'Разделены занятость и территориальное закрепление',
  'расчёт':f'Наблюдаемых страт E04 {len(E04_CELLS):,}, E05 {len(E05_CELLS):,}',
  'ограничение':'Нет документированной связи выпускник → фактический регион работы'})
mark_stage('Выпускники')

## 6А. Проверка переносимости моделей занятости и зарплат E04 на незнакомые регионы
Это **поперечный межрегиональный тест**, а не прогноз закрепления выпускников и не предсказание региона работы. На отложенных целиком регионах проверяются средняя обучающей выборки, Ridge и случайный лес. Доступны только уровень образования, пол, численность выпуска и год из той же E04, без скрытой экономической реконструкции. Модели дополняют анализ отчётных показателей; их точность не подменяет вопросы образовательной мобильности.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
GRAD_MODELS=[];GRAD_PRED=pd.DataFrame();GRAD_METRICS=pd.DataFrame();GRAD_REPLAY=[]
base=E04_CELLS[E04_CELLS.object_level.astype(str).str.contains('регион|субъект',case=False,regex=True,na=False)].copy()
base=base[base.graduates.gt(0)&base.year.notna()&base.region_key.notna()].copy()
if len(base)>=200 and base.region_key.nunique()>=10:
    splitter=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=CFG.SEED)
    itrain,itest=next(splitter.split(base,groups=base.region_key))
    train=base.iloc[itrain].copy();test=base.iloc[itest].copy()
    assert set(train.region_key).isdisjoint(test.region_key)
    def make_x(df,levels=None):
        x=df[['graduates','year','gender','level']].copy()
        x['log_graduates']=np.log1p(x.graduates.clip(lower=0));x=x.drop(columns='graduates')
        x['gender']=x.gender.fillna('__MISSING__').astype(str)
        x['level']=x.level.fillna('__MISSING__').astype(str)
        x=pd.get_dummies(x,columns=['gender','level'],dtype=float)
        if levels is not None:x=x.reindex(columns=levels,fill_value=0.)
        return x
    Xtr=make_x(train);Xte=make_x(test,levels=Xtr.columns)
    predicted=[];metrics=[]
    for target,valid_col in [('employed_pct','rate_valid'),('salary_median','salary_valid')]:
        tr=train[train[valid_col]].copy();te=test[test[valid_col]].copy()
        if len(tr)<150 or len(te)<50:continue
        X=make_x(tr).reindex(columns=Xtr.columns,fill_value=0.)
        Z=make_x(te).reindex(columns=Xtr.columns,fill_value=0.)
        actual=te[target].astype(float).to_numpy()
        methods={'train_mean':np.full(len(te),float(tr[target].mean()))}
        for kind in ('ridge','random_forest'):
            model=Ridge(alpha=50.) if kind=='ridge' else RandomForestRegressor(
                n_estimators=100,max_depth=9,min_samples_leaf=5,random_state=CFG.SEED,n_jobs=CFG.WORKERS)
            model.fit(X,tr[target].astype(float))
            pred=model.predict(Z)
            pred=np.clip(pred,0,100) if target=='employed_pct' else np.maximum(0,pred)
            methods[kind]=pred
            path=ROOT/'models'/f'graduate_{target}_{kind}.joblib'
            joblib.dump({'estimator':model,'columns':list(X.columns)},path,compress=3)
            restored=joblib.load(path)
            again=restored['estimator'].predict(Z[restored['columns']])
            again=np.clip(again,0,100) if target=='employed_pct' else np.maximum(0,again)
            success=bool(np.allclose(pred,again,atol=1e-8,rtol=1e-9))
            if not success:raise RuntimeError('Graduate model replay failed')
            GRAD_REPLAY.append({'target':target,'model':kind,'sha256':sha256(path),'independent_replay':success})
        for kind,pred in methods.items():
            for obs_id,region,year,true_val,pred_val in zip(te.index,te.region_key,te.year,actual,pred):
                predicted.append({'observation_id':int(obs_id),'region_key':region,'year':year,'target':target,'model':kind,
                  'actual':float(true_val),'prediction':float(pred_val),'protocol':'REGIONAL_HOLDOUT_CROSS_SECTION'})
            metrics.append({'target':target,'model':kind,'n_test':len(te),
              'n_regions_heldout':te.region_key.nunique(),
              'MAE':float(mean_absolute_error(actual,pred)),
              'RMSE':float(np.sqrt(mean_squared_error(actual,pred)))})
    GRAD_PRED=put('Занятость_тестовые_прогнозы',pd.DataFrame(predicted))
    GRAD_METRICS=put('Занятость_тестовые_метрики',pd.DataFrame(metrics))
    put('Занятость_проверка_моделей',pd.DataFrame(GRAD_REPLAY))
    issue('ML-5.5','CROSS_SECTIONAL_REGION_HOLDOUT',
          f'Отложены {test.region_key.nunique()} регионов; не ex-ante и не география работы')
else:
    issue('ML-5.5','NOT_FEASIBLE_HOLDOUT','Недостаточно однозначных региональных страт E04 для теста')
mark_stage('Занятость')

## 6В. Построчное независимое подтверждение прогнозов выпускников

In [ ]:
# Независимые построчные метрики E04 для всех сохраненных предсказаний.
G_AUDIT=[]
if len(GRAD_PRED) and len(GRAD_METRICS):
 for (target,model),g in GRAD_PRED.groupby(['target','model']):
  ref=GRAD_METRICS.loc[(GRAD_METRICS.target==target)&(GRAD_METRICS.model==model)]
  mae=float(np.abs(g.actual-g.prediction).mean());rmse=float(np.sqrt(np.mean((g.actual-g.prediction)**2)))
  ok=len(ref)==1 and int(ref.iloc[0].n_test)==len(g)
  G_AUDIT.append({'target':target,'model':model,'n':len(g),'MAE_recalc':mae,'RMSE_recalc':rmse,
    'MAE_match':bool(ok and abs(float(ref.iloc[0].MAE)-mae)<1e-8),
    'RMSE_match':bool(ok and abs(float(ref.iloc[0].RMSE)-rmse)<1e-8)})
GRAD_AUDIT=put('Занятость_воспроизводимость_метрик',pd.DataFrame(G_AUDIT))
if len(GRAD_AUDIT) and not (GRAD_AUDIT.MAE_match.all() and GRAD_AUDIT.RMSE_match.all()):
 raise AssertionError('E04: исходные строки не воспроизводят отчётные метрики')

## 6Б. Сравнимые региональные исходы выпускников и парная проверка прогнозов

Сравниваем реальные E04/E05 в их исходных единицах, без «закрепления» и без смешения объединённых категорий по полу/уровню. Региональные модели оцениваются на одних и тех же тестовых строках, ошибки агрегируются по фактически отложенным регионам.

In [ ]:
E05_WAGES=[];E04_STABILITY=[];GRAD_REGION_TEST=[]
if len(E05_CELLS):
    s=E05_CELLS.loc[E05_CELLS.rate_valid & E05_CELLS.salary_valid & E05_CELLS.graduates.ge(10)].copy()
    for k,g in s.groupby(['year','study_area','level','gender'],dropna=False):
        if len(g)<8:continue
        r,p=spearmanr(g.salary_median,g.employed_pct)
        E05_WAGES.append(dict(zip(['year','study_area','level','gender'],k)) | {
          'число_отчётных_ячеек':len(g),'число_регионов':int(g.region_key.nunique()),
          'медиана_зарплаты_среди_отчётных_ячеек':float(g.salary_median.median()),
          'межквартильный_размах_зарплаты':float(g.salary_median.quantile(.75)-g.salary_median.quantile(.25)),
          'медиана_доли_занятых':float(g.employed_pct.median()),
          'Spearman_зарплата_занятость':float(r) if np.isfinite(r) else np.nan,
          'p_описательная':float(p) if np.isfinite(p) else np.nan,
          'ограничение':'Корреляция внутри отчётных ячеек; не местонахождение работы выпускника'})
put('E05_зарплаты_по_профилям',pd.DataFrame(E05_WAGES))
if len(E04_CELLS):
    xx=E04_CELLS.loc[E04_CELLS.rate_valid & E04_CELLS.salary_valid & E04_CELLS.graduates.ge(10)].copy()
    for k,g in xx.groupby(['year','region_key'],dropna=False):
        if len(g)<5:continue
        E04_STABILITY.append({'year':k[0],'region_key':k[1], 'число_отчётных_ячеек':len(g),
          'медиана_занятости':float(g.employed_pct.median()),'медиана_зарплаты':float(g.salary_median.median()),
          'q10_зарплаты':float(g.salary_median.quantile(.1)),'q90_зарплаты':float(g.salary_median.quantile(.9)),
          'отношение_p90_p10':float(g.salary_median.quantile(.9)/g.salary_median.quantile(.1)) if g.salary_median.quantile(.1)>0 else np.nan})
put('E04_устойчивость_регионов',pd.DataFrame(E04_STABILITY))
if len(GRAD_PRED) and 'observation_id' in GRAD_PRED:
    for target in GRAD_PRED.target.unique():
        g=GRAD_PRED.loc[GRAD_PRED.target.eq(target)].copy()
        pivot=g.pivot(index=['observation_id','region_key'],columns='model',values='prediction')
        truth=g.drop_duplicates('observation_id').set_index('observation_id')['actual']
        for region,x in pivot.groupby(level='region_key'):
            ids=x.index.get_level_values('observation_id')
            obs=truth.reindex(ids).to_numpy(float)
            if len(obs)==0:continue
            row={'target':target,'регион':region,'n':len(obs)}
            for model in ['train_mean','ridge','random_forest']:
                if model in x:row['MAE_'+model]=float(np.abs(obs-x[model].to_numpy(float)).mean())
            GRAD_REGION_TEST.append(row)
put('Занятость_сравнение_по_регионам',pd.DataFrame(GRAD_REGION_TEST))
FINDINGS.append({'модуль':'ML-5.5','вывод':'Оценены фактические различия занятости и зарплат по регионам и направлениям',
 'расчёт':f'{len(E05_WAGES)} сопоставимых профилей E05, {len(E04_STABILITY)} регионально-временных профилей E04',
 'ограничение':'Нет координат или региона трудоустройства выпускника, эффект закрепления не определён'})

## 7. ML-5.6. Региональная сеть и устойчивость образовательных маршрутов
Сеть — направленный взвешенный граф **положительных наблюдаемых** OD СПО. Межгодовые сравнения выполняются дважды: по фактически наблюдавшимся положительным связям и на постоянном покрытии. Нули учитываются как зарегистрированные нули, пропущенные пары никогда не достраиваются. Центры, коридоры, изменение роли региона и устойчивость сообществ рассчитываются, а не декларируются.

In [ ]:
from sklearn.metrics import adjusted_rand_score
NET_ROWS=[]; COMMUNITY=[]; NETWORK_PERIODS=[];GRAPH_BY_YEAR={};COMM_BY_YEAR={}
for universe_name,edges in [('НАБЛЮДАЕМЫЕ',INTER),('ПОСТОЯННОЕ_ПОКРЫТИЕ',FIXED)]:
    if edges.empty:continue
    prev_positive=None;prev_comm=None
    for year in sorted(edges.year.unique()):
        f=edges.loc[edges.year.eq(year) & edges.value.gt(0)]
        G=nx.DiGraph()
        for r in f.itertuples(index=False):G.add_edge(str(r.origin_code),str(r.destination_code),weight=float(r.value))
        if G.number_of_edges()==0:continue
        pr=nx.pagerank(G,weight='weight')
        # При объединении двух направлений суммируем их веса, а не перезаписываем одно другим.
        U=nx.Graph()
        for u,v,data in G.edges(data=True):
            previous=U[u][v]['weight'] if U.has_edge(u,v) else 0.
            U.add_edge(u,v,weight=previous+float(data['weight']))
        groups=list(nx.community.greedy_modularity_communities(U,weight='weight')) if U.number_of_edges() else [set(G.nodes)]
        comm={n:i for i,group in enumerate(groups) for n in group}
        outgoing=dict(G.out_degree(weight='weight'));incoming=dict(G.in_degree(weight='weight'))
        partners_out=dict(G.out_degree());partners_in=dict(G.in_degree())
        for node in sorted(G.nodes):
            NET_ROWS.append({'год':int(year),'совокупность':universe_name,'region_code':node,
             'регион':REG_MAP.get(node,node),'исходящая_сила':float(outgoing.get(node,0)),
             'входящая_сила':float(incoming.get(node,0)),
             'наблюдаемый_сетевой_баланс':float(incoming.get(node,0)-outgoing.get(node,0)),
             'исходящих_партнёров':int(partners_out.get(node,0)),'входящих_партнёров':int(partners_in.get(node,0)),
             'pagerank':float(pr.get(node,0)),'сообщество_года':comm.get(node)})
        COMMUNITY.extend([{'год':int(year),'совокупность':universe_name,'region_code':n,
          'регион':REG_MAP.get(n,n),'сообщество':k} for n,k in comm.items()])
        positive=set(G.edges());jaccard=(len(positive&prev_positive)/len(positive|prev_positive)) if prev_positive is not None and (positive|prev_positive) else np.nan
        common_nodes=sorted(set(comm)&set(prev_comm)) if prev_comm is not None else []
        stability=(adjusted_rand_score([prev_comm[n] for n in common_nodes],[comm[n] for n in common_nodes])
                   if len(common_nodes)>=4 else np.nan)
        total=sum(d['weight'] for _,_,d in G.edges(data=True))
        mutual=sum(1 for u,v in G.edges if G.has_edge(v,u))
        NETWORK_PERIODS.append({'год':int(year),'совокупность':universe_name,
         'регионов':G.number_of_nodes(),'положительных_рёбер':G.number_of_edges(),
         'наблюдаемая_сумма':float(total),'сообществ':len(groups),
         'доля_взаимных_дуг':mutual/G.number_of_edges(),
         'jaccard_положительных_связей_к_прошлому':jaccard,
         'ARI_сообществ_общих_регионов':stability,'сопоставимых_узлов_для_ARI':len(common_nodes)})
        if universe_name=='ПОСТОЯННОЕ_ПОКРЫТИЕ':GRAPH_BY_YEAR[int(year)]=G;COMM_BY_YEAR[int(year)]=comm
        prev_positive=positive;prev_comm=comm

NETWORK_NODES=put('Сеть_регионы_по_годам',pd.DataFrame(NET_ROWS))
NETWORK_YEARS=put('Сеть_устойчивость_лет',pd.DataFrame(NETWORK_PERIODS))
put('Сеть_сообщества',pd.DataFrame(COMMUNITY))
if len(NETWORK_NODES):
    latestnet=NETWORK_NODES[NETWORK_NODES.год.eq(LATEST) & NETWORK_NODES.совокупность.eq('ПОСТОЯННОЕ_ПОКРЫТИЕ')]
    if latestnet.empty:latestnet=NETWORK_NODES[NETWORK_NODES.год.eq(LATEST)]
    put('Образовательные_центры',latestnet.sort_values('входящая_сила',ascending=False).head(50))
issue('ML-5.6','OBSERVED','Построены действительные центры, веса, направленные сообщества и межгодовая устойчивость; неудачная прогностическая модель не отменяет сетевую аналитику')
FINDINGS.append({'модуль':'ML-5.6','вывод':'Измерены изменения структуры образовательной сети и её центров',
 'расчёт':f'{len(NETWORK_YEARS)} записей по годам и совокупностям; ARI сообществ и Jaccard связей',
 'ограничение':'Смена покрытия E01 и нулевые записи отдельно учитываются'})
mark_stage('Сети')

## 7Б. Устойчивость образовательной сети к исключению сильнейших маршрутов

Для каждого года и постоянной совокупности удаляем крупнейшие 1% и 5% положительных связей. Отслеживаем изменение PageRank и классификации сообществ (ARI). Это проверка чувствительности сетевого результата, а не эксперимент «что случится при запрете поступления».

In [ ]:
NET_SENS=[];NET_CENTERS=[];NET_TRANSITION=[]
for year,G in GRAPH_BY_YEAR.items():
    if G.number_of_edges()<10:continue
    original=nx.pagerank(G,weight='weight');origcomm=COMM_BY_YEAR[year]
    weight_in=sorted(dict(G.in_degree(weight='weight')).items(),key=lambda x:(-x[1],x[0]))
    tot=sum(v for _,v in weight_in)
    NET_CENTERS.append({'год':int(year),'узлов':G.number_of_nodes(),'положительных_маршрутов':G.number_of_edges(),
      'доля_двух_крупнейших_центров':sum(v for _,v in weight_in[:2])/tot if tot else np.nan,
      'доля_пяти_центров':sum(v for _,v in weight_in[:5])/tot if tot else np.nan,
      'доля_десяти_центров':sum(v for _,v in weight_in[:10])/tot if tot else np.nan,
      'крупнейшие_центры':'; '.join(f'{REG_MAP.get(u,u)} ({v:.0f})' for u,v in weight_in[:5])})
    ranked=sorted(G.edges(data=True),key=lambda x:(-x[2]['weight'],x[0],x[1]))
    for frac in [0.01,0.05]:
        rm=max(1,int(np.ceil(frac*len(ranked))))
        H=G.copy();H.remove_edges_from([(a,b) for a,b,_ in ranked[:rm]])
        H.add_nodes_from(G.nodes)
        pr=nx.pagerank(H,weight='weight')
        U=nx.Graph()
        U.add_nodes_from(H.nodes)
        for a,b,attr in H.edges(data=True):
            w=U[a][b]['weight'] if U.has_edge(a,b) else 0.
            U.add_edge(a,b,weight=w+float(attr['weight']))
        groups=list(nx.community.greedy_modularity_communities(U,weight='weight')) if U.number_of_edges() else [set(U.nodes)]
        revised={node:i for i,grp in enumerate(groups) for node in grp}
        nodes=sorted(G.nodes)
        rho,p=spearmanr([original[n] for n in nodes],[pr[n] for n in nodes])
        NET_SENS.append({'год':int(year),'удалено_доля_связей':frac,'удалено_рёбер':rm,
          'доля_исключённого_наблюдаемого_веса':sum(float(z[2]['weight']) for z in ranked[:rm])/tot if tot else np.nan,
          'spearman_PageRank_до_после':float(rho) if np.isfinite(rho) else np.nan,
          'ARI_сообществ_до_после':float(adjusted_rand_score([origcomm[n] for n in nodes],[revised[n] for n in nodes])),
          'n_регионов':len(nodes),'смысл':'Устойчивость наблюдаемой структуры к удалению крупнейших связей'})
if len(NETWORK_NODES):
    z=NETWORK_NODES.loc[NETWORK_NODES['совокупность'].eq('ПОСТОЯННОЕ_ПОКРЫТИЕ')].copy()
    for y0,y1 in zip(YEARS[:-1],YEARS[1:]):
        a=z.loc[z['год'].eq(y0),['region_code','pagerank','входящая_сила']]
        b=z.loc[z['год'].eq(y1),['region_code','pagerank','входящая_сила']]
        joint=a.merge(b,on='region_code',suffixes=('_prev','_next'),validate='one_to_one')
        if len(joint)<5:continue
        joint['pagerank_rank_prev']=joint.pagerank_prev.rank(ascending=False,method='min')
        joint['pagerank_rank_next']=joint.pagerank_next.rank(ascending=False,method='min')
        joint['изменение_места']=joint.pagerank_rank_prev-joint.pagerank_rank_next
        joint['from_year']=int(y0);joint['to_year']=int(y1)
        NET_TRANSITION.append(joint)
put('Сеть_чувствительность',pd.DataFrame(NET_SENS))
put('Сеть_концентрация_центров',pd.DataFrame(NET_CENTERS))
put('Сеть_переходы_центров',pd.concat(NET_TRANSITION,ignore_index=True) if NET_TRANSITION else pd.DataFrame())
FINDINGS.append({'модуль':'ML-5.6','вывод':'Проверена устойчивость региональной сетевой структуры',
 'расчёт':f'{len(NET_SENS)} тестов удаления крупнейших маршрутов; {len(NET_CENTERS)} годовых оценок концентрации центров',
 'ограничение':'Sensitivity-анализ не является причинным эффектом инфраструктурных вмешательств'})
mark_stage('Чувствительность сетей')

## 7А. ML-5.6. Ретроспективный прогноз сетевых характеристик на неизменном покрытии

Отдельный эксперимент по **наблюдаемым** центральностям и связям, не повторение OD-прогноза под новым названием. На каждом годовом срезе используются только узлы, присутствующие во всех трёх последовательных годах: отсутствующие наблюдения не заменяются искусственными нулями. Признаки берутся из двух предшествующих лет. Сравниваются инерционный прогноз, среднее двух лет, Ridge и случайный лес. 2024 — DEV, 2025 — TEST при достаточном охвате. Изменение рёбер, центральности и сил трактуется как характеристика наблюдаемой сети, а не миграция населения. Недостаток покрытия обозначается `NOT_FEASIBLE`.

In [ ]:
# Исходные сетевые показатели V8.0.1 сохранены; добавлен отдельный воспроизводимый прогноз.
from sklearn.ensemble import RandomForestRegressor
NET_PRED=pd.DataFrame();NET_METRICS=pd.DataFrame();NET_REPLAY=[];NET_CANARY=[]
NETWORK_TARGETS=['pagerank','входящая_сила','исходящая_сила',
                 'входящих_партнёров','исходящих_партнёров']

NET_OBS=NETWORK_NODES.loc[NETWORK_NODES['совокупность'].eq('ПОСТОЯННОЕ_ПОКРЫТИЕ')].copy()
if NET_OBS.duplicated(['год','region_code']).any():raise RuntimeError('ML-5.6: неуникальные узлы/годы')
NET_YEARS=sorted(NET_OBS['год'].dropna().astype(int).unique())

# Признаки каждого target строятся только из предшествующей наблюдаемой сети.
def network_lag_panel(node_table,target):
    # Уникальный набор столбцов: target может совпадать с одним из предикторов.
    base=['год','region_code','входящая_сила','исходящая_сила',
          'входящих_партнёров','исходящих_партнёров','pagerank']
    source=node_table[base].copy()
    chunks=[]
    for ty in sorted(source['год'].unique()):
        ty=int(ty)
        prev=source.loc[source['год'].eq(ty-1)].copy()
        before=source.loc[source['год'].eq(ty-2),['region_code',target]].copy()
        actual=source.loc[source['год'].eq(ty),['region_code',target]].copy()
        if prev.empty or before.empty or actual.empty:continue
        prev['lag1']=prev[target]
        one=prev[['region_code','lag1','входящая_сила','исходящая_сила',
                  'входящих_партнёров','исходящих_партнёров','pagerank']].copy()
        one.columns=['region_code','lag1','prev_in','prev_out','prev_in_degree','prev_out_degree','prev_pagerank']
        before.columns=['region_code','lag2'];actual.columns=['region_code','actual']
        z=actual.merge(one,on='region_code',how='inner',validate='one_to_one')
        z=z.merge(before,on='region_code',how='inner',validate='one_to_one')
        z['target_year']=ty;z['information_cutoff']=ty-1
        chunks.append(z)
    return pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame()

NET_FEATURES=['lag1','lag2','prev_in','prev_out','prev_in_degree','prev_out_degree','prev_pagerank']
NET_MODELS=[];NET_PRED_ROWS=[]
if CFG.RUN_FORECAST:
    for target in NETWORK_TARGETS:
        hist=network_lag_panel(NET_OBS,target)
        if hist.empty or hist['target_year'].nunique()<2:
            NET_MODELS.append({'target':target,'status':'NOT_FEASIBLE','why':'Нет ≥2 лет с двумя предыдущими сетевыми срезами'})
            continue
        if not hist['information_cutoff'].lt(hist['target_year']).all():
            raise RuntimeError('ML-5.6: будущий год попал в сетевые признаки')
        # Изменение только целевых значений года TEST не меняет его исторические признаки.
        testyear=2025 if 2025 in hist['target_year'].unique() else int(hist['target_year'].max())
        modified=NET_OBS.copy()
        modified.loc[modified['год'].eq(testyear),target]=modified.loc[modified['год'].eq(testyear),target]+1e4
        h2=network_lag_panel(modified,target)
        a=hist.loc[hist['target_year'].eq(testyear)].sort_values('region_code')
        b=h2.loc[h2['target_year'].eq(testyear)].sort_values('region_code')
        canary=bool(len(a)==len(b) and np.allclose(a[NET_FEATURES],b[NET_FEATURES]))
        NET_CANARY.append({'target':target,'year':testyear,'n':len(a),'past_features_unaffected':canary})
        if not canary:raise RuntimeError('ML-5.6: сетевой future-mutation canary провален: '+target)
        # DEV 2024 предшествует финальному TEST 2025; ни один TEST не участвует в выборе модели.
        prior_years=sorted(y for y in hist['target_year'].unique() if y<testyear)
        evaluation_years=([int(prior_years[-1])] if prior_years else [])+[testyear]
        for ev_year in evaluation_years:
            protocol='TEMPORAL_TEST' if ev_year==testyear else 'TEMPORAL_DEV'
            train=hist.loc[hist['target_year'].lt(ev_year)].copy()
            test=hist.loc[hist['target_year'].eq(ev_year)].copy()
            if len(train)<40 or len(test)<20 or train['target_year'].nunique()<1:
                NET_MODELS.append({'target':target,'protocol':protocol,'status':'NOT_FEASIBLE',
                                  'why':f'train={len(train)} test={len(test)}'})
                continue
            outputs={
                'persistence':test['lag1'].to_numpy(float),
                'rolling_mean_2':test[['lag1','lag2']].mean(axis=1).to_numpy(float),
            }
            for kind in ('ridge','random_forest'):
                m=(Ridge(alpha=10.0) if kind=='ridge' else RandomForestRegressor(
                    n_estimators=60,max_depth=7,min_samples_leaf=3,random_state=CFG.SEED,n_jobs=CFG.WORKERS))
                m.fit(train[NET_FEATURES],train['actual'])
                predicted=np.maximum(0,m.predict(test[NET_FEATURES]))
                path=ROOT/'models'/f'NETWORK_{target}_{kind}_{ev_year}.joblib'
                joblib.dump({'model':m,'features':NET_FEATURES,'target':target,
                             'test_year':ev_year,'protocol':protocol},path,compress=3)
                replay=np.maximum(0,joblib.load(path)['model'].predict(test[NET_FEATURES]))
                identical=bool(np.allclose(predicted,replay,atol=1e-9,rtol=1e-9))
                NET_REPLAY.append({'target':target,'model':kind,'test_year':ev_year,
                                   'protocol':protocol,'identical':identical,'sha256':sha256(path)})
                if not identical:raise RuntimeError('ML-5.6: сетевые предсказания не повторились')
                outputs[kind]=predicted
            for kind,yhat in outputs.items():
                yy=test['actual'].to_numpy(float);pred=np.maximum(0,np.asarray(yhat,dtype=float))
                pe=test[['region_code','target_year','information_cutoff','actual']].copy()
                pe['target']=target;pe['model']=kind;pe['protocol']=protocol;pe['prediction']=pred
                pe['error_abs']=np.abs(yy-pred);NET_PRED_ROWS.append(pe)
                NET_MODELS.append({'target':target,'model':kind,'protocol':protocol,'test_year':ev_year,
                                   'n_train':len(train),'n_test':len(test),
                                   'MAE':float(np.mean(np.abs(yy-pred))),
                                   'RMSE':float(np.sqrt(np.mean((yy-pred)**2))),
                                   'status':'RETROSPECTIVE_TEST','source':'E01_FIXED_OBSERVED_NETWORK'})
            baseline=next(r for r in NET_MODELS if r.get('target')==target
                          and r.get('protocol')==protocol and r.get('model')=='persistence')
            for row in NET_MODELS:
                if row.get('target')==target and row.get('protocol')==protocol and row.get('status')=='RETROSPECTIVE_TEST':
                    row['MAE_improvement_vs_persistence']=float(baseline['MAE']-row['MAE'])

NET_PRED=put('Сеть_прогноз_наблюдения',pd.concat(NET_PRED_ROWS,ignore_index=True) if NET_PRED_ROWS else pd.DataFrame())
NET_METRICS=put('Сеть_прогноз_метрики',pd.DataFrame(NET_MODELS))
put('Сеть_прогноз_future_canary',pd.DataFrame(NET_CANARY))
put('Сеть_прогноз_повторение',pd.DataFrame(NET_REPLAY))
if NET_PRED.empty:
    issue('ML-5.6','NOT_FEASIBLE_NETWORK_FORECAST','Недостаточная временная история для независимого прогноза сетевых характеристик')
else:
    issue('ML-5.6','NETWORK_FORECAST_RETROSPECTIVE',
          f'Отдельные DEV и TEST-прогнозы по {NET_METRICS.target.nunique()} наблюдаемым характеристикам сети, только на стабильном покрытии')
    FINDINGS.append({'модуль':'ML-5.6','вывод':'Проверена прогнозируемость сетевых характеристик',
       'расчёт':f'{NET_METRICS.target.nunique()} метрик сети; сравнение persistence, rolling mean, Ridge, RF',
       'ограничение':'Ретроспективная оценка, только общие наблюдаемые узлы и годы'})
print('ML-5.6: сетевой прогноз',len(NET_PRED),'предсказаний;',len(NET_METRICS),'метрик; replay:',len(NET_REPLAY))

mark_stage('Прогноз сети')

## 8. ML-5.7. Муниципальная образовательная статистика и граница возможностей
Полный потоковый аудит образовательного раздела БДМО и муниципальной демографии D01. **Муниципальный E0/E1 по кампаниям/кампусам не выполняется:** размещение исторических кампусов не подтверждено. Не подменяем фактическое местоположение филиала юридическим адресом головного учреждения.

In [ ]:
munpath=get_source(BDMO_EDU)
# Сквозной SQL по ВСЕМ строкам источника, группировка без загрузки 1.57 млн строк в RAM.
MUNICIPAL=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("indicator_section" AS VARCHAR) AS "section", CAST("indicator_name" AS VARCHAR) AS "indicator", CAST("indicator_unit" AS VARCHAR) AS "unit", COUNT(*)::BIGINT AS "n_records", COUNT(DISTINCT CAST("oktmo" AS VARCHAR))::BIGINT AS "n_oktmo", COUNT(DISTINCT CAST("region_name" AS VARCHAR))::BIGINT AS "n_regions", COUNT(*) FILTER (WHERE TRY_CAST("indicator_value" AS DOUBLE) IS NOT NULL)::BIGINT AS "n_numeric" FROM read_parquet(?) GROUP BY 1,2,3,4""",[str(munpath)])
MUNICIPAL=put('БДМО_образование_профиль',MUNICIPAL,required=True)
MUNICIPAL_TOTAL=int(MUNICIPAL.n_records.sum())
# Отдельная полная обработка тематической совокупности предыдущего протокола (референс 664 917),
# а НЕ сравнение произвольных 1.57 млн строк с фиксированным числом.
municipal_pattern='(профессиональн|высш[еиг]|колледж|техникум|университет|образовательн.*организац)'
MUNICIPAL_PANEL=ROOT/'derived'/'bdmo_professional_education_observed_full.parquet'
municipal_sql="""SELECT CAST("oktmo" AS VARCHAR) AS "oktmo", TRY_CAST("year" AS INTEGER) AS "year", CAST("indicator_code" AS VARCHAR) AS "indicator_code", CAST("indicator_name" AS VARCHAR) AS "indicator_name", CAST("indicator_unit" AS VARCHAR) AS "indicator_unit", COUNT(*)::BIGINT AS "source_records", MIN(TRY_CAST("indicator_value" AS DOUBLE)) AS "value_min", MAX(TRY_CAST("indicator_value" AS DOUBLE)) AS "value_max" FROM read_parquet(?) WHERE regexp_matches(lower(CAST("indicator_name" AS VARCHAR)),?) AND "oktmo" IS NOT NULL AND "year" IS NOT NULL GROUP BY 1,2,3,4,5"""
with db() as connection:
    query=municipal_sql.replace('read_parquet(?)','read_parquet('+sqlquote(munpath)+')').replace(
        'regexp_matches(lower(CAST("indicator_name" AS VARCHAR)),?)',
        'regexp_matches(lower(CAST("indicator_name" AS VARCHAR)),'+sqlquote(municipal_pattern)+')')
    connection.execute('COPY ('+query+') TO '+sqlquote(MUNICIPAL_PANEL)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
mpf=pq.ParquetFile(MUNICIPAL_PANEL)
with db() as connection:
    stat=connection.execute('SELECT COUNT(*),COALESCE(SUM(source_records),0),'
        'COUNT(*) FILTER (WHERE "source_records"=1 AND "value_min"="value_max") '
        'FROM read_parquet(?)',[str(MUNICIPAL_PANEL)]).fetchone()
MUN_CANDIDATE_AUDIT=put('БДМО_профессиональное_образование',pd.DataFrame([{
    'полных_строк_раздела':MUNICIPAL_TOTAL,'подходящих_исходных_строк':int(stat[1]),
    'уникальных_наблюдаемых_ячеек':int(stat[0]),'однозначных_ячеек':int(stat[2]),
    'предыдущий_эталон_однозначных_ячеек':664917,
    'неоднозначных_или_недопустимых_ячеек':int(stat[0])-int(stat[2]),
    'совпало_с_эталоном':int(stat[2])==664917,
    'полный_потоковый_аудит':True,'сопоставление_кампусов':False}]))

D01_RAW=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("oktmo" AS VARCHAR) AS "oktmo", CAST("region" AS VARCHAR) AS "region", TRY_CAST("population" AS DOUBLE) AS "population", TRY_CAST("births" AS DOUBLE) AS "births", TRY_CAST("deaths" AS DOUBLE) AS "deaths", TRY_CAST("migration_rate" AS DOUBLE) AS "migration_rate" FROM read_parquet(?)""",[str(get_source(D01))])
D01_RAW['oktmo']=clean_code(D01_RAW.oktmo)
D01_QA=put('Муниципалитеты_демография_QA',D01_RAW.groupby('year',dropna=False).agg(
  наблюдений=('oktmo','size'),уникальных_октмо=('oktmo','nunique'),
  положительных_населений=('population',lambda x:int(x.gt(0).sum())),
  медиана_населения=('population','median'),
  медиана_коэф_миграции=('migration_rate','median')).reset_index())

# Региональная экономика D05: только словарь реальных показателей, не новые фиктивные сопоставления.
try:
    labor=qry("""SELECT CAST("indicator_code" AS VARCHAR) AS "code", CAST("indicator_name" AS VARCHAR) AS "indicator", CAST("indicator_unit" AS VARCHAR) AS "unit", CAST("object_level" AS VARCHAR) AS "object_level", COUNT(*)::BIGINT AS "rows", MIN(TRY_CAST("year" AS INTEGER)) AS "from_year", MAX(TRY_CAST("year" AS INTEGER)) AS "to_year" FROM read_parquet(?) GROUP BY 1,2,3,4""",[str(get_source(D05_LABOR))])
    put('D05_труд_доступные_показатели',labor)
except Exception as exc:
    issue('ML-5.7','D05_DESCRIPTION_UNAVAILABLE','Не удалось загрузить словарь D05: '+type(exc).__name__)

issue('ML-5.7','NOT_FEASIBLE_FOR_MUNICIPAL_E0_E1',
 'Подтверждённой географии кампусов/филиалов на исторических ОКТМО нет. Анализируется только первичная муниципальная статистика без муниципального образовательного эффекта')
FINDINGS.append({'модуль':'ML-5.7','вывод':'Муниципальное моделирование образовательных площадок исключено по данным',
  'расчёт':f'Профилировано {MUNICIPAL_TOTAL:,} записей образовательного БДМО, {int(stat[1]):,} тематически относящихся к профобразованию, {len(D01_RAW):,} строк D01',
  'ограничение':'Нет подтверждённой исторической географии кампусов для E0/E1'})
print('BDMO: записи полного образовательного раздела:',MUNICIPAL_TOTAL)
mark_stage('Муниципалитеты')

## 8.1. Муниципальные показатели образования и демографии по точным ОКТМО и годам

Проверка только однозначных, непосредственно измеренных пар БДМО и D01. Без реконструкции географии кампусов и без E0/E1.

In [ ]:
MUN_REGION=[];MUN_MATCH_STATS=[];MUN_OBSERVED=pd.DataFrame()
d=D01_RAW[['oktmo','year','region','population','migration_rate']].copy()
d=d.loc[d.oktmo.notna() & d.year.notna()].copy()
dups=d.duplicated(['oktmo','year'],keep=False)
if dups.any():QUARANTINE.append({'table':'D01_municipal','problem':'Неоднозначный ОКТМО×год исключён','count':int(dups.sum())})
d=d.loc[~dups].copy();d['year']=d.year.astype(int)
with db() as con:
    con.register('d01_unique',d)
    # Первичные значения БДМО не меняются; JOIN строго на одном муниципальном коде и годе.
    statement=('SELECT m."oktmo" AS "oktmo",m."year" AS "year",m."indicator_code" AS "indicator_code",'
      'm."indicator_name" AS "indicator_name",m."indicator_unit" AS "unit",m."value_min" AS "value",'
      'd."region" AS "region",d."population" AS "population",d."migration_rate" AS "migration_rate" '
      'FROM read_parquet(?) m INNER JOIN d01_unique d ON m."oktmo"=d."oktmo" AND m."year"=d."year" '
      'WHERE m."source_records"=1 AND m."value_min"=m."value_max" AND m."value_min" IS NOT NULL')
    MUN_OBSERVED=con.execute(statement,[str(MUNICIPAL_PANEL)]).df()
if len(MUN_OBSERVED):
    MUN_OBSERVED['region_key']=clean_name(MUN_OBSERVED.region)
    grp=MUN_OBSERVED.groupby(['year','region_key','indicator_code','indicator_name','unit'],dropna=False)
    MUN_REGION=grp.agg(наблюдаемых_муниципалитетов=('oktmo','nunique'),
       медиана_значения_показателя=('value','median'),
       q25_значения=('value',lambda s:s.quantile(.25)),
       q75_значения=('value',lambda s:s.quantile(.75)),
       доля_с_наблюдаемым_населением=('population',lambda x:float(x.notna().mean())),
       медиана_численности_населения=('population','median'),
       медиана_коэффициента_миграции=('migration_rate','median')).reset_index()
    MUN_MATCH_STATS=[{'пар_муниципалитет_год':int(MUN_OBSERVED[['oktmo','year']].drop_duplicates().shape[0]),
      'число_тематических_строк':len(MUN_OBSERVED),'число_точных_индикаторов':int(MUN_OBSERVED.indicator_code.nunique()),
      'регионов_по_именам_D01':int(MUN_OBSERVED.region_key.nunique()),
      'статус':'Совпадение наблюдаемых статистик; не фактическое расположение кампуса'}]
put('Муниципальное_образование_регионы',MUN_REGION if isinstance(MUN_REGION,pd.DataFrame) else pd.DataFrame())
put('Муниципальный_перекрестный_охват',pd.DataFrame(MUN_MATCH_STATS))
FINDINGS.append({'модуль':'ML-5.7','вывод':'Сопоставлены муниципальные образовательные и демографические наблюдения',
  'расчёт':f'{len(MUN_OBSERVED)} точных тематических муниципальных ячеек, {len(MUN_REGION)} однородных региональных профилей',
  'ограничение':'Это наблюдаемость БДМО×D01, а не эффект образовательной инфраструктуры или география кампусов'})
mark_stage('Сопоставление БДМО')

## 8А. Региональный рынок труда и образовательные потоки — только проверяемые совпадения
D05 (`labor`) предоставляет отдельные наблюдаемые региональные показатели. Сопоставляются только совпадающие **регион и год**, а показатель остаётся с исходным названием, кодом и единицей. Неоднозначные или несопоставленные строки не используются. Корреляции с зарегистрированными E01-потоками рассчитываются для одного фактического года, с поправкой на множественные сравнения. Это описание возможных связей, не доказательство причин выбора места учёбы.

In [ ]:
LABOR_ASSOC=[];LABOR_MATCH=pd.DataFrame()
try:
    labor_rows=qry("""SELECT TRY_CAST("year" AS INTEGER) AS "year", CAST("object_name" AS VARCHAR) AS "region", CAST("object_level" AS VARCHAR) AS "level", CAST("indicator_code" AS VARCHAR) AS "indicator_code", CAST("indicator_name" AS VARCHAR) AS "indicator_name", CAST("indicator_unit" AS VARCHAR) AS "unit", COUNT(*)::BIGINT AS "n_source", MIN(TRY_CAST("indicator_value" AS DOUBLE)) AS "low", MAX(TRY_CAST("indicator_value" AS DOUBLE)) AS "high" FROM read_parquet(?) WHERE regexp_matches(lower(CAST("object_level" AS VARCHAR)),'регион|субъект') GROUP BY 1,2,3,4,5,6""",[str(get_source(D05_LABOR))])
    labor_rows['region_key']=clean_name(labor_rows.region)
    good= (labor_rows.n_source.eq(1)&labor_rows.low.notna()&labor_rows.low.eq(labor_rows.high)
            &labor_rows.region_key.notna()&labor_rows.year.notna())
    labor_rows['однозначно']=good
    put('D05_региональные_трудовые_ряды',labor_rows)
    eligible=labor_rows[good].copy()
    source_bal=BAL_FIXED if len(BAL_FIXED) else BAL_FULL
    source_bal=source_bal.copy();source_bal['region_key']=clean_name(source_bal.название_региона)
    LABOR_MATCH=eligible.merge(source_bal[['year','region_key','region_code',
      'прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе','наблюдаемый_баланс']],
      on=['year','region_key'],how='inner',validate='many_to_one')
    put('D05_точные_сопоставления_с_OD',LABOR_MATCH)
    LABOR_COVERAGE=LABOR_MATCH.groupby(['year','indicator_code','indicator_name','unit'],dropna=False).agg(
      n_регионов=('region_code','nunique'),n_наблюдений=('region_code','size')).reset_index()
    LABOR_COVERAGE=LABOR_COVERAGE.sort_values(['year','n_регионов'],ascending=[False,False])
    put('D05_охват_сопоставлений',LABOR_COVERAGE)
    if not LABOR_COVERAGE.empty:
        maxyear=int(LABOR_COVERAGE.year.max())
        candidates=LABOR_COVERAGE[(LABOR_COVERAGE.year==maxyear)&(LABOR_COVERAGE.n_регионов>=20)]
        candidates=candidates.head(30) # Отбор только по охвату, до просмотра корреляций.
        for item in candidates.itertuples(index=False):
            subset=LABOR_MATCH[(LABOR_MATCH.year==item.year)&
                (LABOR_MATCH.indicator_code==item.indicator_code)&
                (LABOR_MATCH.indicator_name==item.indicator_name)&
                (LABOR_MATCH.unit.fillna('')==('' if pd.isna(item.unit) else item.unit))]
            subset=subset.drop_duplicates('region_code',keep=False)
            if len(subset)<20 or subset.low.nunique()<4:continue
            for target in ('прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе'):
                t=subset[[target,'low']].dropna()
                if len(t)<20 or t[target].nunique()<4:continue
                rho,pval=spearmanr(t.low,t[target])
                if not np.isfinite(rho) or not np.isfinite(pval):continue
                LABOR_ASSOC.append({'year':int(item.year),'indicator_code':item.indicator_code,
                  'indicator_name':item.indicator_name,'unit':item.unit,
                  'E01_variable':target,'n_regions':len(t),'rho':float(rho),'p_unadjusted':float(pval),
                  'interpretation':'Наблюдаемая региональная ассоциация, без причинной идентификации'})
        # FDR: до 60 описательных тестов, явно исследовательский статус.
        if LABOR_ASSOC:
            order=np.argsort([z['p_unadjusted'] for z in LABOR_ASSOC]);m=len(order)
            running=1.
            for rank,i in reversed(list(enumerate(order))):
                running=min(running,LABOR_ASSOC[i]['p_unadjusted']*m/(rank+1))
                LABOR_ASSOC[i]['q_BH_exploratory']=float(running)
except Exception as exc:
    issue('ML-5.1','LABOR_CONTEXT_BLOCKED',f'Проверяемое сопоставление D05 недоступно: {type(exc).__name__}')
put('Региональные_факторные_связи',pd.DataFrame(LABOR_ASSOC))
if len(LABOR_ASSOC):
    FINDINGS.append({'модуль':'ML-5.1 / 5.3','вывод':'Проверены региональные связи E01 и источника D05',
      'расчёт':f'{len(LABOR_ASSOC)} описательных ассоциаций; действительные названия и единицы измерения сохранены',
      'ограничение':'Не причинный эффект; экономические несопоставленные строки не использованы'})
mark_stage('Рынок труда')

## 8Б. Проверка региональных изменений рынка труда и образовательных маршрутов

Когда источник D05 содержит одни и те же точно идентифицированные региональные показатели в последовательные годы, рассчитываются **межгодовые изменения** и сопоставляются с динамикой E01 на фиксированном покрытии. Группировки по разным единицам измерения, программам и годам запрещены; статистическая значимость корректируется по всему числу выполненных сравнений.

In [ ]:
LABOR_LONGITUDINAL=[]
if isinstance(LABOR_MATCH,pd.DataFrame) and len(LABOR_MATCH):
    cols=['region_code','year','indicator_code','indicator_name','unit','low',
          'прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе']
    for k,g in LABOR_MATCH[cols].groupby(['indicator_code','indicator_name','unit'],dropna=False):
        y=g.drop_duplicates(['region_code','year'],keep=False).sort_values(['region_code','year'],kind='mergesort')
        if len(y)<50:continue
        prior=y.copy();prior['year']+=1
        both=y.merge(prior,on=['region_code','year'],suffixes=('_current','_previous'),validate='one_to_one')
        if len(both)<25 or both.region_code.nunique()<15:continue
        both['delta_labor']=both.low_current-both.low_previous
        for target in ['прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе']:
            both['delta_edu']=np.log1p(both[target+'_current'].clip(lower=0))-np.log1p(both[target+'_previous'].clip(lower=0))
            dat=both[['delta_labor','delta_edu','region_code']].dropna()
            if len(dat)<25 or dat.region_code.nunique()<15 or dat.delta_labor.nunique()<5:continue
            # Не считать повторяющиеся годы одного региона независимыми. Средняя дельта каждого региона.
            reduced=dat.groupby('region_code')[['delta_labor','delta_edu']].mean()
            if len(reduced)<15 or reduced.delta_labor.nunique()<4:continue
            rho,pval=spearmanr(reduced.delta_labor,reduced.delta_edu)
            if not np.isfinite(rho):continue
            LABOR_LONGITUDINAL.append({'indicator_code':k[0],'indicator_name':k[1],'unit':k[2],
              'target':target,'n_regions':len(reduced),'n_consecutive_region_year_pairs':len(dat),
              'spearman_rho_region_mean_deltas':float(rho),'p_unadjusted':float(pval),
              'scope':'Сопоставимые региональные изменения на фиксированном E01; без причинной идентификации'})
if LABOR_LONGITUDINAL:
    ps=np.asarray([r['p_unadjusted'] for r in LABOR_LONGITUDINAL]);order=np.argsort(ps);n=len(ps);adj=np.empty(n);running=1.
    for j in range(n-1,-1,-1):
        idx=order[j];running=min(running,ps[idx]*n/(j+1));adj[idx]=running
    for r,q in zip(LABOR_LONGITUDINAL,adj):r['q_FDR_BH']=float(q)
put('D05_динамика_показателей',pd.DataFrame(LABOR_LONGITUDINAL))
FINDINGS.append({'модуль':'ML-5.1 / ML-5.3','вывод':'Проверены изменения наблюдаемых показателей рынка труда и OD, если доступны несколько лет',
 'расчёт':f'{len(LABOR_LONGITUDINAL)} сопоставимых региональных межгодовых тестов; FDR при фактическом числе сравнений',
 'ограничение':'Не является оценкой эффекта экономики на миграционные решения; отсутствующие временные пары не восстановлены'})

## 9. Сопоставления с региональными возможностями и управленческие вопросы
Необязательная для идентификации миграции, но необходимая для предметного понимания корреляция: объём наблюдаемого притока СПО и численность организаций СПО в одном регионе. Ровно совпадающие нормализованные названия и один год; сопоставление и потерянные регионы выгружаются отдельно. Корреляция **не является причинным эффектом** создания колледжа. Не производится произвольное присоединение 52 525 несопоставленных экономических строк.

In [ ]:
ASSOCIATIONS=[]
matched=TABLES['Регион_OD_и_СПО_сеть'].copy()
valid=matched.loc[matched.регион_сопоставлен & matched.организаций_СПО.gt(0)
 & matched.прибывшие_в_наблюдаемом_показателе.notna()].copy()
if len(valid)>=10 and valid.организаций_СПО.nunique()>2:
    rho,pval=spearmanr(valid.организаций_СПО,valid.прибывшие_в_наблюдаемом_показателе)
    ASSOCIATIONS.append({'год':LATEST,'показатель_1':'Количество зарегистрированных организаций СПО',
      'показатель_2':'Объём наблюдаемых входящих E01-пар на фиксированном покрытии',
      'n_регионов':int(len(valid)),'spearman_rho':float(rho),'p_exploratory_only':float(pval),
      'смысл':'Описательная региональная ассоциация, НЕ причинный эффект и НЕ межрегиональный приём по организациям'})
else:
    issue('ML-5.3','NOT_FEASIBLE_ASSOCIATION',f'Сопоставимых регионов для контекстной корреляции: {len(valid)}')
put('Контекст_ассоциации',pd.DataFrame(ASSOCIATIONS))
put('Не_сопоставленные_регионы',matched.loc[~matched.регион_сопоставлен])
# Территориальная типология по наблюдаемому балансу и концентрации;
# только описательная, а не оценка постоянных переездов или политического воздействия.
typ=REGION_LATEST.copy()
co=CONCENTRATION[(CONCENTRATION.year==LATEST)&(CONCENTRATION.сторона=='ВХОДЯЩИЕ')]
typ=typ.merge(co[['region_code','HHI','доля_первых_5']],on='region_code',how='left',validate='one_to_one')
q_hi=typ['прибывшие_в_наблюдаемом_показателе'].quantile(.75)
q_conc=typ['HHI'].median()
typ['территориальный_профиль']=np.select([
  typ.наблюдаемый_баланс.gt(0)&typ['прибывшие_в_наблюдаемом_показателе'].ge(q_hi),
  typ.наблюдаемый_баланс.lt(0)&typ.HHI.gt(q_conc),
  typ.наблюдаемый_баланс.lt(0)],
 ['КРУПНЫЙ_НАБЛЮДАЕМЫЙ_ЦЕНТР','ИСХОДЯЩИЙ_ПРОФИЛЬ_И_КОНЦЕНТРАЦИЯ','ПРЕОБЛАДАНИЕ_ИСХОДЯЩИХ'],
 default='СМЕШАННЫЙ_НАБЛЮДАЕМЫЙ_ПРОФИЛЬ')
typ['управленческое_прочтение']='Описание наблюдаемых показателей E01; для решения о финансировании нужны отдельные причинные исследования'
put('Типология_регионов',typ)
print('Участников точного сопоставления с СПО-сетью:',len(valid),'из',len(REGION_LATEST))
mark_stage('Территории')

## 9А. Образовательная ёмкость и потоки: проверка эффекта размера региона

Сопоставляем наблюдаемые OD с реестром СПО на совпадающих регионах. Помимо сырой корреляции проверяем чувствительность к исключению Москвы и Санкт-Петербурга и частную ранговую связь при контроле наблюдаемого исходящего объёма. **Ни один показатель не называется причинным эффектом.**

In [ ]:
REG_SIZE=[];REG_LINKS=[]
try:
    x=joined.loc[joined['регион_сопоставлен'] & joined['организаций_СПО'].gt(0)].copy()
    x['наблюдаемый_входящий_поток_на_организацию_СПО']=x['прибывшие_в_наблюдаемом_показателе']/x['организаций_СПО']
    x['наблюдаемый_исходящий_поток_на_организацию_СПО']=x['выбывшие_в_наблюдаемом_показателе']/x['организаций_СПО']
    x['отношение_входящих_к_исходящим']=x['прибывшие_в_наблюдаемом_показателе']/x['выбывшие_в_наблюдаемом_показателе'].replace(0,np.nan)
    x['проверка_источников']='Сеть организаций СПО на доступный год; OD только постоянные наблюдаемые пары'
    REG_SIZE=x.sort_values(['наблюдаемый_входящий_поток_на_организацию_СПО','region_code'],ascending=[False,True],kind='mergesort')
    if len(x)>=20:
        for label,sub in [('все_сопоставимые',x),('без_Москвы_и_СПб',x.loc[~x['название_региона'].astype(str).str.contains('Москва$|Санкт-Петербург',regex=True,case=False)])]:
            if len(sub)<20:continue
            a=sub['организаций_СПО'].rank().to_numpy(float)
            b=sub['прибывшие_в_наблюдаемом_показателе'].rank().to_numpy(float)
            c=sub['выбывшие_в_наблюдаемом_показателе'].rank().to_numpy(float)
            ra,rb=spearmanr(a,b),spearmanr(a,c)
            # Частная ранговая ассоциация после контроля за наблюдаемым исходящим объёмом.
            A=np.column_stack([np.ones(len(sub)),c]);eps_a=a-A@np.linalg.lstsq(A,a,rcond=None)[0]
            eps_b=b-A@np.linalg.lstsq(A,b,rcond=None)[0]
            partial=float(np.corrcoef(eps_a,eps_b)[0,1]) if np.std(eps_a)>0 and np.std(eps_b)>0 else np.nan
            REG_LINKS.append({'совокупность':label,'число_регионов':len(sub),'Spearman_организаций_и_входящих':float(ra.statistic),
             'p_описательное_входящие':float(ra.pvalue),'Spearman_организаций_и_исходящих':float(rb.statistic),
             'p_описательное_исходящие':float(rb.pvalue),
             'частная_ранговая_корреляция_входящих_и_организаций_при_исходящих':partial,
             'оговорка':'Контроль только наблюдаемого исходящего потока, не является каузальной оценкой'})
except Exception as exc:
    issue('ML-5.3','REGIONAL_CAPACITY_DIAGNOSTIC_BLOCKED',str(type(exc).__name__))
put('Региональная_ёмкость_и_потоки',REG_SIZE if isinstance(REG_SIZE,pd.DataFrame) else pd.DataFrame())
put('Региональная_ёмкость_корреляции',pd.DataFrame(REG_LINKS))
FINDINGS.append({'модуль':'ML-5.3','вывод':'Региональная концентрация образовательной сети сопоставлена с наблюдаемыми потоками',
  'расчёт':f'{len(REG_SIZE)} регионов с организациями СПО, {len(REG_LINKS)} проверок чувствительности',
  'ограничение':'Нормировка на число организаций СПО не равна доле всех абитуриентов и не устраняет систематическую неоднородность регионов'})

## 9Б. Проверка региональных связей при контроле наблюдаемого масштаба

In [ ]:
# Проверка региональной емкости с контролем наблюдаемого исходящего E01, без причинных заявлений.
REGIONAL_CONTROLS=[]
if isinstance(REG_SIZE,pd.DataFrame) and len(REG_SIZE)>=20:
 x=REG_SIZE.dropna(subset=['организаций_СПО','прибывшие_в_наблюдаемом_показателе','выбывшие_в_наблюдаемом_показателе','region_code']).copy()
 x=x[x['организаций_СПО'].gt(0)].drop_duplicates('region_code',keep=False)
 for label,df in [('ВСЕ',x),('БЕЗ_ДВУХ_КРУПНЕЙШИХ',x.drop(x.nlargest(min(len(x),2),'прибывшие_в_наблюдаемом_показателе').index))]:
  if len(df)<15:continue
  X=np.log1p(df[['организаций_СПО','выбывшие_в_наблюдаемом_показателе']].to_numpy(float))
  y=np.log1p(df['прибывшие_в_наблюдаемом_показателе'].to_numpy(float))
  A=np.column_stack([np.ones(len(df)),X]);coef=np.linalg.lstsq(A,y,rcond=None)[0]
  rng=np.random.default_rng(CFG.SEED);boot=[]
  for _ in range(1000):
   idx=rng.integers(0,len(df),len(df));B=A[idx]
   if np.linalg.matrix_rank(B)==3:boot.append(float(np.linalg.lstsq(B,y[idx],rcond=None)[0][1]))
  REGIONAL_CONTROLS.append({'stratum':label,'n_regions':len(df),'coef_log_SPO_organizations':float(coef[1]),
    'coef_log_observed_outflow':float(coef[2]),'region_bootstrap_ci_low':float(np.quantile(boot,.025)),
    'region_bootstrap_ci_high':float(np.quantile(boot,.975)),
    'interpretation':'Региональная ассоциация с неполным контролем масштаба, не причинный эффект'})
REGIONAL_CONTROLS=put('Региональные_контрольные_сравнения',pd.DataFrame(REGIONAL_CONTROLS))
mark_stage('Региональный контроль')

## 10. Вспомогательный прогноз OD: честная проверка пригодности данных, а не цель проекта
Только постоянно наблюдаемые межрегиональные пары E01; обучающие признаки — значения до года цели, включая предыдущие силы регионов. 2024 — development, 2025 — контрольный год; базовые методы `последний год` и `среднее двух лет`. Одна прозрачная Ridge и одна стохастическая RandomForest. Для последней допускаются 50 заранее фиксированных сидов с отчётом распределения, но без выбора лучшего по TEST. Дополнительно выполняется тест переноса на регионы происхождения, не входившие в обучение. Исторического *ex-ante* эксперимента нет.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
FORECAST=pd.DataFrame();F_METRICS=pd.DataFrame();F_STABILITY=pd.DataFrame()
MODEL_REPLAY=[];REFITS=[];BOOTSTRAP=[]

def forecast_features(edges):
    if edges.empty:return pd.DataFrame()
    rows=[];years=sorted(edges.year.astype(int).unique())
    for target_year in years[2:]:
        past=edges[edges.year.eq(target_year-1)][['edge','origin_code','destination_code','value']].rename(columns={'value':'lag1'})
        prev2=edges[edges.year.eq(target_year-2)][['edge','value']].rename(columns={'value':'lag2'})
        actual=edges[edges.year.eq(target_year)][['edge','value']].rename(columns={'value':'actual'})
        out=past.groupby('origin_code').lag1.sum().rename('origin_prev_out')
        inc=past.groupby('destination_code').lag1.sum().rename('destination_prev_in')
        y=actual.merge(past,on='edge',validate='one_to_one').merge(prev2,on='edge',validate='one_to_one')
        y=y.merge(out,on='origin_code',validate='many_to_one').merge(inc,on='destination_code',validate='many_to_one')
        y['target_year']=int(target_year);y['information_cutoff']=int(target_year-1)
        rows.append(y)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

F_FEATURES=forecast_features(FIXED)
# Future-mutation canary: изменение целевого года не должно менять исходные признаки на этот год.
FUTURE_CANARY={'tested':False,'passed':False}
if len(F_FEATURES) and len(FIXED):
    probe_year=int(F_FEATURES.target_year.max())
    mutated=FIXED.copy()
    mutated.loc[mutated.year.eq(probe_year),'value']+=1_000_000.
    altered=forecast_features(mutated)
    l=F_FEATURES[F_FEATURES.target_year.eq(probe_year)].sort_values('edge')
    r=altered[altered.target_year.eq(probe_year)].sort_values('edge')
    assert len(l)==len(r) and np.allclose(l[['lag1','lag2','origin_prev_out','destination_prev_in']],
                                           r[['lag1','lag2','origin_prev_out','destination_prev_in']])
    FUTURE_CANARY={'tested':True,'passed':True,'target_year':probe_year,'n_tested':len(l)}
put('Прогноз_future_canary',pd.DataFrame([FUTURE_CANARY]))
if len(F_FEATURES) and not (F_FEATURES.information_cutoff<F_FEATURES.target_year).all():
    raise RuntimeError('Временная утечка: information_cutoff >= target_year')
FEATURES=['lag1','lag2','origin_prev_out','destination_prev_in']

def regressor(kind,seed):
    if kind=='ridge':return Pipeline([('imp',SimpleImputer(strategy='median')),('est',Ridge(alpha=100.))])
    if kind=='random_forest':return RandomForestRegressor(n_estimators=40,max_depth=7,
       min_samples_leaf=5,max_features=0.8,random_state=int(seed),n_jobs=CFG.WORKERS)
    raise ValueError(kind)

def register_model(mod,kind,seed,tr,te,protocol):
    m=regressor(kind,seed);m.fit(tr[FEATURES],tr.actual)
    yp=np.maximum(0,m.predict(te[FEATURES]))
    path=ROOT/'models'/f'{mod}_{protocol}_{kind}_{seed}.joblib'
    joblib.dump({'estimator':m,'columns':FEATURES,'train_hash':hashlib.sha256(
       pd.util.hash_pandas_object(tr[FEATURES+['actual']],index=False).values.tobytes()).hexdigest()},path,compress=3)
    loaded=joblib.load(path)['estimator']
    repeat=np.maximum(0,loaded.predict(te[FEATURES]))
    replay=bool(np.allclose(yp,repeat,rtol=1e-10,atol=1e-8))
    MODEL_REPLAY.append({'experiment':mod,'kind':kind,'seed':seed,'protocol':protocol,
      'sha256':sha256(path),'predictions_identical':replay,'test_rows':len(te)})
    if not replay:raise RuntimeError('Сохранённая модель даёт другие предсказания')
    return yp

def run_models(tr,te,protocol,full_seed_check=False):
    if len(tr)<100 or len(te)<40:raise RuntimeError('Недостаточно наблюдений для вспомогательного прогноза')
    pred=te[['edge','origin_code','destination_code','target_year','information_cutoff','actual']].copy()
    pred['protocol']=protocol
    outputs={'persistence':te.lag1.to_numpy(float),'rolling_mean_2':te[['lag1','lag2']].mean(axis=1).to_numpy(float)}
    outputs['ridge']=register_model('E01', 'ridge',CFG.SEED,tr,te,protocol)
    outputs['random_forest']=register_model('E01', 'random_forest',CFG.SEED,tr,te,protocol)
    rows=[]
    for kind,values in outputs.items():
        vals=np.maximum(0,np.asarray(values,dtype=float));t=pred.copy();t['model']=kind;t['seed']=CFG.SEED
        t['prediction']=vals;rows.append(t)
    extra=[]
    if full_seed_check:
        for i in range(50):
            seed=CFG.SEED+i+1
            y=register_model('E01','random_forest',seed,tr,te,protocol+'_stability')
            t=pred.copy();t['model']='random_forest_50';t['seed']=seed;t['prediction']=np.maximum(0,y)
            extra.append(t)
    return pd.concat(rows+extra,ignore_index=True)

if CFG.RUN_FORECAST and not F_FEATURES.empty and len(F_FEATURES.target_year.unique())>=2:
    years=sorted(F_FEATURES.target_year.unique())
    testyear=2025 if 2025 in years else int(years[-1]);devyear=int(max(y for y in years if y<testyear))
    devtr=F_FEATURES[F_FEATURES.target_year<devyear].copy();devte=F_FEATURES[F_FEATURES.target_year==devyear].copy()
    testtr=F_FEATURES[F_FEATURES.target_year<testyear].copy();testte=F_FEATURES[F_FEATURES.target_year==testyear].copy()
    if len(devtr)>=100 and len(devte)>=40 and len(testtr)>=100 and len(testte)>=40:
        preds=[run_models(devtr,devte,'TEMPORAL_DEV',False),
          run_models(testtr,testte,'TEMPORAL_TEST',CFG.RUN_50_SEED_CHECK and CFG.MODE=='FULL')]
        # Незнакомые источники на тесте: группы не пересекаются по региону происхождения.
        if len(testtr.origin_code.unique())>=8:
            rng=np.random.default_rng(CFG.SEED)
            origins=np.sort(testtr.origin_code.dropna().unique())
            hold=set(rng.choice(origins,max(1,int(round(len(origins)*.2))),replace=False))
            crossed_tr=testtr[~testtr.origin_code.isin(hold)].copy()
            crossed_te=testte[testte.origin_code.isin(hold)].copy()
            assert set(crossed_tr.origin_code).isdisjoint(crossed_te.origin_code)
            if len(crossed_te)>=40:
                preds.append(run_models(crossed_tr,crossed_te,'CROSSED_NEW_ORIGINS',False))
        FORECAST=put('Прогноз_OD_наблюдения',pd.concat(preds,ignore_index=True))
        m=[]
        for (protocol,model,seed),g in FORECAST.groupby(['protocol','model','seed']):
            y=g.actual.to_numpy(float);yp=g.prediction.to_numpy(float)
            m.append({'protocol':protocol,'model':model,'seed':int(seed),'target_year':int(g.target_year.iloc[0]),
              'n_test':len(g),'MAE':float(mean_absolute_error(y,yp)),
              'RMSE':float(np.sqrt(mean_squared_error(y,yp))),
              'WAPE':float(100*np.abs(y-yp).sum()/y.sum()) if y.sum()>0 else np.nan,
              'bias':float(np.mean(yp-y))})
        F_METRICS=put('Прогноз_метрики',pd.DataFrame(m))
        # Присвоение статуса только по фактически выполненному исследованию.
        F_STABILITY=put('Прогноз_50_сидов',F_METRICS[F_METRICS.model=='random_forest_50'].copy())
        # Контрольное ПЕРЕОБУЧЕНИЕ одного фиксированного ядра, не на тесте.
        for kind in ('ridge','random_forest'):
            m1=regressor(kind,CFG.SEED).fit(testtr[FEATURES],testtr.actual)
            m2=regressor(kind,CFG.SEED).fit(testtr[FEATURES],testtr.actual)
            p1=m1.predict(testte[FEATURES]);p2=m2.predict(testte[FEATURES])
            REFITS.append({'model':kind,'test_year':testyear,'same_train_and_seed':True,
                          'identical_predictions':bool(np.allclose(p1,p2,rtol=1e-9,atol=1e-8)),
                          'max_abs_difference':float(np.max(np.abs(p1-p2)))})
        if not all(x['identical_predictions'] for x in REFITS):raise RuntimeError('Повторное обучение не воспроизводится')
        issue('ML-5.1','RETROSPECTIVE_FORECAST','Прогноз на фиксированных E01-парах проверен на прошлых годах; не является экс-анте')
    else:issue('ML-5.1','NOT_FEASIBLE_FORECAST','Недостаточно реальных временных наблюдений после разбиения')
else:issue('ML-5.1','NOT_FEASIBLE_FORECAST','Недостаточная история E01 или прогноз выключен конфигурацией')

put('Повторные_предсказания',pd.DataFrame(MODEL_REPLAY))
put('Контрольное_переобучение',pd.DataFrame(REFITS))
print('Вспомогательная программа прогнозирования:',len(FORECAST),'строк,',len(F_METRICS),'метрик')
mark_stage('Прогноз OD')

## 10А. Проверка факторов регионального образовательного выбора

Проверяем добавочную предсказательную ценность **наблюдаемой сети СПО и исторической ёмкости организаций** к лагам маршрута. Географическую дистанцию используем только если в источнике TRANSPORT найдены действительные маршруты *между городами с подтверждённым признаком столицы региона* и совпали регионы; иначе географический тест объявляется недоступным. Сравнение на одинаковых наблюдениях и одинаковом тестовом годе. В 2026-м снимке это ретроспективный тест, не исторический ex ante.

In [ ]:
# Этап A: исторически предыдущие характеристики сети/реестра по тем же OD-ключам.
FACTORS=pd.DataFrame();ABLATION_METRICS=[];FACTOR_PRED=[];DISTANCE_QA=[]
if len(F_FEATURES):
    f=F_FEATURES.copy()
    # ПРИЗНАКИ берутся только из предыдущего года, не из сети года прогноза.
    nodes=NETWORK_NODES.loc[NETWORK_NODES['совокупность'].eq('ПОСТОЯННОЕ_ПОКРЫТИЕ'),
         ['год','region_code','pagerank','входящих_партнёров','исходящих_партнёров']].copy()
    for side,code in [('origin','origin_code'),('dest','destination_code')]:
        nd=nodes.rename(columns={'год':'year_lag', 'region_code':code,
            'pagerank':side+'_pagerank_prev','входящих_партнёров':side+'_in_degree_prev',
            'исходящих_партнёров':side+'_out_degree_prev'})
        f['year_lag']=f.target_year-1
        f=f.merge(nd,on=['year_lag',code],how='left',validate='many_to_one')
    # Из реестра СПО: только уникальные организации одного региона в предыдущем году.
    org=REGISTRY_COUNTS.loc[REGISTRY_COUNTS['сектор'].eq('СПО'),['year','region_key','число_учреждений']].copy()
    org=org.groupby(['year','region_key'],as_index=False)['число_учреждений'].sum()
    for side,code in [('origin','origin_code'),('dest','destination_code')]:
        regmap=pd.Series(f[code].map(REG_MAP),index=f.index)
        f[side+'_region_key']=clean_name(regmap)
        cap=org.rename(columns={'year':'year_lag','region_key':side+'_region_key',
                                 'число_учреждений':side+'_SPO_org_count_prev'})
        f=f.merge(cap,on=['year_lag',side+'_region_key'],how='left',validate='many_to_one')
    # Источник транспортных маршрутов документирует тип и статус столиц — никаких суррогатов расстояния.
    try:
        trasset=asset(source='TRANSPORT');trpath=get_source(trasset)
        raw=qry("""SELECT CAST("region_from" AS VARCHAR) AS "from_region", CAST("region_to" AS VARCHAR) AS "to_region", CAST("transport_type" AS VARCHAR) AS "transport_mode", TRY_CAST("distance_km" AS DOUBLE) AS "distance_km", TRY_CAST("is_capital_from" AS BOOLEAN) AS "from_is_capital", TRY_CAST("is_capital_to" AS BOOLEAN) AS "to_is_capital" FROM read_parquet(?)""",[str(trpath)])
        raw=raw.loc[raw.from_is_capital.eq(True)&raw.to_is_capital.eq(True)&raw.distance_km.gt(0)].copy()
        raw['from_key']=clean_name(raw.from_region);raw['to_key']=clean_name(raw.to_region)
        summary=raw.groupby(['from_key','to_key','transport_mode'],dropna=False).agg(
             records=('distance_km','size'),unique_distances=('distance_km','nunique'),distance_km=('distance_km','median')).reset_index()
        summary=summary.loc[summary.unique_distances.eq(1)].copy()
        if len(summary):
            mode=summary.groupby('transport_mode').size().sort_values(ascending=False).index[0]
            pairs=summary.loc[summary.transport_mode.eq(mode),['from_key','to_key','distance_km']].dropna(subset=['from_key','to_key'])
            if pairs.duplicated(['from_key','to_key']).any():raise RuntimeError('Неоднозначная дистанция столиц')
            f=f.merge(pairs,left_on=['origin_region_key','dest_region_key'],right_on=['from_key','to_key'],how='left',validate='many_to_one')
            f=f.drop(columns=['from_key','to_key'])
            DISTANCE_QA.append({'тип':'regional_capital_transport','mode':str(mode),'unambiguous_pairs':len(pairs),
                'observed_OD_matches':int(f.distance_km.notna().sum()),'total_OD_feature_rows':len(f),
                'status':'VERIFIED_SOURCE_MATCH' if f.distance_km.notna().sum()>0 else 'NO_MATCH'})
        else:DISTANCE_QA.append({'type':'regional_capital_transport','status':'NO_UNAMBIGUOUS_DISTANCES'})
    except Exception as exc:
        # Отсутствие источника / несопоставимая схема => нет псевдогеографии.
        DISTANCE_QA.append({'type':'regional_capital_transport','status':'NOT_FEASIBLE','reason':type(exc).__name__})
    if 'distance_km' not in f:f['distance_km']=np.nan
    FACTORS=f.copy()
    put('OD_классификация_факторов',pd.DataFrame([{
       'feature':s,'past_available':True,'scientific_level':'OD / регион в предыдущем году',
       'source':'E01+SPO_REGISTRY+LAG_NETWORK'} for s in ['lag1','lag2','origin_prev_out','destination_prev_in',
       'origin_pagerank_prev','dest_pagerank_prev','origin_SPO_org_count_prev','dest_SPO_org_count_prev']] +
       [{'feature':'distance_km','past_available':bool(f.distance_km.notna().any()),
         'scientific_level':'столицы регионов при точном совпадении','source':'TRANSPORT'}]))
    # Абляции на одном тесте: сравнение за 2025 год внутри полной совокупности фиксированных рёбер.
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import PoissonRegressor
    baseline_cols=['lag1','lag2','origin_prev_out','destination_prev_in']
    extra_cols=['origin_pagerank_prev','dest_pagerank_prev','origin_in_degree_prev',
       'dest_in_degree_prev','origin_SPO_org_count_prev','dest_SPO_org_count_prev']
    if f.target_year.nunique()>=2:
        maxyear=int(f.target_year.max())
        tr=f.loc[f.target_year.lt(maxyear)].copy();te=f.loc[f.target_year.eq(maxyear)].copy()
        for tag,cols,full_support,universe in [
           ('lags_only',baseline_cols,baseline_cols+extra_cols,'COMMON_NETWORK_EDUCATION'),
           ('past_network_and_education',baseline_cols+extra_cols,baseline_cols+extra_cols,'COMMON_NETWORK_EDUCATION'),
           ('past_no_distance_on_geo_pairs',baseline_cols+extra_cols,baseline_cols+extra_cols+['distance_km'],'COMMON_DISTANCE'),
           ('past_network_education_and_distance',baseline_cols+extra_cols+['distance_km'],baseline_cols+extra_cols+['distance_km'],'COMMON_DISTANCE')]:
            # В каждой паре вариантов поддержка ИДЕНТИЧНА; пропуски не заменяются вымышленными нулями.
            tr1=tr.dropna(subset=full_support+['actual']).copy();te1=te.dropna(subset=full_support+['actual']).copy()
            if len(tr1)<100 or len(te1)<40:
                ABLATION_METRICS.append({'experiment':tag,'comparison_universe':universe,'status':'NOT_FEASIBLE_COVERAGE','n_train':len(tr1),'n_test':len(te1)})
                continue
            models={'persistence':te1['lag1'].to_numpy(float),'rolling_mean_2':te1[['lag1','lag2']].mean(axis=1).to_numpy(float)}
            for model_name in ['ridge','random_forest','poisson_regression']:
                mod=(Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('est',Ridge(alpha=50.))])
                    if model_name=='ridge' else Pipeline([('scale',StandardScaler()),('est',PoissonRegressor(alpha=1.0,max_iter=250))])
                    if model_name=='poisson_regression' else RandomForestRegressor(
                    n_estimators=80,max_depth=7,min_samples_leaf=5,random_state=CFG.SEED,n_jobs=CFG.WORKERS))
                mod.fit(tr1[cols],tr1.actual)
                models[model_name]=np.maximum(0,mod.predict(te1[cols]))
            for name,pred in models.items():
                vals=np.maximum(0,np.asarray(pred,float));y=te1.actual.to_numpy(float)
                ABLATION_METRICS.append({'experiment':tag,'comparison_universe':universe,'model':name,'status':'RETROSPECTIVE_TEST',
                     'year':maxyear,'n_train':len(tr1),'n_test':len(y),'MAE':float(np.abs(y-vals).mean()),
                     'RMSE':float(np.sqrt(np.mean((y-vals)**2))),
                     'same_support_hash':hashlib.sha256(pd.util.hash_pandas_object(te1[['edge','target_year']],index=False).values.tobytes()).hexdigest()})
                z=te1[['edge','origin_code','destination_code','target_year','actual']].copy()
                z['experiment']=tag;z['comparison_universe']=universe;z['model']=name;z['prediction']=vals;FACTOR_PRED.append(z)
put('OD_факторы_и_абляции',pd.DataFrame(ABLATION_METRICS))
put('OD_факторные_прогнозы',pd.concat(FACTOR_PRED,ignore_index=True) if FACTOR_PRED else pd.DataFrame())
put('OD_источники_дистанции',pd.DataFrame(DISTANCE_QA))
FINDINGS.append({'модуль':'ML-5.1 / ML-5.3','вывод':'Проверена дополнительная прогнозная ценность прошлой сети и образовательной инфраструктуры',
  'расчёт':f'{len(ABLATION_METRICS)} строк сравнений; признаки только до года цели, совместимая поддержка в каждом эксперименте',
  'ограничение':'Ретроспективная оценка; влияние внешних факторов причинно не идентифицировано'})
mark_stage('Факторы потоков')

## 10Б. Гравитационная модель с исходящими и входящими региональными эффектами

In [ ]:
# H5.1: Poisson-регрессия с фиксированными эффектами регионов происхождения/назначения.
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
GRAVITY_RESULTS=[];GRAVITY_PRED_ROWS=[];GRAVITY_REPLAY=[]
def gravity_features(df,geography):
 num=['lag1','lag2','origin_prev_out','destination_prev_in','origin_SPO_org_count_prev','dest_SPO_org_count_prev']
 if geography:num+=['distance_km']
 x=df[['origin_code','destination_code']+num].copy()
 for c in num:x[c]=np.log1p(pd.to_numeric(x[c],errors='coerce').clip(lower=0))
 for c in ('origin_code','destination_code'):x[c]=x[c].astype(str)
 return x,num
if len(FACTORS) and FACTORS.target_year.nunique()>=2:
 testyear=int(FACTORS.target_year.max());trall=FACTORS[FACTORS.target_year.lt(testyear)];teall=FACTORS[FACTORS.target_year.eq(testyear)]
 for universe,geo in [('full_observed_fe',False),('verified_capital_distance_fe',True)]:
  required=['actual','lag1','lag2','origin_prev_out','destination_prev_in','origin_SPO_org_count_prev','dest_SPO_org_count_prev']
  if geo:required+=['distance_km']
  train=trall.dropna(subset=required+['origin_code','destination_code']).copy()
  test=teall.dropna(subset=required+['origin_code','destination_code']).copy()
  if len(train)<100 or len(test)<40:
   GRAVITY_REPLAY.append({'universe':universe,'status':'NOT_FEASIBLE_COVERAGE','train':len(train),'test':len(test)});continue
  X,nums=gravity_features(train,geo);Z,_=gravity_features(test,geo)
  if not np.isfinite(X[nums].to_numpy(float)).all() or not np.isfinite(Z[nums].to_numpy(float)).all():
   raise AssertionError('Гравитация: нечисловые признаки в одинаково наблюдаемой совокупности')
  prep=ColumnTransformer([('region_effects',OneHotEncoder(handle_unknown='ignore'),['origin_code','destination_code']),
      ('historic_factors',StandardScaler(),nums)])
  est=Pipeline([('encoding',prep),('poisson',PoissonRegressor(alpha=.5,max_iter=400,tol=1e-6))])
  est.fit(X,train.actual.astype(float).clip(lower=0));pred=np.maximum(0,est.predict(Z))
  path=ROOT/'models'/('gravity_'+universe+'.joblib');joblib.dump(est,path,compress=3)
  same=bool(np.allclose(pred,np.maximum(0,joblib.load(path).predict(Z)),atol=1e-8,rtol=1e-8))
  GRAVITY_REPLAY.append({'universe':universe,'status':'REPLAYED' if same else 'FAIL','sha256':sha256(path),'test':len(test)})
  if not same:raise AssertionError('Повторная загрузка гравитационной модели не совпала')
  support=hashlib.sha256(pd.util.hash_pandas_object(test[['edge','target_year']].sort_values(['edge','target_year']),index=False).values.tobytes()).hexdigest()
  methods={'persistence':test.lag1.to_numpy(float),'rolling_mean_2':test[['lag1','lag2']].mean(axis=1).to_numpy(float),'Poisson_origin_destination_FE':pred}
  for name,values in methods.items():
   values=np.maximum(0,np.asarray(values,float));actual=test.actual.to_numpy(float)
   GRAVITY_RESULTS.append({'universe':universe,'model':name,'test_year':testyear,'n_train':len(train),'n_test':len(test),
      'MAE':float(np.abs(actual-values).mean()),'RMSE':float(np.sqrt(np.mean((actual-values)**2))),
      'same_support_hash':support,'scope':'ретроспективная регуляризованная Poisson с FE, не причинная оценка'})
   chunk=test[['edge','origin_code','destination_code','target_year','actual']].copy()
   chunk['universe']=universe;chunk['model']=name;chunk['prediction']=values
   GRAVITY_PRED_ROWS.append(chunk)
GRAVITY_RESULTS=put('Гравитация_метрики',pd.DataFrame(GRAVITY_RESULTS))
GRAVITY_PRED=put('Гравитация_прогнозы',pd.concat(GRAVITY_PRED_ROWS,ignore_index=True) if GRAVITY_PRED_ROWS else pd.DataFrame())
put('Гравитация_replay',pd.DataFrame(GRAVITY_REPLAY))
mark_stage('Гравитационный baseline')

## 11. Независимая проверка ошибок и устойчивости
Проверяем арифметику прогнозов, расхождения с простым методом и **кластерную** неопределённость по исходным регионам. 50 сидов относятся к одному стохастическому методу на одинаковой тестовой выборке и не образуют 50 независимых исторических экспериментов. Коррекция Холма применяется только к заранее выбранным двум сравнениям с инерционным прогнозом.

In [ ]:
STAT_TESTS=[];INDEPENDENT=[]
if len(FORECAST):
    for (protocol,model,seed),g in FORECAST.groupby(['protocol','model','seed']):
        y=g.actual.to_numpy(float);yp=g.prediction.to_numpy(float)
        INDEPENDENT.append({'protocol':protocol,'model':model,'seed':int(seed),'MAE_recalc':float(np.abs(y-yp).mean()),
                            'n_recalc':int(len(g))})
    INDEPENDENT=put('Прогноз_независимые_метрики',pd.DataFrame(INDEPENDENT))
    ck=F_METRICS.merge(INDEPENDENT,on=['protocol','model','seed'],validate='one_to_one')
    if len(ck)!=len(F_METRICS) or not np.allclose(ck.MAE,ck.MAE_recalc,atol=1e-8):
        raise RuntimeError('Независимая проверка MAE не совпадает')
    base=FORECAST[(FORECAST.protocol=='TEMPORAL_TEST')&(FORECAST.model=='persistence')]
    from scipy.stats import ttest_rel
    pvalues=[];subtests=[]
    for kind in ('ridge','random_forest'):
        other=FORECAST[(FORECAST.protocol=='TEMPORAL_TEST')&(FORECAST.model==kind)&(FORECAST.seed==CFG.SEED)]
        a=base.merge(other,on=['edge','origin_code','destination_code','target_year'],
           suffixes=('_base','_model'),validate='one_to_one')
        if len(a)<40:continue
        a['improvement']=abs(a.actual_base-a.prediction_base)-abs(a.actual_model-a.prediction_model)
        # Кластерная неопределённость: два альтернативных уровня — регион отправления и назначения.
        # Не выдаём отдельные пары за независимые кластеры.
        diagnostics={}
        for cluster_type,col in [('origin','origin_code'),('destination','destination_code')]:
            groups=a.groupby(col).improvement.agg(['sum','count'])
            if len(groups)<8:continue
            vals=groups.to_numpy(float);n=len(vals);rand=np.random.default_rng(CFG.SEED)
            boots=np.empty(1200)
            for k in range(len(boots)):
                sampled=vals[rand.integers(0,n,n)]
                boots[k]=sampled[:,0].sum()/sampled[:,1].sum()
            # Перестановочная кластерная sign-flip-проверка при симметричном нуле,
            # а не псевдо-p из процентилей bootstrap-распределения.
            observed=float(vals[:,0].sum()/vals[:,1].sum())
            sim=np.empty(4000)
            for k in range(len(sim)):
                signs=rand.choice([-1.,1.],size=n)
                sim[k]=float(np.sum(vals[:,0]*signs)/np.sum(vals[:,1]))
            p_sign=float((1+np.count_nonzero(np.abs(sim)>=abs(observed)))/(len(sim)+1))
            diagnostics[cluster_type]={'n':n,'lower':float(np.quantile(boots,.025)),
                   'upper':float(np.quantile(boots,.975)),'p':p_sign}
        if not diagnostics:continue
        actual_delta=float(a.improvement.mean())
        # Для консервативного решения используем наибольшее p двух кластеризаций.
        p=float(max(d['p'] for d in diagnostics.values()))
        subtests.append({'compared_with_persistence':kind,'target_year':int(a.target_year.iloc[0]),
          'paired_rows':len(a),'origin_clusters':diagnostics.get('origin',{}).get('n',0),
          'destination_clusters':diagnostics.get('destination',{}).get('n',0),
          'MAE_improvement':actual_delta,
          'CI_origin_low':diagnostics.get('origin',{}).get('lower',np.nan),
          'CI_origin_high':diagnostics.get('origin',{}).get('upper',np.nan),
          'CI_destination_low':diagnostics.get('destination',{}).get('lower',np.nan),
          'CI_destination_high':diagnostics.get('destination',{}).get('upper',np.nan),
          'p_signflip_conservative_exploratory':p})
        pvalues.append(p)
        # Ошибки по стратам: отчёт без отбрасывания нулей и без отбора модели по TEST.
        thresholds=np.quantile(a.actual_base,[.33,.67])
        a['strata']=np.select([a.actual_base<=thresholds[0],a.actual_base<=thresholds[1]],
                              ['МАЛЫЕ_НАБЛЮДАЕМЫЕ','СРЕДНИЕ_НАБЛЮДАЕМЫЕ'],default='КРУПНЫЕ_НАБЛЮДАЕМЫЕ')
        for stratum,g in a.groupby('strata'):
            BOOTSTRAP.append({'model':kind,'stratum':stratum,'n':len(g),
               'MAE_persistence':float(np.abs(g.actual_base-g.prediction_base).mean()),
               'MAE_model':float(np.abs(g.actual_model-g.prediction_model).mean())})
    if subtests:
        # Holm, m=2: последовательная step-down с ограничением монотонности.
        ordered=sorted(range(len(pvalues)),key=lambda i:pvalues[i]);adjusted=[None]*len(pvalues)
        running=0.
        for rank,i in enumerate(ordered):
            running=max(running,min(1.,pvalues[i]*(len(pvalues)-rank)));adjusted[i]=running
        for i,t in enumerate(subtests):t['p_holm']=adjusted[i]
        STAT_TESTS=subtests
put('Сравнения_с_Holm',pd.DataFrame(STAT_TESTS))
put('Прогноз_ошибки_по_стратам',pd.DataFrame(BOOTSTRAP))

mark_stage('Статистика')

## 11А. Парная региональная неопределённость

Отдельные исследования потоков, занятости и сети; блоковый бутстреп по территориям, поправка Холма.

In [ ]:
# Дополнительная парная статистика по регионам, без независимости отдельных OD-строк.
PAIR_RESULTS=[]
def paired_clusters(df,groupcol,seed=20261123,boots=600,flips=1200):
    a=df.groupby(groupcol).improvement.agg(['sum','count']).to_numpy(float)
    if len(a)<8:return None
    rng=np.random.default_rng(seed);k=len(a)
    sample=a[rng.integers(0,k,size=(boots,k))]
    vals=sample[:,:,0].sum(axis=1)/sample[:,:,1].sum(axis=1)
    signs=rng.choice([-1.,1.],size=(flips,k))
    null=(signs*a[:,0]).sum(axis=1)/a[:,1].sum()
    observed=float(df.improvement.mean())
    return {'n_clusters':k,'low':float(np.quantile(vals,.025)),
            'high':float(np.quantile(vals,.975)),
            'p_exploratory':float((np.count_nonzero(np.abs(null)>=abs(observed))+1)/(flips+1))}
def record_pair(family,target,comparison,frame,groups):
    info={g:paired_clusters(frame,g) for g in groups}
    valid=[x for x in info.values() if x is not None]
    PAIR_RESULTS.append({'family':family,'target':target,'comparison':comparison,'n_test':len(frame),
       'delta_MAE_positive_is_better':float(frame.improvement.mean()),
       'p_value':float(max((x['p_exploratory'] for x in valid),default=np.nan)),
       'cluster_intervals':json.dumps(info,ensure_ascii=False),
       'scope':'Парная проверка на общих ключах; исследовательская статистика, не причинный эффект'})
def join_check(base,test,keys):
    z=base.merge(test,on=keys,suffixes=('_base','_other'),validate='one_to_one')
    if len(z)!=len(base) or len(z)!=len(test) or not np.allclose(z.actual_base,z.actual_other):
        raise AssertionError('Разные TEST-ключи или фактические значения')
    z['improvement']=abs(z.actual_base-z.prediction_base)-abs(z.actual_other-z.prediction_other)
    return z
# H5.1: гравитационная модель против инерции на идентичном OD universe.
if len(GRAVITY_PRED):
    for universe,g in GRAVITY_PRED.groupby('universe'):
        a=g[g.model.eq('persistence')][['edge','target_year','origin_code','destination_code','actual','prediction']]
        b=g[g.model.str.startswith('Poisson')][['edge','target_year','actual','prediction']]
        if len(a) and len(b):record_pair('H5.1_GRAVITY','E01',universe,join_check(a,b,['edge','target_year']),('origin_code','destination_code'))
# H5.5: перенос на новые регионы, исходный TEST одинаков по идентификатору наблюдения.
if len(GRAD_PRED):
    for target,g in GRAD_PRED.groupby('target'):
        a=g[g.model.eq('train_mean')][['observation_id','region_key','actual','prediction']]
        for model in ('ridge','random_forest'):
            b=g[g.model.eq(model)][['observation_id','actual','prediction']]
            if len(a) and len(b):record_pair('H5.5_GRADUATES',target,model,join_check(a,b,['observation_id']),('region_key',))
# H5.6: сетевые показатели 2025, именно регион является единицей бутстрепа.
if len(NET_PRED):
    for target,g in NET_PRED[NET_PRED.protocol.eq('TEMPORAL_TEST')].groupby('target'):
        a=g[g.model.eq('persistence')][['region_code','target_year','actual','prediction']]
        for model in ('rolling_mean_2','ridge','random_forest'):
            b=g[g.model.eq(model)][['region_code','target_year','actual','prediction']]
            if len(a) and len(b):record_pair('H5.6_NETWORK',target,model,join_check(a,b,['region_code','target_year']),('region_code',))
# Holm внутри заранее определённых семей; знак эффекта не меняется.
for family in set(r['family'] for r in PAIR_RESULTS):
    matches=[i for i,x in enumerate(PAIR_RESULTS) if x['family']==family and np.isfinite(x['p_value'])]
    matches.sort(key=lambda i:PAIR_RESULTS[i]['p_value'])
    running=0.
    for rank,i in enumerate(matches):
        running=max(running,min(1.,PAIR_RESULTS[i]['p_value']*(len(matches)-rank)))
        PAIR_RESULTS[i]['p_holm']=running
SCIENCE_PAIRED=put('Парная_статистика_гипотез',pd.DataFrame(PAIR_RESULTS))
if len(SCIENCE_PAIRED) and not SCIENCE_PAIRED.n_test.ge(20).all():
    raise AssertionError('Неприемлемо малая тестовая группа для парного анализа')
print('Дополнительных статистических сравнений:',len(SCIENCE_PAIRED))

## 11Б. Независимый построчный пересчёт всех прогнозных семейств

In [ ]:
# Дополнительные самостоятельные проверки реальных строк, не доверяя только таблицам метрик.
POSTROW_QA=[]
def _group_metric_recalc(frame, keys, actual='actual', pred='prediction'):
    if frame is None or frame.empty:return pd.DataFrame(columns=keys+['n_recalc','MAE_recalc','RMSE_recalc'])
    w=frame[keys+[actual,pred]].copy()
    if not np.isfinite(pd.to_numeric(w[actual],errors='coerce')).all() or not np.isfinite(pd.to_numeric(w[pred],errors='coerce')).all():
        raise AssertionError('В прогнозах есть NaN/бесконечность')
    w['_abs']=(w[actual].astype(float)-w[pred].astype(float)).abs()
    w['_sq']=(w[actual].astype(float)-w[pred].astype(float))**2
    r=w.groupby(keys,dropna=False,as_index=False).agg(n_recalc=('_abs','size'),MAE_recalc=('_abs','mean'),MSE_recalc=('_sq','mean'))
    r['RMSE_recalc']=np.sqrt(r.pop('MSE_recalc'))
    return r

def _audit_metric_family(family, predictions, published, keys, unique_keys):
    if predictions is None or predictions.empty:
        POSTROW_QA.append({'family':family,'status':'NOT_FEASIBLE','tested_models':0,'rows':0,'detail':'Нет наблюдений/прогнозов'});return
    if predictions.duplicated(unique_keys).any():raise AssertionError(f'{family}: дубли прогнозных ключей')
    recalc=_group_metric_recalc(predictions,keys)
    needed=keys+['n_test','MAE','RMSE']
    if published is None or published.empty or not set(needed).issubset(published.columns):
        raise AssertionError(f'{family}: отсутствуют обязательные опубликованные метрики {needed}')
    chk=published[needed].merge(recalc,on=keys,how='outer',validate='one_to_one',indicator=True)
    if not chk['_merge'].eq('both').all():raise AssertionError(f'{family}: разные группы прогнозов и метрик')
    ok=(chk['n_test'].eq(chk['n_recalc']).all() and np.allclose(chk['MAE'],chk['MAE_recalc'],atol=1e-9,rtol=1e-9)
        and np.allclose(chk['RMSE'],chk['RMSE_recalc'],atol=1e-9,rtol=1e-9))
    POSTROW_QA.append({'family':family,'status':'PASS' if ok else 'FAIL','tested_models':int(len(chk)),'rows':int(len(predictions)),
        'detail':'MAE/RMSE и N пересчитаны из каждой строки'} )
    if not ok:raise AssertionError(f'{family}: независимые метрики не совпали')

_audit_metric_family('OD_E01',FORECAST,F_METRICS,['protocol','model','seed'],['protocol','model','seed','edge','target_year'])
_audit_metric_family('GRAVITY_E01',GRAVITY_PRED,GRAVITY_RESULTS,['universe','model'],['universe','model','edge','target_year'])
_audit_metric_family('NETWORK_E01',NET_PRED,NET_METRICS,['target','model','protocol'],['target','model','protocol','region_code','target_year'])
_audit_metric_family('GRADUATES_E04',GRAD_PRED,GRAD_METRICS,['target','model'],['target','model','observation_id'])
RECALCULATED_ALL=put('QA_построчные_прогнозы',pd.DataFrame(POSTROW_QA))
if CFG.MODE=='FULL' and not RECALCULATED_ALL.status.eq('PASS').all():
    raise AssertionError('Полный прогон: один или несколько наборов предсказаний не подтверждены построчно')
# Паспорт каждой прогнозной строки по исходному контракту. Это не превращает ретроспективный
# эксперимент в ex-ante и не выдумывает год доступности показателя занятости.
for frame,kind in ((FORECAST,'OD'),(GRAVITY_PRED,'GRAVITY'),(NET_PRED,'NETWORK'),(GRAD_PRED,'GRADUATES')):
    if frame is None or frame.empty:continue
    frame['entity_id']=(frame['edge'].astype(str) if kind in ('OD','GRAVITY') else
                        frame['region_code'].astype(str)+'|'+frame['target'].astype(str) if kind=='NETWORK'
                        else frame['observation_id'].astype(str))
    if kind=='GRADUATES':
        frame['target_year']=frame['year'];frame['origin_year']=pd.NA;frame['horizon']=pd.NA
        frame['information_cutoff']=pd.NA;frame['split_id']=frame['protocol']
    else:
        frame['origin_year']=frame['target_year'].astype(int)-1
        frame['horizon']=1
        frame['information_cutoff']=frame['origin_year']
        frame['split_id']=(frame['protocol'] if 'protocol' in frame.columns else frame['universe'])
    frame['source_vintage']='Stage I snapshot 2026; retrospective'
    frame['sector']='ВО' if kind=='GRADUATES' else 'СПО'
    frame['programme']=pd.NA


## 12. Графики для предметного анализа (не по количеству алгоритмов)
Рисунки строятся исключительно по заполненным наблюдаемым таблицам. Одна величина — одна шкала; территории подписаны своими исходными именами. Графики предназначены для научного анализа, а не для демонстрации сложности ML.

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','axes.titlesize':12,'axes.labelsize':10,'figure.dpi':130})
FIGURES=[]

def savefig(name):
    p=ROOT/'figures'/(name+'.png')
    plt.tight_layout();plt.savefig(p,dpi=160,bbox_inches='tight');plt.close()
    FIGURES.append(str(p))

def barh(frame,label,value,title,name,n=15):
    f=frame.dropna(subset=[value]).head(n).iloc[::-1]
    if len(f)==0:return
    plt.figure(figsize=(11,max(4,0.43*len(f)+1)))
    plt.barh(f[label].fillna('').astype(str),f[value].astype(float))
    plt.title(title);plt.xlabel(value.replace('_',' '));savefig(name)

if not COVERAGE.empty:
    plt.figure(figsize=(9,4));plt.plot(COVERAGE.год,COVERAGE.наблюдаемых_пар,'o-',label='Зарегистрированные пары')
    plt.plot(COVERAGE.год,COVERAGE.неизменных_пар,'o-',label='Пары, общие всем годам')
    plt.title('Покрытие E01: изменение числа наблюдаемых направлений')
    plt.xlabel('Год');plt.ylabel('Количество направлений');plt.legend();savefig('01_покрытие_OD')
    plt.figure(figsize=(9,4));plt.plot(COVERAGE.год,COVERAGE.все_наблюдаемые_значения,'o-',label='Все зарегистрированные')
    plt.plot(COVERAGE.год,COVERAGE.значения_общих_пар,'o-',label='На общих парах')
    plt.title('E01: наблюдаемые значения при разном покрытии');plt.xlabel('Год');plt.ylabel('Исходная величина E01')
    plt.legend();savefig('02_наблюдаемые_значения')

reg=REGION_LATEST.sort_values('прибывшие_в_наблюдаемом_показателе',ascending=False)
barh(reg,'название_региона','прибывшие_в_наблюдаемом_показателе',
     'Крупнейшие центры притока в наблюдаемом E01, постоянное покрытие','03_центры_СПО')
reg=REGION_LATEST.sort_values('выбывшие_в_наблюдаемом_показателе',ascending=False)
barh(reg,'название_региона','выбывшие_в_наблюдаемом_показателе',
     'Регионы происхождения E01: наблюдаемое исходящее значение','04_исходящие_СПО')
reg=REGION_LATEST.sort_values('наблюдаемый_баланс',ascending=False)
barh(pd.concat([reg.head(7),reg.tail(7)]),'название_региона','наблюдаемый_баланс',
     'Региональный баланс E01 на постоянных парах (не миграция населения)','05_баланс_СПО')

if len(NETWORK_YEARS):
    x=NETWORK_YEARS[NETWORK_YEARS.совокупность=='ПОСТОЯННОЕ_ПОКРЫТИЕ']
    if len(x):
        plt.figure(figsize=(9,4));plt.plot(x.год,x.положительных_рёбер,'o-');plt.title('Сеть E01: положительные рёбра на общей совокупности')
        plt.xlabel('Год');plt.ylabel('Положительных направлений');savefig('06_сетевые_рёбра')
        plt.figure(figsize=(9,4));plt.plot(x.год,x.ARI_сообществ_общих_регионов,'o-');
        plt.title('Устойчивость сетевых сообществ: Adjusted Rand Index')
        plt.xlabel('Год');plt.ylabel('ARI на общих регионах');savefig('07_устойчивость_сообществ')

if 'Образовательные_центры' in TABLES:
    net=TABLES['Образовательные_центры'].sort_values('pagerank',ascending=False)
    barh(net,'регион','pagerank','Наиболее центральные регионы сети E01 (PageRank)','08_сетевые_центры')

if 'ВПО_ведущие_направления' in TABLES:
    tp=TABLES['ВПО_ведущие_направления']
    tp=tp[tp.year.eq(tp.year.max())]
    if len(tp):
        best=tp.sort_values('low',ascending=False).iloc[0]
        same=tp[(tp.region_key==best.region_key)&(tp.level==best.level)&
         (tp.form==best.form)&(tp.admission_type==best.admission_type)&
         (tp.university_type==best.university_type)]
        same=same.sort_values('low',ascending=False).drop_duplicates('code').head(10)
        if len(same):
            barh(same,'code','low',f'ВПО-1: {str(best.region)[:35]} · {str(best.level)[:30]} · точный разрез',
             '09_профиль_специальностей',n=10)

if 'E05_программы_итоги' in TABLES:
    emp=TABLES['E05_программы_итоги'];emp=emp[emp.year.eq(emp.year.max())]
    if not emp.empty:
        # Одно точно наблюдаемое сочетание уровня и пола для сопоставимости:
        dominant=emp.groupby(['level','gender'],dropna=False).строк_отчета.sum().idxmax()
        emp=emp[(emp.level==dominant[0])&(emp.gender==dominant[1])]
    emp=emp[emp.строк_отчета.ge(10)].sort_values('наблюдаемая_медиана_зарплаты',ascending=False)
    barh(emp,'study_area','наблюдаемая_медиана_зарплаты',
     'E05: медианная зарплата по отчётным ячейкам групп подготовки','10_зарплаты_выпускников')

if not F_METRICS.empty:
    m=F_METRICS[(F_METRICS.protocol=='TEMPORAL_TEST')&(F_METRICS.seed==CFG.SEED)&(F_METRICS.model!='random_forest_50')]
    if len(m):barh(m.sort_values('MAE',ascending=False),'model','MAE',
      'Вспомогательный прогноз E01: ошибка на неизменных парах','11_проверка_прогнозов')

# Однотипная шкала (% улучшения MAE относительно инерции) для пяти сетевых величин.
if not NET_METRICS.empty and 'protocol' in NET_METRICS.columns:
    nm=NET_METRICS.loc[NET_METRICS['protocol'].eq('TEMPORAL_TEST') &
                       NET_METRICS['status'].eq('RETROSPECTIVE_TEST')].copy()
    baseline=nm.loc[nm['model'].eq('persistence'),['target','MAE']].rename(columns={'MAE':'baseline_MAE'})
    skill=nm.loc[nm['model'].eq('random_forest')].merge(baseline,on='target',validate='one_to_one')
    skill=skill.loc[skill['baseline_MAE'].gt(0)].copy()
    if len(skill):
        skill['выигрыш_к_инерции_процентов']=100*(skill['baseline_MAE']-skill['MAE'])/skill['baseline_MAE']
        plt.figure(figsize=(11,4.6));plt.barh(skill['target'],skill['выигрыш_к_инерции_процентов'])
        plt.axvline(0,linestyle='--',linewidth=1)
        plt.xlabel('Уменьшение MAE относительно инерции, % (отрицательное = хуже)')
        plt.title('ML-5.6: прогноз характеристик сети против инерционной модели')
        savefig('12_сетевой_прогноз_против_инерции')

print('Сформировано содержательных графиков:',len(FIGURES))
mark_stage('Графики')

## 13. Предметные выводы и структура итогового Excel
Автоматически вычисляются главные численные итоги: охват, регионы-притяжения и происхождения, структура направлений, сеть, специальности, выпускники, ограничения. Каждое утверждение опирается на таблицу, а не на произвольную оценку точности CatBoost. Большие панели остаются в Parquet; Excel содержит читаемые срезы и ссылки на полные данные.

In [ ]:
# Добавляем фактические лидирующие территории по постоянному покрытию; значения подписаны как E01.
if len(REGION_LATEST):
    inc=REGION_LATEST.nlargest(5,'прибывшие_в_наблюдаемом_показателе')
    out=REGION_LATEST.nlargest(5,'выбывшие_в_наблюдаемом_показателе')
    FINDINGS.extend([{'модуль':'ML-5.1 / 5.3','вывод':'Крупнейшие регионы назначения E01',
      'расчёт':'; '.join(f"{r.название_региона}: {r.прибывшие_в_наблюдаемом_показателе:.0f}" for r in inc.itertuples()),
      'ограничение':'СПО, только наблюдаемая E01-совокупность; не размер всех студенческих миграций'},
      {'модуль':'ML-5.1 / 5.3','вывод':'Крупнейшие регионы происхождения E01',
      'расчёт':'; '.join(f"{r.название_региона}: {r.выбывшие_в_наблюдаемом_показателе:.0f}" for r in out.itertuples()),
      'ограничение':'СПО, только наблюдаемая E01-совокупность; не доля выпускников школ'}])
if not F_METRICS.empty:
    v=F_METRICS[(F_METRICS.protocol=='TEMPORAL_TEST')&(F_METRICS.seed==CFG.SEED)&(F_METRICS.model!='random_forest_50')]
    if len(v):
        b=v.loc[v.model=='persistence','MAE']
        best=v.loc[v.MAE.idxmin()]
        FINDINGS.append({'модуль':'ДОПОЛНИТЕЛЬНО','вывод':'Проверка прогнозной пригодности зарегистрированных OD-пар',
          'расчёт':f'На тесте 2025 лучший метод {best.model}, MAE {best.MAE:.3f}; инерционный MAE {b.iloc[0]:.3f}' if len(b) else 'База отсутствует',
          'ограничение':'Ретроспективный результат; не является главным результатом исследования'})

STUDY=put('Исследовательские_выводы',pd.DataFrame(FINDINGS),required=True)
module_rows=[]
for k,goal in QUESTIONS.items():
    evidence=MODULES.get(k,[])
    if not evidence:status='НЕ_ПОДТВЕРЖДЕНО';reason='Нет проведённых тестов'
    else:
        statuses=[r['status'] for r in evidence]
        status='ЧАСТИЧНО_ПО_ДАННЫМ' if any('NOT_FEASIBLE' in s for s in statuses) or 'PARTIALLY_OBSERVED' in statuses else 'НАБЛЮДАЕМАЯ_ЧАСТЬ_ГОТОВА'
        reason='; '.join(r['reason'] for r in evidence)
    module_rows.append({'модуль':k,'исходная_задача':goal,'статус':status,'фактически_измерено_и_ограничения':reason})
MOD_STATUS=put('Семь_исследовательских_задач',pd.DataFrame(module_rows))
put('Ограничения_и_ошибки',pd.DataFrame(QUARANTINE))

# Паспорт научной интерпретации: нельзя смешивать разрезы, а отсутствие данных — это не ноль.
PASSPORT=put('Паспорт_исследования',pd.DataFrame([
 {'параметр':'Проект','значение':'Демографический советник, ML-5 V10.0'},
 {'параметр':'Научная цель','значение':'География образовательной мобильности, центры, организации, специальности, рынок труда'},
 {'параметр':'Единица наблюдения E01','значение':'Регион предыдущего образования × регион обучения × год; СПО'},
 {'параметр':'Единица наблюдения ВПО-1','значение':'Регион × специальность × уровень × форма × категория приема × тип вуза × год'},
 {'параметр':'Единица наблюдения E02','значение':'Организация × год × точный разрез area; контингент не равен приёму'},
 {'параметр':'Единица наблюдения E04/E05','значение':'Отчётная категория выпускников, занятость/зарплата; регион трудоустройства не определен'},
 {'параметр':'Единица наблюдения БДМО','значение':'Муниципальный показатель; нельзя реконструировать географию кампуса'},
 {'параметр':'Источники','значение':f'Каталог Stage I, 52 набора, commit {SOURCE_COMMIT}'},
 {'параметр':'Годы OD','значение':', '.join(map(str,YEARS))},
 {'параметр':'Постоянных наблюдаемых OD-пар','значение':str(len(fixed))},
 {'параметр':'Режим','значение':CFG.MODE},
 {'параметр':'Тип валидации','значение':'Ретроспективная; исторические снапшоты недоступны'},
 {'параметр':'Публикация','значение':'Отключена' if not CFG.PUBLISH_PRIVATE_PRERELEASE else 'Только prerelease после проверки'}]))

print('ГЛАВНЫЕ ПРЕДМЕТНЫЕ РЕЗУЛЬТАТЫ')
display(STUDY[['модуль','вывод','расчёт','ограничение']].head(12))
display(MOD_STATUS[['модуль','статус','фактически_измерено_и_ограничения']])
mark_stage('Выводы')

## 13А. Исходные семь гипотез: доказательства и ограничения

Итог — семь **предметных исследовательских проверок**, а не объединённый «средний балл моделей». Каждое направление связывается с вычисленной таблицей, источником и честным статусом. Статус `НЕ_ВЫПОЛНЕНО_ПО_ДАННЫМ` не заменяется предположением.

In [ ]:
EVIDENCE_MATRIX=[]
def evidence(module,hypothesis,source,tables,explanation,limitation):
    present=[x for x in tables if x in TABLES and len(TABLES[x])]
    absent=[x for x in tables if x not in present]
    EVIDENCE_MATRIX.append({'модуль':module,'гипотеза':hypothesis,'реальные_источники':source,
       'доказательственные_таблицы':' | '.join(present),
       'отсутствующие_результаты':' | '.join(absent),
       'что_реально_проверено':explanation,'границы_вывода':limitation,
       'статус':('NOT_FEASIBLE_HYPOTHESIS_WITH_AVAILABLE_DATA' if module=='ML-5.7' else
                 'NOT_FEASIBLE_CORE_ADMISSIONS' if module=='ML-5.2' else
                 'OBSERVATIONAL_PROGRAM_STRUCTURE_OD_NOT_FEASIBLE' if module=='ML-5.4' else 'RETROSPECTIVELY_TESTED_NOT_CAUSAL' if present else 'NO_OBSERVABLE_RESULT')})
evidence('ML-5.1','Географические, образовательные и экономические факторы потоков','E01; SPO; TRANSPORT если сопоставим',
 ['OD_переходы_покрытия','OD_факторы_и_абляции','Гравитация_метрики','Парная_статистика_гипотез','Прогноз_метрики'],
 'Реальные OD, изменчивость их наблюдаемого охвата, прогнозы на общих маршрутах, добавочная ценность наблюдаемых признаков',
 'Не наблюдаемое постоянное место жительства; ретроспективная проверка; абсолютный приём ВО без OD неизвестен')
evidence('ML-5.2','Привлекательность конкретных организаций и характеристик приёма','E02; SPO registry',
 ['ВО_динамика_контингента','ВО_структура_регионов','Сеть_организаций_регион'],
 'Изменение контингента одной организации в одной страте, пространственная сеть организаций, концентрация',
 'Контингент не равен приёму; происхождение поступивших по отдельным организациям отсутствует')
evidence('ML-5.3','Неравномерный отток и территориальные возможности','E01; SPO registry; D05',
 ['OD_региональные_сдвиги','OD_устойчивость_регионов','Типология_регионов','Региональные_факторные_связи','Региональная_ёмкость_и_потоки','Региональная_ёмкость_корреляции','Региональные_контрольные_сравнения'],
 'Потоки по регионам, устойчивость доноров и центров и контекстные региональные сопоставления',
 'Нет полного знаменателя всех поступивших, нет доказанного причинного эффекта')
evidence('ML-5.4','Региональная неоднородность структуры направлений подготовки','VPO_2_1_1; E01 SPO regional specialties',
 ['ВПО_разнообразие_профилей','ВПО_динамика_кодов','СПО_разнообразие_программ'],
 'Различия профессиональной структуры по точным кодам, формам, типам, годам',
 'Нет OD×специальность; нет доказанного универсального исторического crosswalk')
evidence('ML-5.5','Связь профиля обучения с занятостью и заработком','E04; E05',
 ['E05_зарплаты_по_профилям','E04_устойчивость_регионов','Занятость_сравнение_по_регионам','Занятость_воспроизводимость_метрик'],
 'Различия отчётных страт, тест переноса на незнакомые регионы',
 'Не наблюдаются место работы и постоянное проживание выпускника')
evidence('ML-5.6','Устойчивость сети центров и сообществ во времени','E01',
 ['Сеть_чувствительность','Сеть_концентрация_центров','Сеть_переходы_центров','Сеть_прогноз_метрики'],
 'Центральности, сообщества, динамика и устойчивость к исключению крупнейших маршрутов',
 'Сеть лишь наблюдаемых межрегиональных потоков СПО, изменение покрытий отдельно')
evidence('ML-5.7','Добавочная ценность фактического присутствия образовательных кампусов для муниципальной демографии','BDMO education; D01; SPO registry',
 ['БДМО_профессиональное_образование','Муниципалитеты_демография_QA','Муниципальное_образование_регионы','Муниципальный_перекрестный_охват'],
 'Полный аудит муниципальных образовательных наблюдений, качества и демографической базы',
 'Муниципальные E0/E1 не идентифицируются без действительных исторических географий кампусов')
EVIDENCE_TABLE=put('Реестр_гипотез_и_доказательств',pd.DataFrame(EVIDENCE_MATRIX),required=True)
PANELS=[('edu_org_registry',['Реестр_ВО','Реестр_СПО']),('edu_org_year',['ВО_динамика_контингента','ВО_структура_регионов']),
 ('edu_od_year',['OD_исходные','OD_постоянные_пары']),('edu_program_year',['ВПО_прием_точные_страты','СПО_региональные_спец_индикаторы']),
 ('graduate_outcomes',['Выпускники_E04_страты','Выпускники_E05_страты']),
 ('municipality_education_context',['БДМО_профессиональное_образование','Муниципалитеты_демография_QA','Муниципальное_образование_регионы','Муниципальный_перекрестный_охват'])]
put('Панели_исследования',pd.DataFrame([{'логическая_панель':a,'наличие_частей':'; '.join(x for x in k if x in TABLES and len(TABLES[x])),
 'нет_подтверждённых_частей':'; '.join(x for x in k if x not in TABLES or not len(TABLES[x])),
 'статус':'OBSERVED_ONLY' if all(x in TABLES and len(TABLES[x]) for x in k) else 'PARTIAL_BY_SOURCE'} for a,k in PANELS]))
put('Контроль_покрытия_доказательств',pd.DataFrame([{'модуль':x['модуль'],'число_таблиц':len(x['доказательственные_таблицы'].split(' | ')) if x['доказательственные_таблицы'] else 0,
   'блокированная_часть':x['границы_вывода']} for x in EVIDENCE_MATRIX]))
print('Научные доказательства:',len(EVIDENCE_TABLE),'модулей; панели:',len(PANELS))

## 14. Доказательная база, полные таблицы, научный Excel и компактная выгрузка
Сохраняются полные научно допустимые таблицы в Parquet, проверяемые Excel-срезы, рисунки и состав данных. Первичные 52 файла Stage I не копируются повторно: в манифесте — их хэши и точные идентификаторы. Evidence содержит только **производные** исследовательские панели и небольшие модели, чтобы не раздувать архив. Отдельный компактный пакет для экспертизы можно скачать без повторного выполнения расчётов.

In [ ]:
# Сохранение производных данных БЕЗ потери строк. Excel — читаемая выборка, полная таблица — Parquet.
CORE=['OD_исходные','OD_постоянные_пары','Регионы_постоянный_охват',
      'Региональные_профили','Сеть_регионы_по_годам','Сеть_устойчивость_лет',
      'Реестр_ВО','Реестр_СПО','ВПО_прием_точные_страты','СПО_региональные_спец_индикаторы',
      'Выпускники_E04_страты','Выпускники_E05_страты','Прогноз_OD_наблюдения','Занятость_тестовые_прогнозы','Сеть_прогноз_наблюдения']
DERIVED_FILES=[]
if 'MUNICIPAL_PANEL' in globals() and MUNICIPAL_PANEL.is_file():
    DERIVED_FILES.append({'table':'БДМО_тематические_ячеки_полностью','file':MUNICIPAL_PANEL.name,
       'rows':int(pq.ParquetFile(MUNICIPAL_PANEL).metadata.num_rows),
       'SHA256':sha256(MUNICIPAL_PANEL),'bytes':MUNICIPAL_PANEL.stat().st_size})
if 'MUNICIPAL_PANEL' in globals() and MUNICIPAL_PANEL.is_file():
    # Уже сохранён полный потоковый тематический Parquet. Не создавать копию миллионов записей.
    pass
for i,n in enumerate(CORE):
    if n not in TABLES or TABLES[n].empty:continue
    fname=f'{i+1:02d}_{n}.parquet'
    file=ROOT/'derived'/fname
    TABLES[n].to_parquet(file,index=False,compression='zstd')
    if len(pd.read_parquet(file,columns=[TABLES[n].columns[0]]))!=len(TABLES[n]):
        raise RuntimeError('Parquet: пропали строки '+n)
    DERIVED_FILES.append({'table':n,'file':fname,'rows':len(TABLES[n]),'SHA256':sha256(file),'bytes':file.stat().st_size})
# Таблицы исследовательской проверки сохраняем полностью. Excel — лишь детерминированный срез.
for name in dict.fromkeys(['OD_переходы_покрытия','OD_региональные_сдвиги','OD_устойчивость_регионов',
 'ВО_динамика_контингента','ВО_структура_регионов','ВО_структура_охвата',
 'ВПО_разнообразие_профилей','ВПО_динамика_кодов','СПО_разнообразие_программ',
 'E05_зарплаты_по_профилям','E04_устойчивость_регионов','Занятость_сравнение_по_регионам',
 'Сеть_чувствительность','Сеть_концентрация_центров','Сеть_переходы_центров',
 'Региональная_ёмкость_и_потоки','Региональная_ёмкость_корреляции',
 'Гравитация_метрики','Гравитация_прогнозы','Гравитация_replay','Занятость_воспроизводимость_метрик','Региональные_контрольные_сравнения',
 'Гравитация_метрики','Гравитация_replay','Занятость_воспроизводимость_метрик','Региональные_контрольные_сравнения',
 'OD_факторы_и_абляции','OD_факторные_прогнозы','OD_источники_дистанции','OD_классификация_факторов','D05_динамика_показателей','Муниципальное_образование_регионы','Муниципальный_перекрестный_охват',
 'Парная_статистика_гипотез','Реестр_гипотез_и_доказательств','Панели_исследования','Контроль_покрытия_доказательств',
 'ВПО_ведущие_направления','ВПО_география_специальностей','ВО_контингент_страты','E05_детальные_зарплаты']):
    d=TABLES.get(name)
    if d is None or d.empty or name in CORE:continue
    filepath=ROOT/'derived'/('full_'+hashlib.sha1(name.encode()).hexdigest()[:12]+'.parquet')
    d.to_parquet(filepath,index=False,compression='zstd')
    rows_actual=pq.ParquetFile(filepath).metadata.num_rows
    if rows_actual!=len(d):raise RuntimeError(f'Потеря строк в Parquet {name}')
    DERIVED_FILES.append({'table':name,'file':filepath.name,'rows':len(d),
      'SHA256':sha256(filepath),'bytes':filepath.stat().st_size})
DERIVED_REG=put('Реестр_выходных_Parquet',pd.DataFrame(DERIVED_FILES))
SRC_REG=put('Проверенные_источники',pd.DataFrame([{'asset_id':k,**v} for k,v in SOURCE_USED.items()]))

# Приёмка только семи исходных модулей и проверяемых ограничений — без архивной SHA201.
ORIGINAL_ACCEPTANCE_ROWS=[]
requirements={
 'ML-5.1':('E01, OD с действительными кодами','OD_охват_по_годам',len(INTER)>0),
 'ML-5.2':('Реестры ВО/СПО; реальный приём не подтверждён','Реестр_ВО',len(vo)>0 and len(sp)>0),
 'ML-5.3':('Отток и баланс по единой OD-совокупности','Регионы_постоянный_охват',len(BAL_FIXED)>0),
 'ML-5.4':('Наблюдаемая география программ ВПО/СПО','ВПО_прием_точные_страты',len(PROGRAM)>0),
 'ML-5.5':('Наблюдаемые занятость и зарплата без retention','Занятость_тестовые_метрики',len(E04_CELLS)>0),
 'ML-5.6':('Образовательная сеть, устойчивость и прогноз её метрик','Сеть_прогноз_метрики',len(NET_PRED)>0),
 'ML-5.7':('Муниципальный аудит; кампусный E0/E1 недоступен','БДМО_профессиональное_образование',bool(MUN_CANDIDATE_AUDIT.iloc[0]['совпало_с_эталоном']))
}
for mod,(scope,table,ok) in requirements.items():
    status=('PARTIAL_OBSERVED' if mod in ('ML-5.2','ML-5.4') and ok else
            'NOT_FEASIBLE_E0_E1_WITH_VERIFIED_AUDIT' if mod=='ML-5.7' and ok else
            'COMPUTED_PENDING_SCIENTIFIC_REVIEW' if ok else 'NOT_PROVEN')
    ORIGINAL_ACCEPTANCE_ROWS.append({'модуль':mod,'исходное_требование':scope,
        'исполнено_на_доступных_данных':bool(ok),'статус':status,
        'проверяемая_таблица':table})
ORIGINAL_ACCEPTANCE=put('Приёмка_семи_модулей',pd.DataFrame(ORIGINAL_ACCEPTANCE_ROWS))

EXCEL=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS_V10_2.xlsx'
# Полный перечень научных таблиц; в Excel большие таблицы представлены первыми N строками,
# но полные данные не теряются — отдельный Parquet с истинным числом строк и SHA.
SHEETS=['Паспорт_исследования','DATA_GATE_источников','Исследовательские_выводы','Семь_исследовательских_задач',
 'OD_охват_по_годам','OD_состав','Региональные_профили','Регионы_постоянный_охват',
 'Крупнейшие_OD_направления','Концентрация_направлений','Изменение_общих_маршрутов',
 'Реестр_ВО','Реестр_СПО','Сеть_организаций_регион','ВО_контингент_страты',
 'OD_переходы_покрытия','OD_региональные_сдвиги','OD_устойчивость_регионов',
 'ВО_динамика_контингента','ВО_структура_регионов','ВО_структура_охвата',
 'ВПО_контроль_страт','ВПО_ведущие_направления','ВПО_география_специальностей',
 'СПО_словарь_программ','ВПО_разнообразие_профилей','ВПО_динамика_кодов','СПО_разнообразие_программ',
 'E04_регионы_и_годы','E05_программы_итоги','E05_зарплаты_по_профилям','E04_устойчивость_регионов',
 'Занятость_сравнение_по_регионам',
 'Сеть_устойчивость_лет','Сеть_регионы_по_годам','Сеть_сообщества','Типология_регионов',
 'Сеть_чувствительность','Сеть_концентрация_центров','Сеть_переходы_центров',
 'Контекст_ассоциации','Региональные_факторные_связи','D05_охват_сопоставлений','БДМО_образование_профиль','БДМО_профессиональное_образование','Муниципалитеты_демография_QA',
 'Занятость_тестовые_метрики','Занятость_проверка_моделей',
 'Прогноз_метрики','Прогноз_50_сидов','Сравнения_с_Holm','Прогноз_ошибки_по_стратам',
 'Региональная_ёмкость_и_потоки','Региональная_ёмкость_корреляции',
 'Гравитация_метрики','Гравитация_прогнозы','Гравитация_replay','Занятость_воспроизводимость_метрик','Региональные_контрольные_сравнения',
 'Гравитация_метрики','Гравитация_replay','Занятость_воспроизводимость_метрик','Региональные_контрольные_сравнения',
 'OD_факторы_и_абляции','OD_факторные_прогнозы','OD_источники_дистанции','OD_классификация_факторов','D05_динамика_показателей','Муниципальное_образование_регионы','Муниципальный_перекрестный_охват',
 'Сеть_прогноз_метрики','Сеть_прогноз_future_canary','Сеть_прогноз_повторение','Приёмка_семи_модулей',
 'Каталог_52','Проверенные_источники','Реестр_выходных_Parquet','Ограничения_и_ошибки',
 'Парная_статистика_гипотез','Реестр_гипотез_и_доказательств','Панели_исследования','Контроль_покрытия_доказательств']
# Удаляем повторное включение одного и того же листа, сохраняя исходный порядок.
SHEETS=list(dict.fromkeys(SHEETS))
# Формирование Excel перенесено ПОСЛЕ окончательной научной и технической проверки.
def write_final_excel():
    # Сохранены все научные таблицы; в Excel крупные массивы — детерминированный срез.
    names=list(dict.fromkeys(SHEETS+['QA_построчные_прогнозы','Техническая_QA','Научная_QA_матрица','Приёмка_семи_модулей','Контракт_QA_стадий']))
    rows_excel=[]
    with pd.ExcelWriter(EXCEL,engine='xlsxwriter',engine_kwargs={'options':{'strings_to_urls':False}}) as writer:
        used=set()
        for name in names:
            d=TABLES.get(name)
            if d is None or d.empty:continue
            sheet=''.join(c if c not in '[]:*?/\\' else '_' for c in name)[:31]
            stem=sheet;idx=1
            while sheet in used:sheet=stem[:28]+str(idx);idx+=1
            used.add(sheet)
            cap=20000
            # Разделение: ПОЛНАЯ таблица в Parquet, Excel — один и тот же детерминированный срез.
            d2=d.copy()
            for col in d2.columns:
                if pd.api.types.is_datetime64_any_dtype(d2[col]):d2[col]=d2[col].astype(str)
                elif d2[col].dtype=='object':d2[col]=d2[col].map(lambda x:json.dumps(x,ensure_ascii=False,default=str) if isinstance(x,(dict,list,tuple,set)) else x)
            if len(d2)>cap:
                stable=pd.util.hash_pandas_object(d2.astype(str),index=False,categorize=True).to_numpy(dtype='uint64')
                d2=d2.iloc[np.argsort(stable,kind='mergesort')[:cap]].copy()
                d2=d2.assign(_rowhash=stable[np.argsort(stable,kind='mergesort')[:cap]]).sort_values('_rowhash',kind='mergesort').drop(columns='_rowhash')
            d2.to_excel(writer,sheet_name=sheet,index=False)
            ws=writer.sheets[sheet];ws.freeze_panes(1,0);ws.autofilter(0,0,len(d2),len(d2.columns)-1)
            ws.set_row(0,29)
            for ci,col in enumerate(d2.columns):
                maxlen=max(len(str(col)),int(d2[col].astype(str).str.len().quantile(.85)) if len(d2) else 12)
                ws.set_column(ci,ci,min(42,max(12,maxlen+2)))
            rows_excel.append({'sheet':sheet,'полных_строк':len(d),'строк_в_Excel':len(d2),
                               'полная_таблица_Parquet':next((f['file'] for f in DERIVED_FILES if f['table']==name),'')})
        pd.DataFrame(rows_excel).to_excel(writer,sheet_name='Состав_книги',index=False)
        writer.sheets['Состав_книги'].set_column(0,0,31);writer.sheets['Состав_книги'].set_column(1,3,27)
    print('Excel:',EXCEL.name,'листов:',len(rows_excel)+1)
    return rows_excel



## 15. Внутренняя приёмка и экспорт текущего выполняемого ноутбука

1. **Источник:** каждый используемый Parquet сверяется с `dataset_index.json` по SHA-256 и схеме; необоснованные JOIN и подстановки запрещены.
2. **Предмет:** семь исходных гипотез получают отдельные доказательства или `NOT_FEASIBLE`, без «универсального среднего балла».
3. **Расчёт:** все прогнозы E01, гравитации, сети и выпускников заново пересчитываются построчно и сравниваются с отчётными метриками; отдельно проверяются балансы, ключи, целостность моделей.
4. **Публикация:** только полный локальный QA; в `main` не пишем. Публикация в приватный релиз имеет повторное чтение Git-файлов и сверку SHA-256 скачанных Release assets. **Внешнюю научную экспертизу ноутбук самому себе не присваивает.**


In [ ]:
# 15. Проверка полных результатов, научной постановки, исходников, Excel и архивов.
# Нельзя объявить PASS из наличия листов: проверяются арифметика, связи, ключи и реально исполненные модели.
TECH=[]
def check(name,ok,detail=''):
    row={'check':name,'status':'PASS' if bool(ok) else 'FAIL','detail':str(detail)};TECH.append(row)
    if not ok:raise RuntimeError(f'QA {name}: {detail}')

check('STAGE_I_CATALOG_52',len(ASSETS)==52 and index_sha==CFG.INDEX_BLOB_SHA)
check('STAGE_I_HANDOFF_FILES',len(HANDOFF_PREFLIGHT)==2)
check('PRIVATE_REPOSITORY',bool(REPO_METADATA.get('private')))
check('USED_SOURCE_SHA_AND_SCHEMA',bool(SOURCE_USED) and all(v.get('verified') for v in SOURCE_USED.values()))
check('DATA_GATES',bool((TABLES['DATA_GATE_источников']['status']=='PASS_SCHEMA').all()))
check('SEVEN_MODULES',len(EVIDENCE_TABLE)==7 and set(EVIDENCE_TABLE['модуль'])==set(QUESTIONS))
check('SIX_RESEARCH_PANELS',len(TABLES['Панели_исследования'])==6)
check('OD_CATEGORIES_EXCLUSIVE',len(OD)==len(INTER)+int(OD.category.ne('МЕЖРЕГИОНАЛЬНО').sum()))
check('OD_REGION_CODES',bool(known_region_code(INTER.origin_code).all() and known_region_code(INTER.destination_code).all()))
check('OD_KEYS_UNIQUE',not INTER.duplicated(['year','origin_code','destination_code']).any())
check('OD_FIXED_BY_YEAR',len(FIXED)==len(fixed)*len(YEARS) and set(FIXED.edge).issubset(set(INTER.edge)))
check('OD_BALANCED_FULL',all(abs(float(g['наблюдаемый_баланс'].sum()))<1e-7 for _,g in BAL_FULL.groupby('year')))
check('OD_BALANCED_STABLE',all(abs(float(g['наблюдаемый_баланс'].sum()))<1e-7 for _,g in BAL_FIXED.groupby('year')))
check('OD_COVERAGE_DECOMPOSITION',len(TABLES['OD_переходы_покрытия'])>0 and bool(TABLES['OD_переходы_покрытия']['проверка_арифметики'].all()))
check('ORGANIZATIONS_SEPARATE_VO_SPO',len(vo)>0 and len(sp)>0 and set(vo.sector.unique())=={'ВО'} and set(sp.sector.unique())=={'СПО'})
check('SPECIALTY_AGGREGATES_EXCLUDED',len(PROGRAM)>0 and not PROGRAM['aggregated_code'].any())
check('E04_E05_OBSERVED',len(E04_CELLS)>0 and len(E05_CELLS)>0)
check('NETWORK_SENSITIVITY',len(TABLES.get('Сеть_чувствительность',pd.DataFrame()))>=2)
check('NETWORK_NODES_UNIQUE',not NETWORK_NODES.duplicated(['совокупность','год','region_code']).any())
check('NETWORK_FUTURE_CANARY',bool(NET_CANARY) and all(x['past_features_unaffected'] for x in NET_CANARY))
check('NETWORK_MODELS_RELOADED',bool(NET_REPLAY) and all(x['identical'] for x in NET_REPLAY))
check('OD_FUTURE_CANARY',bool(FUTURE_CANARY.get('tested') and FUTURE_CANARY.get('passed')))
check('OD_MODELS_RELOADED',bool(MODEL_REPLAY) and all(x['predictions_identical'] for x in MODEL_REPLAY))
check('GRADUATE_MODELS_RELOADED',bool(GRAD_REPLAY) and all(x['independent_replay'] for x in GRAD_REPLAY))
check('MODEL_REFIT_DETERMINISTIC',bool(REFITS) and all(x['identical_predictions'] for x in REFITS))
check('GRAVITY_FE_PRESENT',isinstance(GRAVITY_RESULTS,pd.DataFrame) and not GRAVITY_RESULTS.empty and
      (GRAVITY_RESULTS['model']=='Poisson_origin_destination_FE').any())
check('GRAVITY_RELOADED',bool(GRAVITY_REPLAY) and all(x['status']=='REPLAYED' for x in GRAVITY_REPLAY if x.get('status')!='NOT_FEASIBLE_COVERAGE'))
check('GRAVITY_SAME_TEST_SUPPORT',bool((GRAVITY_RESULTS.groupby('universe')['same_support_hash'].nunique()==1).all()))
check('PREDICTIONS_RECALCULATED_4_FAMILIES',len(RECALCULATED_ALL)==4 and RECALCULATED_ALL.status.eq('PASS').all())
check('FORECAST_PROVENANCE_PASSPORT',all(set(('entity_id','target_year','information_cutoff','origin_year','horizon','split_id','source_vintage','sector','programme')).issubset(df.columns) for df in (FORECAST,GRAVITY_PRED,NET_PRED,GRAD_PRED) if len(df)))
check('SCIENTIFIC_PAIRED_TESTS',len(SCIENCE_PAIRED)>=3 and SCIENCE_PAIRED.n_test.ge(20).all())
check('MUNICIPAL_SOURCE_AUDIT',len(MUN_CANDIDATE_AUDIT)>0 and int(MUN_CANDIDATE_AUDIT.iloc[0]['однозначных_ячеек'])>0)
check('MUNICIPAL_CAMPUS_FABRICATION_ABSENT',not any('campus_e0' in str(k).lower() for k in TABLES))
check('DERIVED_REGISTRY_UNIQUE',len(DERIVED_FILES)==len({r['file'] for r in DERIVED_FILES}))
check('DERIVED_PARQUETS_REOPEN',bool(DERIVED_FILES) and all(
      (ROOT/'derived'/r['file']).is_file() and pq.ParquetFile(ROOT/'derived'/r['file']).metadata.num_rows==r['rows']
      and sha256(ROOT/'derived'/r['file'])==r['SHA256'] for r in DERIVED_FILES))
if CFG.MODE=='FULL' and CFG.RUN_FORECAST and len(FORECAST):
    stable=FORECAST.loc[FORECAST.protocol.eq('TEMPORAL_TEST') & FORECAST.model.eq('random_forest_50')]
    base=FORECAST.loc[FORECAST.protocol.eq('TEMPORAL_TEST') & FORECAST.model.eq('persistence')]
    check('50_SEEDS_RETAINED',len(base)>0 and stable.seed.nunique()==50 and len(stable)==50*len(base))
    replay=[a for a in MODEL_REPLAY if a['experiment']=='E01' and a['protocol']=='TEMPORAL_TEST_stability']
    check('50_SEEDS_RELOADED',len(replay)==50 and all(a['predictions_identical'] for a in replay))

# Предметные критерии: недоступные цели не подменять другими переменными.
SCIENCE_CRITERIA=[
 ('H5.1_OD_AND_FACTOR_FORECAST',bool(len(FIXED) and len(FORECAST) and len(GRAVITY_RESULTS)), 'E01; только наблюдаемые OD и ретроспективное тестирование'),
 ('H5.2_ORGANIZATION_OBSERVED',bool(len(vo) and len(sp) and len(CONT)), 'Контингент и реестры — НЕ прием; core admissions NOT_FEASIBLE'),
 ('H5.3_REGIONAL_OUTFLOW',bool(len(BAL_FIXED) and len(TABLES['OD_региональные_сдвиги'])), 'Отток в известной совокупности, не миграция населения'),
 ('H5.4_SPECIALTY_STRUCTURE',bool(len(PROGRAM) and len(TABLES['ВПО_разнообразие_профилей'])), 'География программ, не OD×специальность'),
 ('H5.5_EMPLOYMENT_AND_WAGES',bool(len(E04_CELLS) and len(E05_CELLS) and len(GRAD_PRED)), 'Занятость/зарплаты без географии работы и retention'),
 ('H5.6_NETWORK_STABILITY_AND_FORECAST',bool(len(NET_PRED) and len(TABLES['Сеть_чувствительность'])), 'Стабильные OD-срезы, прогноз только при истории'),
 ('H5.7_MUNICIPAL_AUDIT',bool(len(MUN_CANDIDATE_AUDIT) and len(TABLES['Муниципалитеты_демография_QA'])), 'E0/E1 NOT_FEASIBLE без архивной географии кампусов')]
SCIENCE_ROWS=[]
for scope,ok,explanation in SCIENCE_CRITERIA:
    SCIENCE_ROWS.append({'критерий':scope,'статус':'PASS_OBSERVED_SCOPE' if ok else 'FAIL','основание':explanation})
SCIENTIFIC_BLOCKERS=[r['критерий'] for r in SCIENCE_ROWS if r['статус']=='FAIL']
SCIENCE_ROWS.extend([
 {'критерий':'NON_OBSERVABLE_ADMISSIONS','статус':'NOT_FEASIBLE','основание':'Нет подтверждённого полного приема и происхождения по организациям'},
 {'критерий':'NON_OBSERVABLE_SPECIALTY_OD','статус':'NOT_FEASIBLE','основание':'Нет OD×специальность'},
 {'критерий':'NON_OBSERVABLE_RETENTION','статус':'NOT_FEASIBLE','основание':'Нет подтвержденного региона работы/проживания'},
 {'критерий':'NON_OBSERVABLE_CAMPUS_E0_E1','статус':'NOT_FEASIBLE','основание':'Нет достоверной исторической географии обучения всех кампусов'},
 {'критерий':'HISTORICAL_EX_ANTE','статус':'NOT_FEASIBLE','основание':'Нет архивных информационных vintages'}])
SCIENCE_QA=put('Научная_QA_матрица',pd.DataFrame(SCIENCE_ROWS),required=True)
SCIENTIFIC_STATUS=('SMOKE_NOT_SCIENTIFIC_RUN' if CFG.MODE!='FULL' else
    'BLOCKED_FEASIBLE_COMPUTATIONS' if SCIENTIFIC_BLOCKERS else
    'LOCAL_ALL_FEASIBLE_RESEARCH_PASS_PENDING_EXTERNAL_REVIEW')
TECH_QA=put('Техническая_QA',pd.DataFrame(TECH),required=True)
CONTRACT_STAGE_STATUS={
 'PASS_INPUT':'PASS' if all(r['status']=='PASS' for r in TECH[:3]) else 'FAIL',
 'PASS_DATA':'PASS' if len(TABLES['Панели_исследования'])==6 else 'FAIL',
 'PASS_SPLITS':'PASS' if FUTURE_CANARY.get('passed') and all(r['past_features_unaffected'] for r in NET_CANARY) else 'FAIL',
 'PASS_TRAINING':'PASS' if len(MODEL_REPLAY) and len(GRAD_REPLAY) and len(NET_REPLAY) and len(GRAVITY_REPLAY) else 'FAIL',
 'PASS_VALIDATION':'PASS' if RECALCULATED_ALL.status.eq('PASS').all() and len(SCIENCE_PAIRED)>=3 else 'FAIL',
 'PASS_INTERPRETATION':'PASS' if len(EVIDENCE_TABLE)==7 and len(FINDINGS)>0 else 'FAIL',
 'PASS_OUTPUTS':'PENDING_FINAL_ARCHIVE_QA',
 'PASS_NOTEBOOK':'PENDING_NOTEBOOK_SELF_EXPORT',
 'PASS_GITHUB':'PENDING_PRIVATE_UPLOAD_AND_REDOWNLOAD'}
put('Контракт_QA_стадий',pd.DataFrame([{'этап':k,'статус':v} for k,v in CONTRACT_STAGE_STATUS.items()]),required=True)

# Полный Excel с QA строится ТЕПЕРЬ: не до проведения проверок.
rows_excel=write_final_excel()
check('EXCEL_CREATED',EXCEL.is_file() and EXCEL.stat().st_size>1000)
with zipfile.ZipFile(EXCEL) as z:
    check('EXCEL_ZIP_OPEN',z.testzip() is None)
    xml=z.read('xl/workbook.xml')
    check('EXCEL_HAS_FINAL_QA',b'QA' in xml and b'\xd0' in xml)
check('EXCEL_SHEET_COUNT',len(rows_excel)>=20)
check('SCIENTIFIC_FIGURES',len(FIGURES)>=5 and all(Path(f).is_file() for f in FIGURES))
check('ONE_COMPLETE_NOTEBOOK_IN_MEMORY','SELF_BUNDLE_B64' in globals() and len(SELF_BUNDLE_B64)>1000)

# Проверяемый исполняемый notebook восстанавливается из ВСТРОЕННОЙ копии собственных ячеек.
# Не зависит от возможности скачать код Colab API и не требует внешнего .py.
import zlib,nbformat
bundle=json.loads(zlib.decompress(base64.b64decode(SELF_BUNDLE_B64)).decode('utf-8'))
self_cell={'cell_type':'code','execution_count':None,'metadata':{},'outputs':[],
           'source':"SELF_BUNDLE_B64 = "+repr(SELF_BUNDLE_B64)+"\n",'id':'ml5-v10-embedded-source'}
bundle['cells'].insert(1,self_cell)
nbformat.validate(nbformat.from_dict(bundle))
for ci,cell in enumerate(bundle['cells']):
    if cell['cell_type']=='code':
        compile(cell['source'],f'V10_2_self_cell_{ci}','exec')
NOTEBOOK=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_V10_2_SINGLE_RUN_FULL_PIPELINE.ipynb'
with NOTEBOOK.open('w',encoding='utf-8') as f:nbformat.write(nbformat.from_dict(bundle),f)
check('SELF_CONTAINED_NOTEBOOK_VALID',NOTEBOOK.stat().st_size>10000)
CONTRACT_STAGE_STATUS['PASS_NOTEBOOK']='PASS'
check('MODEL_AND_FORECAST_ARTIFACTS',len(list((ROOT/'models').glob('*.joblib')))>0 and len(list((ROOT/'derived').glob('*.parquet')))>0)
TECH_QA=put('Техническая_QA',pd.DataFrame(TECH),required=True)

# QA и scientific status после всех вычислений, до сборки архивов. Никакого фиктивного FULL SCIENTIFIC PASS.
MANIFEST={
 'version':'ML5_V10.2_SINGLE_RUN_STAGE2','mode':CFG.MODE,'run_started_utc':RUN_START.isoformat(),
 'scientific_goal':'territorial educational mobility, seven original modules',
 'run_id':RUN_ID,'run_mode':'SINGLE_RUN_ALL','source_commit_sha':SOURCE_COMMIT,'index_sha':index_sha,'catalog_assets':len(ASSETS),
 'handoff_preflight':HANDOFF_PREFLIGHT,'private_repo_write_permission_checked':GITHUB_WRITE_PREFLIGHT,
 'sources_used_and_verified':SOURCE_USED,'derived_files':DERIVED_FILES,
 'notebook_file':NOTEBOOK.name,'notebook_sha256':sha256(NOTEBOOK),
 'excel_file':EXCEL.name,'excel_sha256':sha256(EXCEL),
 'model_replay':MODEL_REPLAY,'graduate_replay':GRAD_REPLAY,'network_replay':NET_REPLAY,'gravity_replay':GRAVITY_REPLAY,
 'prediction_counts':{'OD':len(FORECAST),'gravity':len(GRAVITY_PRED),'network':len(NET_PRED),'graduate':len(GRAD_PRED)},
 'table_counts':{k:len(v) for k,v in TABLES.items()},'figures':[Path(f).name for f in FIGURES],
 'scientific_status':SCIENTIFIC_STATUS,'scientific_blockers':SCIENTIFIC_BLOCKERS,
 'technical_status':'PASS','external_independent_audit':'NOT_YET_PERFORMED',
 'publication_policy':'One FULL Run all: mandatory private prerelease, immediate readback, then article pack',
 'source_vintage':'Stage I 2026 snapshot: retrospective, not historical ex ante',
 'seven_modules':ORIGINAL_ACCEPTANCE.to_dict(orient='records'),
 'evidence_matrix':EVIDENCE_TABLE.to_dict(orient='records'),
 'not_feasible_subtasks':[r for r in SCIENCE_ROWS if r['статус']=='NOT_FEASIBLE'],
 'qa_checks':TECH,'contract_stages_before_final_packaging':CONTRACT_STAGE_STATUS.copy(),
 'full_scientific_pass_claimed':False}
META=ROOT/'exports'/'ML5_RUN_MANIFEST_V10_2.json'
QA_FILE=ROOT/'exports'/'ML5_FINAL_QA_V10_2.json'
META.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str,allow_nan=False),'utf-8')
QA_FILE.write_text(json.dumps({'technical':TECH,'technical_status':'PASS',
 'scientific_status':SCIENTIFIC_STATUS,'science_criteria':SCIENCE_ROWS,
 'module_acceptance':ORIGINAL_ACCEPTANCE.to_dict(orient='records'),
 'independent_prediction_checks':POSTROW_QA,
 'contract_stages_before_final_packaging':CONTRACT_STAGE_STATUS.copy(),
 'blockers':SCIENTIFIC_BLOCKERS,'quarantine':QUARANTINE},ensure_ascii=False,indent=2,default=str,allow_nan=False),'utf-8')

# Все артефакты перечислены единожды, проверяются SHA распакованного содержимого.
EVIDENCE=ROOT/'exports'/'ML5_EVIDENCE_V10_2.zip'
files=list(dict.fromkeys(sorted((ROOT/'derived').glob('*.parquet'))+
    sorted((ROOT/'models').glob('*.joblib'))+sorted((ROOT/'figures').glob('*.png'))+
    [EXCEL,META,QA_FILE]))
with zipfile.ZipFile(EVIDENCE,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
    for p in files:z.write(p,p.relative_to(ROOT).as_posix())
with zipfile.ZipFile(EVIDENCE) as z:
    if z.testzip() is not None:raise RuntimeError('Поврежден Evidence ZIP')
    if len(z.namelist())!=len(files):raise RuntimeError('Количество Evidence-файлов не совпало')
    for p in files:
        inner=p.relative_to(ROOT).as_posix()
        if z.getinfo(inner).file_size!=p.stat().st_size:raise RuntimeError('Размер Evidence: '+inner)
        h=hashlib.sha256()
        with z.open(inner) as f:
            for block in iter(lambda:f.read(4*2**20),b''):h.update(block)
        if h.hexdigest()!=sha256(p):raise RuntimeError('SHA256 Evidence: '+inner)
EVID_SHA=sha256(EVIDENCE)
CONTRACT_STAGE_STATUS['PASS_OUTPUTS']='PASS'

# Научный архив создаётся ПОСЛЕ повторной загрузки проверенных файлов из приватного GitHub.
LOCAL_FULL_QA_PASS=(CFG.MODE=='FULL' and all(r['status']=='PASS' for r in TECH)
                    and not SCIENTIFIC_BLOCKERS and SCIENTIFIC_STATUS.startswith('LOCAL_ALL_FEASIBLE'))
FINAL_LOCAL_STATUS='LOCAL_FULL_QA_PASS' if LOCAL_FULL_QA_PASS else 'PARTIAL_OR_BLOCKED'
print('V10.2 ЛОКАЛЬНЫЙ QA:',FINAL_LOCAL_STATUS,'| научный:',SCIENTIFIC_STATUS)
print('Независимая внешняя приёмка: НЕ ПРОВОДИЛАСЬ (её нельзя симулировать)')
print('Excel:',round(EXCEL.stat().st_size/2**20,2),'MiB | Evidence:',round(EVIDENCE.stat().st_size/2**20,2),
      'MiB | Научный архив: после GitHub readback')
mark_stage('Excel и Parquet')
mark_stage('QA и архив')


## 16. Публикация тяжёлых расчётов — обязательный этап того же запуска

В `FULL` результаты помещаются только в новую ветку и приватный GitHub Release. Их нельзя заменять существующим релизом. После публикации все объекты проверяются повторным чтением. При сбое выполнение останавливается; старые данные не используются.

In [ ]:
# 16. PRIVATE GitHub после полного локального QA, повторное скачивание и SHA-256.
# Исходный main не затрагивать. Если научное исполнение заблокировано — архив только локально.
PUBLISH_RECEIPT={'status':'NOT_REQUESTED','scientific_status':SCIENTIFIC_STATUS,
                 'local_qa':FINAL_LOCAL_STATUS,'source_commit':SOURCE_COMMIT,'run_id':RUN_ID}
if CFG.MODE=='FULL' and CFG.PUBLISH_PRIVATE_PRERELEASE and LOCAL_FULL_QA_PASS:
    meta=github_json('')
    if not meta.get('private') or not (meta.get('permissions') or {}).get('push'):
        raise PermissionError('GitHub private repository and write permission required')
    stamp=RUN_START.strftime('%Y%m%d-%H%M%S%f')+'-'+EVID_SHA[:8]
    branch='ml5-v102-full-'+stamp
    tag='ml5-v102-stage2-'+stamp
    segment='stage2/ml05_education/v10.2/'+stamp
    # Branch creation. GitHub 404 is an expected signal that versioned branch is new.
    existing=SESSION.get(API+'/git/ref/heads/'+quote(branch,safe='/'),timeout=(25,60))
    if existing.status_code==404:
        existing.close()
        with request('POST',API+'/git/refs',json={'ref':'refs/heads/'+branch,'sha':SOURCE_COMMIT}):pass
    elif existing.ok:existing.close()
    else:
        status=existing.status_code;existing.close()
        raise RuntimeError('GitHub cannot inspect versioned branch, HTTP '+str(status))
    def commit_document(path,source):
        found=SESSION.get(API+'/contents/'+quote(path,safe='/'),params={'ref':branch},timeout=(25,60))
        if found.status_code==404:old=None
        elif found.ok:old=found.json().get('sha')
        else:raise RuntimeError('GitHub content preflight: HTTP '+str(found.status_code))
        found.close()
        payload={'message':'ML-5 V10.2: full research and verified science data',
                 'branch':branch,'content':base64.b64encode(source.read_bytes()).decode('ascii')}
        if old:payload['sha']=old
        with request('PUT',API+'/contents/'+quote(path,safe='/'),json=payload) as r:
            return r.json()['commit']['sha']
    git_commits=[]
    for srcpath in (NOTEBOOK,EXCEL,META):
        relpath=segment+'/'+srcpath.name
        commitsha=commit_document(relpath,srcpath)
        # Explicit Git read-back from the created branch.
        fetched=github_json('/contents/'+quote(relpath,safe='/'),params={'ref':branch})
        # GitHub Contents returns encoding=none for 1–100 MiB files; use Git Blob API then.
        if fetched.get('encoding')=='base64' and fetched.get('content'):
            bb=base64.b64decode(fetched['content'])
        else:
            blob=github_json('/git/blobs/'+fetched['sha'])
            if blob.get('encoding')!='base64':raise RuntimeError('GitHub blob cannot be decoded')
            bb=base64.b64decode(blob['content'])
        if hashlib.sha256(bb).hexdigest()!=sha256(srcpath):
            raise RuntimeError('GitHub readback SHA differs: '+srcpath.name)
        git_commits.append({'path':relpath,'commit':commitsha,'sha256':sha256(srcpath),'readback_verified':True})
    with request('POST',API+'/releases',json={
        'tag_name':tag,'target_commitish':branch,'name':'ML-5 V10.2 education mobility '+stamp,
        'draft':True,'prerelease':True,
        'body':'Full local QA of all feasible Stage I calculations: PASS. Independent external scientific review: pending. Unobservable subtasks: documented NOT_FEASIBLE.'}) as r:
        release=r.json()
    url=release['upload_url'].split('{',1)[0]
    assets=[]
    def publish_asset(path,name):
        digest=sha256(path)
        with open(path,'rb') as body:
            with request('POST',url,params={'name':name},data=body,headers={'Content-Type':'application/octet-stream'}) as response:
                item=response.json()
        if item.get('size')!=path.stat().st_size:raise RuntimeError('Wrong GitHub asset size: '+name)
        if item.get('digest') and item['digest']!='sha256:'+digest:
            raise RuntimeError('Wrong GitHub asset SHA: '+name)
        assets.append({'name':name,'size':path.stat().st_size,'sha256':digest,
                       'asset_id':item['id'],'server_digest':item.get('digest'),'verified_download':False})
    size_limit=CFG.RELEASE_MAX_PART_MIB*2**20
    if EVIDENCE.stat().st_size<=size_limit:publish_asset(EVIDENCE,EVIDENCE.name)
    else:
        piece=ROOT/'tmp'/'release_piece.tmp'
        try:
            with EVIDENCE.open('rb') as stream:
                part=0
                while True:
                    block=stream.read(size_limit)
                    if not block:break
                    part+=1;piece.write_bytes(block)
                    publish_asset(piece,f'{EVIDENCE.name}.part{part:03d}')
        finally:piece.unlink(missing_ok=True)
    for p in (QA_FILE,):publish_asset(p,p.name)
    # Основные научные таблицы публикуются отдельными assets. Во второй части они будут
    # независимо скачаны из GitHub, прочитаны и пересчитаны; не используются локальные копии.
    SCIENCE_TABLE_NAMES=[
       'OD_исходные','OD_постоянные_пары','Регионы_постоянный_охват','Региональные_профили',
       'OD_переходы_покрытия','OD_региональные_сдвиги','Реестр_ВО','Реестр_СПО',
       'ВО_динамика_контингента','ВПО_прием_точные_страты','ВПО_разнообразие_профилей',
       'СПО_разнообразие_программ','Выпускники_E04_страты','Выпускники_E05_страты',
       'Занятость_тестовые_прогнозы','Прогноз_OD_наблюдения','Гравитация_прогнозы',
       'Гравитация_метрики','Сеть_регионы_по_годам','Сеть_устойчивость_лет',
       'Сеть_чувствительность','Сеть_прогноз_наблюдения','Муниципальное_образование_регионы',
       'Муниципальный_перекрестный_охват','Парная_статистика_гипотез','Реестр_гипотез_и_доказательств']
    source_by_table={r['table']:r for r in DERIVED_FILES}
    required_tables={'OD_исходные','OD_постоянные_пары','Регионы_постоянный_охват',
        'ВПО_прием_точные_страты','Выпускники_E04_страты','Выпускники_E05_страты',
        'Сеть_регионы_по_годам','Прогноз_OD_наблюдения','Занятость_тестовые_прогнозы'}
    missing=required_tables-set(source_by_table)
    if missing:raise RuntimeError('Отсутствуют основные научные таблицы для публикации: '+str(sorted(missing)))
    science_assets=[]
    for name in SCIENCE_TABLE_NAMES:
        row=source_by_table.get(name)
        if row is None:continue # Необязательные страты включаются только если реально рассчитаны
        src=ROOT/'derived'/row['file']
        asset_name='ML5_SCI_'+hashlib.sha1(name.encode('utf-8')).hexdigest()[:14]+'.parquet'
        publish_asset(src,asset_name)
        science_assets.append({'table':name,'asset_name':asset_name,'bytes':src.stat().st_size,
             'sha256':sha256(src),'rows':int(pq.ParquetFile(src).metadata.num_rows),
             'columns':pq.ParquetFile(src).schema_arrow.names})
    SCIENCE_DATA_INDEX={'schema_version':1,'project':'ML-5 V10.2','release_tag':tag,'run_id':RUN_ID,
        'research_source_commit':SOURCE_COMMIT,'research_manifest_sha256':sha256(META),
        'full_evidence_sha256':EVID_SHA,'research_qa_sha256':sha256(QA_FILE),
        'research_excel_sha256':sha256(EXCEL),'tables':science_assets,
        'not_feasible':[x for x in SCIENCE_ROWS if x['статус']=='NOT_FEASIBLE'],
        'original_seven_modules':ORIGINAL_ACCEPTANCE.to_dict(orient='records'),
        'scope':'Snapshot Stage I 2026; observed educational mobility, NOT residential migration',
        'science_pass':'NOT_ASSIGNED_INDEPENDENT_REVIEW_REQUIRED'}
    SCIENCE_DATA_INDEX_FILE=ROOT/'exports'/'ML5_SCIENCE_DATA_INDEX_V10_2.json'
    SCIENCE_DATA_INDEX_FILE.write_text(json.dumps(SCIENCE_DATA_INDEX,ensure_ascii=False,indent=2,default=str,allow_nan=False),'utf-8')
    publish_asset(SCIENCE_DATA_INDEX_FILE,SCIENCE_DATA_INDEX_FILE.name)
    # Независимая повторная загрузка всех опубликованных объектов по GitHub API — ниже в verify_asset_download.
    if CFG.VERIFY_PUBLISHED_DOWNLOADS:
        # Verification of downloaded bytes; no trust in uploader's receipt.
        def verify_asset_download(row):
            resource=API+'/releases/assets/'+str(row['asset_id'])
            with request('GET',resource,headers={'Accept':'application/octet-stream'},stream=True) as response:
                if 'json' in response.headers.get('Content-Type','').lower():
                    fallback=response.json().get('browser_download_url')
                    if not fallback:raise RuntimeError('No asset binary redirect: '+row['name'])
                    response.close()
                    with request('GET',fallback,stream=True) as body:
                        h=hashlib.sha256();received=0
                        for part in body.iter_content(4*2**20):
                            if part:h.update(part);received+=len(part)
                else:
                    h=hashlib.sha256();received=0
                    for part in response.iter_content(4*2**20):
                        if part:h.update(part);received+=len(part)
            row['verified_download']=received==row['size'] and h.hexdigest()==row['sha256']
            if not row['verified_download']:raise RuntimeError('GitHub asset download SHA mismatch: '+row['name'])
        for row in assets:verify_asset_download(row)
    receipt={'status':'PUBLISHED_AND_VERIFIED' if CFG.VERIFY_PUBLISHED_DOWNLOADS else 'UPLOADED_NOT_REDOWNLOADED',
       'url':release['html_url'],'tag':tag,'branch':branch,'segment':segment,
       'source_commit':SOURCE_COMMIT,'run_id':RUN_ID,'local_status':FINAL_LOCAL_STATUS,'scientific_status':SCIENTIFIC_STATUS,
       'external_independent_scientific_review':'PENDING','evidence_sha256':EVID_SHA,
       'git_files':git_commits,'release_assets':assets}
    # Publish immutable receipt to versioned Git segment.
    receipt_file=ROOT/'exports'/'ML5_V10_PUBLICATION_RECEIPT.json'
    CONTRACT_STAGE_STATUS['PASS_GITHUB']='PASS' if all(r['verified_download'] for r in assets) and all(g['readback_verified'] for g in git_commits) else 'FAIL'
    receipt['contract_stages_final']=CONTRACT_STAGE_STATUS.copy()
    receipt_file.write_text(json.dumps(receipt,ensure_ascii=False,indent=2,default=str),'utf-8')
    commit_document(segment+'/'+receipt_file.name,receipt_file)
    with request('PATCH',API+'/releases/'+str(release['id']),json={'draft':False,'prerelease':True}):pass
    PUBLISH_RECEIPT=receipt
    print('Приватный GitHub Release:',receipt['url'])
else:
    if CFG.MODE=='SMOKE':PUBLISH_RECEIPT['status']='BLOCKED_SMOKE'
    elif not LOCAL_FULL_QA_PASS:PUBLISH_RECEIPT['status']='BLOCKED_QA'
    elif not CFG.PUBLISH_PRIVATE_PRERELEASE:PUBLISH_RECEIPT['status']='DISABLED_CONFIGURATION'
RECEIPT=ROOT/'exports'/'ML5_V10_PUBLICATION_RECEIPT.json'
PUBLISH_RECEIPT['contract_stages_final']=CONTRACT_STAGE_STATUS.copy()
RECEIPT.write_text(json.dumps(PUBLISH_RECEIPT,ensure_ascii=False,indent=2,default=str),'utf-8')
print('Результаты расчётов:',FINAL_LOCAL_STATUS,'| публикация:',PUBLISH_RECEIPT['status'])
if CFG.MODE=='FULL' and PUBLISH_RECEIPT['status']!='PUBLISHED_AND_VERIFIED':
    raise RuntimeError('Единый Run all остановлен: публикация научных Parquet текущего запуска не подтверждена')
mark_stage('GitHub публикация')


---

## Сквозной процесс продолжается автоматически

Выполнение **не прерывается** между вычислениями и научным анализом. Если GitHub не подтвердил публикацию, предыдущая ячейка остановила `Run all`. Результаты из прошлых запусков не подставляются.

# ЧАСТЬ II ТОГО ЖЕ НОУТБУКА. Автоматическое повторное чтение публикации текущего запуска.

Никакого нового сеанса, ручного вызова отдельных ячеек или поиска чужого/старого релиза.
Все расчёты далее опираются только на файлы, повторно прочитанные из GitHub.

In [ ]:
# GitHub round-trip В ТОМ ЖЕ Runtime/Run all: только свежий опубликованный тег.
if CFG.MODE=='FULL':
    if PUBLISH_RECEIPT.get('status')!='PUBLISHED_AND_VERIFIED' or PUBLISH_RECEIPT.get('run_id')!=RUN_ID:
        raise RuntimeError('Единый Run all: релиз текущего запуска не прошёл GitHub QA')
    selected_tag=PUBLISH_RECEIPT['tag']
    if not selected_tag:raise RuntimeError('В текущем запуске не создан тег релиза')
    REMOTE_SCIENCE={};REMOTE_AUDIT=[]
    rel=github_json('/releases/tags/'+quote(selected_tag,safe=''))
    if rel.get('draft'):raise RuntimeError('Нельзя читать неопубликованный draft Release')
    assert rel.get('prerelease') is True,'Требуется научный prerelease, не production-релиз'
    listed=rel.get('assets',[])
    if len(listed)>=100:listed=github_json('/releases/'+str(rel['id'])+'/assets',params={'per_page':100,'page':1})
    asset_map={x['name']:x for x in listed}
    index_name='ML5_SCIENCE_DATA_INDEX_V10_2.json'
    if index_name not in asset_map:raise RuntimeError('В GitHub Release нет машинного индекса научных Parquet')
    def get_asset_verified(meta,expect_sha=None,expect_bytes=None,destination=None):
        endpoint=API+'/releases/assets/'+str(meta['id'])
        h=hashlib.sha256();got=0;payload=[]
        with request('GET',endpoint,headers={'Accept':'application/octet-stream'},stream=True) as r:
            if 'json' in r.headers.get('Content-Type','').lower():
                url=r.json().get('browser_download_url')
                if not url:raise RuntimeError('GitHub возвратил JSON вместо asset без URL: '+meta['name'])
                r.close();response=request('GET',url,stream=True)
            else:response=None
            try:
                source=response if response is not None else r
                if destination:
                    with open(destination,'wb') as out:
                        for block in source.iter_content(4*2**20):
                            if block:h.update(block);got+=len(block);out.write(block)
                else:
                    for block in source.iter_content(4*2**20):
                        if block:h.update(block);got+=len(block);payload.append(block)
            finally:
                if response is not None:response.close()
        actual=h.hexdigest()
        if expect_sha and actual!=expect_sha:raise RuntimeError('GitHub SHA не совпал: '+meta['name'])
        if expect_bytes is not None and got!=expect_bytes:raise RuntimeError('GitHub размер не совпал: '+meta['name'])
        if got!=int(meta['size']):raise RuntimeError('GitHub asset size не совпал с API: '+meta['name'])
        return b''.join(payload) if destination is None else {'bytes':got,'sha256':actual}
    index_digest=asset_map[index_name].get('digest','')
    index_hash=index_digest[7:] if index_digest.startswith('sha256:') else None
    index_raw=get_asset_verified(asset_map[index_name],index_hash)
    SCIENCE_DATA_INDEX=json.loads(index_raw.decode('utf-8'))
    if SCIENCE_DATA_INDEX['release_tag']!=selected_tag:raise RuntimeError('Тег внутри научного каталога не совпадает')
    if SCIENCE_DATA_INDEX['research_source_commit']!=SOURCE_COMMIT:
        raise RuntimeError('GitHub: научные данные опубликованы из другого снимка Stage I')
    if SCIENCE_DATA_INDEX.get('run_id')!=RUN_ID:
        raise RuntimeError('GitHub: научные файлы относятся к ДРУГОМУ выполнению ноутбука')
    required_tables={'OD_исходные','OD_постоянные_пары','Регионы_постоянный_охват',
        'ВПО_прием_точные_страты','Выпускники_E04_страты','Выпускники_E05_страты',
        'Сеть_регионы_по_годам','Прогноз_OD_наблюдения','Занятость_тестовые_прогнозы'}
    if not required_tables.issubset({r['table'] for r in SCIENCE_DATA_INDEX['tables']}):
        raise RuntimeError('В опубликованном каталоге отсутствуют обязательные научные таблицы')
    REMOTE_DIR=ROOT/'remote_verified';REMOTE_DIR.mkdir(exist_ok=True)
    REMOTE_SCIENCE={};REMOTE_AUDIT=[]
    for row in SCIENCE_DATA_INDEX['tables']:
        asset=asset_map.get(row['asset_name'])
        if not asset:raise RuntimeError('В релизе потерян научный Parquet: '+row['table'])
        target=REMOTE_DIR/row['asset_name']
        result=get_asset_verified(asset,row['sha256'],int(row['bytes']),target)
        pf=pq.ParquetFile(target)
        if int(pf.metadata.num_rows)!=int(row['rows']):raise RuntimeError('Изменилось число строк: '+row['table'])
        if list(pf.schema_arrow.names)!=row['columns']:raise RuntimeError('Изменилась схема: '+row['table'])
        REMOTE_SCIENCE[row['table']]=target
        REMOTE_AUDIT.append({'таблица':row['table'],'строк':int(row['rows']),'байт':result['bytes'],
                             'SHA256':result['sha256'],'источник':'приватный GitHub Release','статус':'PASS'})
    if len(REMOTE_SCIENCE)!=len(SCIENCE_DATA_INDEX['tables']):raise RuntimeError('Не все таблицы повторно получены')
    # Одна цепочка: receipt обязан существовать из предыдущей программной ячейки.
    if PUBLISH_RECEIPT['tag']!=selected_tag:raise RuntimeError('Тег релиза не совпал с текущим запуском')
    # Восстановление исходного XLSX и manifest из версионированной ветки; QA — из Release.
    segment_stamp=selected_tag.removeprefix('ml5-v102-stage2-')
    branch=rel['target_commitish']
    saved_segment='stage2/ml05_education/v10.2/'+segment_stamp
    def restore_git_output(filename,destination,expected_sha):
        path=saved_segment+'/'+filename
        meta=github_json('/contents/'+quote(path,safe='/'),params={'ref':branch})
        if meta.get('encoding')=='base64' and meta.get('content'):
            data=base64.b64decode(meta['content'])
        else:
            blob=github_json('/git/blobs/'+meta['sha'])
            if blob.get('encoding')!='base64':raise RuntimeError('Нет GitHub blob для восстановления '+filename)
            data=base64.b64decode(blob['content'])
        destination.write_bytes(data)
        if hashlib.sha256(data).hexdigest()!=expected_sha:
            raise RuntimeError('Восстановленный GitHub файл имеет неправильный SHA: '+filename)
    EXCEL=globals().get('EXCEL',ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS_V10_2.xlsx')
    META=globals().get('META',ROOT/'exports'/'ML5_RUN_MANIFEST_V10_2.json')
    QA_FILE=globals().get('QA_FILE',ROOT/'exports'/'ML5_FINAL_QA_V10_2.json')
    ROOT.joinpath('exports').mkdir(parents=True,exist_ok=True)
    manifest_expected=SCIENCE_DATA_INDEX['research_manifest_sha256']
    if not META.is_file() or sha256(META)!=manifest_expected:
        restore_git_output(META.name,META,manifest_expected)
    if not EXCEL.is_file() or sha256(EXCEL)!=SCIENCE_DATA_INDEX['research_excel_sha256']:
        restore_git_output(EXCEL.name,EXCEL,SCIENCE_DATA_INDEX['research_excel_sha256'])
    qasset=asset_map.get(QA_FILE.name)
    if not qasset:raise RuntimeError('В GitHub Release нет исходного QA-файла')
    if not QA_FILE.is_file() or sha256(QA_FILE)!=SCIENCE_DATA_INDEX['research_qa_sha256']:
        get_asset_verified(qasset,SCIENCE_DATA_INDEX['research_qa_sha256'],int(qasset['size']),QA_FILE)
    print('GitHub readback:',len(REMOTE_SCIENCE),'научных таблиц, 100% проверены SHA, схема и количество строк')

else:
    REMOTE_SCIENCE={};REMOTE_AUDIT=[];SCIENCE_DATA_INDEX={}
mark_stage('Данные из GitHub')

## II.2. Независимые предметные расчёты по скачанным данным

Все расчёты ниже выполняются на вновь полученных копиях GitHub Parquet. Не используются локальные таблицы из первой части в качестве источника научных чисел.

In [ ]:
# ЧАСТЬ II, ЭТАП 2. Расчёт семи тематических блоков из повторно скачанных данных.
SCIENCE_SUMMARIES={};SCIENCE_STATEMENTS=[];SCIENCE_ISSUES=[]
def remote_table(name,required=False):
    path=REMOTE_SCIENCE.get(name)
    if path is None:
        if required:raise RuntimeError('Нет удалённой научной таблицы '+name)
        return pd.DataFrame()
    return pd.read_parquet(path)
def summary(name,frame):
    if isinstance(frame,pd.DataFrame) and len(frame):SCIENCE_SUMMARIES[name]=frame.reset_index(drop=True)
def statement(mod,observation,limitation):
    SCIENCE_STATEMENTS.append({'модуль':mod,'наблюдаемый_результат':observation,'границы_интерпретации':limitation})
if CFG.MODE=='FULL':
    od=remote_table('OD_исходные',True);fixed_remote=remote_table('OD_постоянные_пары',True)
    balance=remote_table('Регионы_постоянный_охват',True);vpo=remote_table('ВПО_прием_точные_страты',True)
    e04=remote_table('Выпускники_E04_страты',True);e05=remote_table('Выпускники_E05_страты',True)
    net=remote_table('Сеть_регионы_по_годам',True)
    if not {'year','edge','value'}.issubset(fixed_remote.columns):raise RuntimeError('Схема OD отличается от утверждённой')
    if not {'year','наблюдаемый_баланс','название_региона'}.issubset(balance.columns):raise RuntimeError('Схема баланса отличается')
    od_y=fixed_remote.groupby('year',as_index=False).agg(направлений=('edge','nunique'),наблюдаемая_величина=('value','sum')).sort_values('year')
    summary('01_Динамика_OD',od_y)
    balance_check=balance.groupby('year',as_index=False)['наблюдаемый_баланс'].sum()
    if not np.allclose(balance_check['наблюдаемый_баланс'].astype(float),0.,atol=1e-7):raise RuntimeError('Повторный расчёт балансов не равен нулю')
    summary('02_Проверка_балансов',balance_check)
    # Пересчёт баланса ПО КАЖДОМУ региону из удалённых OD-строк, а не только нулевой суммарный баланс.
    expected_cols={'year','origin_code','destination_code','value'}
    if not expected_cols.issubset(fixed_remote.columns):raise RuntimeError('В архиве нет происхождения/назначения OD')
    out=fixed_remote.groupby(['year','origin_code'],as_index=False).value.sum().rename(columns={'origin_code':'region_code','value':'computed_out'})
    inc=fixed_remote.groupby(['year','destination_code'],as_index=False).value.sum().rename(columns={'destination_code':'region_code','value':'computed_in'})
    reconstructed=out.merge(inc,on=['year','region_code'],how='outer',validate='one_to_one').fillna({'computed_out':0,'computed_in':0})
    reconstructed['computed_balance']=reconstructed['computed_in']-reconstructed['computed_out']
    verified=balance.merge(reconstructed,on=['year','region_code'],how='outer',validate='one_to_one',indicator=True)
    if not verified['_merge'].eq('both').all():raise RuntimeError('OD: список регионов/лет отличается от балансной ведомости')
    for a,b in [('наблюдаемый_баланс','computed_balance'),('прибывшие_в_наблюдаемом_показателе','computed_in'),('выбывшие_в_наблюдаемом_показателе','computed_out')]:
        if not np.allclose(verified[a].to_numpy(float),verified[b].to_numpy(float),rtol=1e-10,atol=1e-7):
            raise RuntimeError('OD: не совпал независимый расчёт по региону: '+a)
    summary('02A_Баланс_по_каждому_региону',verified[['year','region_code','computed_in','computed_out','computed_balance']])

    latest=int(balance['year'].max());b=balance.loc[balance.year.eq(latest)].copy()
    summary('03_Центры_притяжения',b.sort_values(['прибывшие_в_наблюдаемом_показателе','название_региона'],ascending=[False,True]).head(30))
    summary('04_Регионы_оттока',b.sort_values(['выбывшие_в_наблюдаемом_показателе','название_региона'],ascending=[False,True]).head(30))
    statement('ML-5.1','Наблюдаемые OD СПО за '+str(len(od_y))+' лет, '+str(int(od_y.направлений.min()))+' постоянных пар в каждом году',
       'Это потоки предыдущего образования к месту обучения, а не подтверждённая смена проживания. Изменение охвата анализировать отдельно.')
    statement('ML-5.3','Баланс наблюдаемых межрегиональных направлений тождественно нулевой, региональные ранги вычислены',
       'Итоги только по постоянным наблюдаемым OD-парам; не доля всех уехавших школьных выпускников.')
    orgvo=remote_table('Реестр_ВО',True);orgspo=remote_table('Реестр_СПО',True)
    summary('05_ВО_организации',orgvo.groupby('year',as_index=False).agg(записей=('id','size'),организаций=('id','nunique')))
    summary('06_СПО_организации',orgspo.groupby('year',as_index=False).agg(записей=('id','size'),организаций=('id','nunique')))
    statement('ML-5.2','Сопоставлена наблюдаемая численность записей ВО и СПО по годам',
       'Нет подтверждённого полного зачисления и происхождения поступающих в отдельные организации.')
    if not {'year','code','low','region_key'}.issubset(vpo.columns):raise RuntimeError('ВПО-1: нет точных исходных разрезов')
    prog=vpo.loc[vpo['code'].notna() & pd.to_numeric(vpo['low'],errors='coerce').ge(0)].copy()
    summary('07_ВПО_коды_и_география',prog.groupby('year',as_index=False).agg(наблюдаемых_региональных_страт=('code','size'),кодов=('code','nunique'),регионов=('region_key','nunique')))
    div=remote_table('ВПО_разнообразие_профилей')
    if len(div):summary('08_Разнообразие_программ',div)
    statement('ML-5.4','Сохранены документированные региональные страты специальностей ВПО-1 без итогового кода «Всего»',
       'Не допускается интерпретировать географию кода обучения как регион происхождения студента.')
    if not {'year','employed_pct','salary_median','rate_valid','salary_valid'}.issubset(e04.columns):raise RuntimeError('E04: неполные измерения')
    if not {'year','employed_pct','salary_median','rate_valid','salary_valid'}.issubset(e05.columns):raise RuntimeError('E05: неполные измерения')
    outcome=[]
    for source_name,table in [('E04',e04),('E05',e05)]:
        for yr,g in table.groupby('year'):
            u=g.loc[g.rate_valid.fillna(False)];w=g.loc[g.salary_valid.fillna(False)]
            outcome.append({'источник':source_name,'год':int(yr),'отчётных_страт':len(g),
              'медиана_доли_занятых_по_стратам':float(u.employed_pct.median()) if len(u) else np.nan,
              'медиана_зарплаты_по_стратам':float(w.salary_median.median()) if len(w) else np.nan,
              'достоверных_строк_занятости':len(u),'достоверных_строк_зарплаты':len(w)})
    summary('09_Выпускники_рынок_труда',pd.DataFrame(outcome))
    statement('ML-5.5','Медианы отчётных ячеек E04 и E05 по годам, с отдельным охватом занятости и зарплат',
       'Это не общая занятость всех выпускников, не география их работы и не закрепление.')
    if not {'год','совокупность','входящая_сила','region_code','pagerank'}.issubset(net.columns):raise RuntimeError('Сеть: недостаточная схема')
    n=net[net['совокупность'].eq('ПОСТОЯННОЕ_ПОКРЫТИЕ')].copy()
    if n.empty:raise RuntimeError('Нет сети на постоянной совокупности')
    checks=n.groupby('год',as_index=False).agg(регионы=('region_code','nunique'),сумма_PageRank=('pagerank','sum'),входящая_сила=('входящая_сила','sum'))
    if not np.allclose(checks['сумма_PageRank'].to_numpy(float),1,atol=1e-6):raise RuntimeError('Сетевой PageRank не нормирован')
    summary('10_Сеть_общая_структура',checks)
    positive=fixed_remote.loc[fixed_remote.value.gt(0)]
    net_in=positive.groupby(['year','destination_code'],as_index=False).value.sum().rename(columns={'year':'год','destination_code':'region_code','value':'recalc_strength'})
    n_check=n.merge(net_in,on=['год','region_code'],how='left',validate='one_to_one')
    if not np.allclose(n_check['входящая_сила'].to_numpy(float),n_check['recalc_strength'].fillna(0).to_numpy(float),atol=1e-7):
        raise RuntimeError('Сетевой входящий вес не совпал с собственными OD-источниками')
    summary('10A_Сетевая_сила_проверена',n_check[['год','region_code','входящая_сила','recalc_strength']])

    summary('11_Сеть_центры',n[n['год'].eq(n['год'].max())].sort_values('входящая_сила',ascending=False).head(30))
    statement('ML-5.6','Сеть пересчитана по постоянным наблюдаемым рёбрам, центральности и входящей силе',
       'PageRank и показатели сети описывают наблюдаемые OD, а не доказанную миграционную привлекательность проживания.')
    municipal=remote_table('Муниципальное_образование_регионы')
    if len(municipal):summary('12_Муниципальное_образование',municipal)
    statement('ML-5.7','Сохранены только реально сопоставимые муниципальные образовательные наблюдения, если имеются',
       'Муниципальный E0/E1 по кампусам NOT_FEASIBLE при отсутствии подтверждённой исторической географии филиалов.')
    # Ключевые показатели качества: заново, из опубликованных построчных прогнозов.
    for mod,tab in [('OD','Прогноз_OD_наблюдения'),('Гравитация','Гравитация_прогнозы'),('Сеть','Сеть_прогноз_наблюдения'),('Выпускники','Занятость_тестовые_прогнозы')]:
        d=remote_table(tab)
        if d.empty:raise RuntimeError('В опубликованном научном архиве отсутствует прогноз: '+mod)
        actual_col='actual' if 'actual' in d.columns else 'y_true' if 'y_true' in d.columns else None
        pred_col='prediction' if 'prediction' in d.columns else 'y_pred' if 'y_pred' in d.columns else None
        if actual_col is None or pred_col is None:
            raise RuntimeError('В опубликованном научном Parquet нет колонок факта/прогноза: '+mod)
        keys=[c for c in ['protocol','universe','target','model','seed'] if c in d.columns]
        if not keys:raise RuntimeError('Прогноз '+mod+': нет имени модели и разбиения')
        d=d.loc[pd.to_numeric(d[actual_col],errors='coerce').notna() & pd.to_numeric(d[pred_col],errors='coerce').notna()].copy()
        d['_abs']=(d[actual_col]-d[pred_col]).abs()
        m=d.groupby(keys,dropna=False,as_index=False).agg(n_test=('_abs','size'),MAE=('_abs','mean'))
        summary('13_Прогнозы_'+mod,m)
    summary('14_Отчёт_об_ограничениях',pd.DataFrame(SCIENCE_DATA_INDEX.get('not_feasible',[])))
    summary('15_Семь_модулей',pd.DataFrame(SCIENCE_DATA_INDEX['original_seven_modules']))
    summary('16_Контроль_GitHub',pd.DataFrame(REMOTE_AUDIT))
    summary('17_Результаты_вопросы',pd.DataFrame(SCIENCE_STATEMENTS))
    required_recomputed={'13_Прогнозы_OD','13_Прогнозы_Гравитация','13_Прогнозы_Сеть','13_Прогнозы_Выпускники'}
    if not required_recomputed.issubset(SCIENCE_SUMMARIES):raise RuntimeError('Недостаёт независимых метрик прогноза из опубликованных Parquet')
    if len(SCIENCE_SUMMARIES)<12:raise RuntimeError('Научный синтез создал недостаточно предметных таблиц')
    print('Из реально полученных в GitHub данных пересчитаны',len(SCIENCE_SUMMARIES),'научных таблиц')
mark_stage('Научный синтез')


## II.3. Итоговая научная выгрузка того же `Run all`

Полные тяжёлые модели и Evidence сохраняются в GitHub. Из подтверждённых данных текущей публикации собирается научный архив с таблицами, графиками, ограничениями, SHA и ссылкой на релиз; в конце выполняется скачивание. Никакого второго запуска.

In [ ]:
# ЧАСТЬ II, ЭТАП 3. XLSX/ZIP научной выгрузки и автоматическое скачивание в Colab.
if CFG.MODE=='FULL' and (PUBLISH_RECEIPT.get('run_id')!=RUN_ID or not REMOTE_SCIENCE):
    raise RuntimeError('Нельзя собирать научный архив не из текущего GitHub Run all')
SCIENCE_ARCHIVE=ROOT/'exports'/'ML5_V10_2_ARTICLE_RESEARCH_PACK.zip'
if CFG.MODE=='FULL':
    SYNTHESIS_BOOK=ROOT/'exports'/'ML5_V10_2_VERIFIED_SCIENTIFIC_SYNTHESIS.xlsx'
    with pd.ExcelWriter(SYNTHESIS_BOOK,engine='xlsxwriter',engine_kwargs={'options':{'strings_to_urls':False}}) as writer:
        used=set()
        for name,df in SCIENCE_SUMMARIES.items():
            sheet=name[:31];original=sheet;n=1
            while sheet in used:sheet=original[:27]+'_'+str(n);n+=1
            used.add(sheet)
            excerpt=df.head(20000).copy()
            for col in excerpt.columns:
                if excerpt[col].dtype=='object':excerpt[col]=excerpt[col].map(lambda x:json.dumps(x,ensure_ascii=False,default=str) if isinstance(x,(dict,list,tuple,set)) else x)
            excerpt.to_excel(writer,sheet_name=sheet,index=False)
            ws=writer.sheets[sheet];ws.freeze_panes(1,0);ws.autofilter(0,0,len(excerpt),len(excerpt.columns)-1)
            for ci,col in enumerate(excerpt.columns):ws.set_column(ci,ci,min(44,max(13,len(str(col))+4)))
    ARTICLE_TABLES=ROOT/'article_tables';ARTICLE_TABLES.mkdir(exist_ok=True)
    ARTICLE_SHEETS=[]
    for name,frame in SCIENCE_SUMMARIES.items():
        identifier=hashlib.sha1(name.encode('utf-8')).hexdigest()[:10]
        table_path=ARTICLE_TABLES/(identifier+'.parquet')
        frame.to_parquet(table_path,index=False,compression='zstd')
        if pq.ParquetFile(table_path).metadata.num_rows!=len(frame):raise RuntimeError('Повреждена научная таблица '+name)
        ARTICLE_SHEETS.append({'name':name,'file':table_path.name,'rows':len(frame),'sha256':sha256(table_path)})
    # Отдельные иллюстрации пересчитываются из повторно скачанных GitHub Parquet.
    science_figures=ROOT/'science_figures';science_figures.mkdir(exist_ok=True)
    x=SCIENCE_SUMMARIES['01_Динамика_OD']
    fig,ax=plt.subplots(figsize=(9,4.8));ax.plot(x['year'],x['наблюдаемая_величина'],marker='o')
    ax.set(title='СПО: наблюдаемая величина на неизменных OD-парах',xlabel='Год',ylabel='Значение показателя E01')
    ax.grid(alpha=.2);fig.tight_layout();chart1=science_figures/'education_od_fixed_trends.png';fig.savefig(chart1,dpi=160);plt.close(fig)
    x=SCIENCE_SUMMARIES['09_Выпускники_рынок_труда']
    fig,ax=plt.subplots(figsize=(9,4.8))
    for srcname,g in x.groupby('источник'):
        ax.plot(g['год'],g['медиана_доли_занятых_по_стратам'],marker='o',label=srcname)
    ax.set(title='Выпускники: медиана отчётных долей занятости',xlabel='Год',ylabel='Медиана отчётных долей, %')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout();chart2=science_figures/'graduate_employment_observed.png';fig.savefig(chart2,dpi=160);plt.close(fig)
    ARTICLE_FIGURES=[chart1,chart2]
    # Научные выводы — строго в пределах полученных чисел; не генерируем причинные или ex-ante утверждения.
    lines=['# ML-5 V10.2. Данные и результаты для научного анализа','',
           'Основание: повторно скачанные Parquet из приватного GitHub Release. Проверено SHA-256, количество строк и схема каждого файла.',
           'GitHub Release: '+PUBLISH_RECEIPT.get('url','https://github.com/'+CFG.REPO+'/releases/tag/'+SCIENCE_DATA_INDEX['release_tag']),
           'Источник: '+SCIENCE_DATA_INDEX['research_source_commit'],'',
           '## Семь предметных направлений']
    for row in SCIENCE_STATEMENTS:
        lines.extend(['','### '+row['модуль'],row['наблюдаемый_результат'],'Граница вывода: '+row['границы_интерпретации']])
    lines.extend(['','## Принципиальные ограничения','- Ретроспективная аналитика на снимке Stage I, а не ex ante.',
        '- Нет индивидуальных миграционных траекторий, закрепления и реконструкции кампусов.',
        '- Прогнозы рассматриваются как вспомогательные модели; отрицательные результаты не скрываются.',
        '- Подготовленные файлы предназначены для смыслового анализа и статьи, не заменяют внешнюю научную рецензию.',
        '', '## Содержимое','- Предметные Parquet: полный объём по указанным в DATA_INDEX тематическим таблицам.',
        '- Итоговая Excel-книга и отдельная книга повторных расчётов.',
        '- Графики и проверенные метаданные.',
        '- Бинарные модели и полный расчётный Evidence сохранены только в GitHub Release.'])
    REPORT=ROOT/'exports'/'ML5_V10_2_RESEARCH_METHODS_AND_FINDINGS.md'
    REPORT.write_text('\n'.join(lines)+'\n',encoding='utf-8')
    ROUNDTRIP_JSON=ROOT/'exports'/'ML5_V10_2_REMOTE_VALIDATION.json'
    ROUNDTRIP_JSON.write_text(json.dumps({'release_tag':SCIENCE_DATA_INDEX['release_tag'],'run_id':RUN_ID,'verified_assets':REMOTE_AUDIT,
        'scientific_tables':{name:len(frame) for name,frame in SCIENCE_SUMMARIES.items()},
        'limitations':SCIENCE_DATA_INDEX.get('not_feasible',[]),'independent_external_pass':False},
        ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    files_to_pack=[EXCEL,SYNTHESIS_BOOK,META,QA_FILE,REPORT,ROUNDTRIP_JSON]
    if 'RECEIPT' in globals() and RECEIPT.is_file():files_to_pack.append(RECEIPT)
    with zipfile.ZipFile(SCIENCE_ARCHIVE,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
        for f in files_to_pack:z.write(f,f.name)
        z.writestr('SCIENCE_DATA_INDEX.json',json.dumps(SCIENCE_DATA_INDEX,ensure_ascii=False,indent=2,default=str))
        for name,path in REMOTE_SCIENCE.items():z.write(path,'science_data/'+path.name)
        z.writestr('RECALCULATED_TABLE_INDEX.json',json.dumps(ARTICLE_SHEETS,ensure_ascii=False,indent=2))
        for r in ARTICLE_SHEETS:z.write(ARTICLE_TABLES/r['file'],'recalculated_tables/'+r['file'])
        for figure in ARTICLE_FIGURES:z.write(figure,'science_figures/'+figure.name)
        for figure in FIGURES:
            if Path(figure).is_file():z.write(figure,'figures/'+Path(figure).name)
    with zipfile.ZipFile(SCIENCE_ARCHIVE) as z:
        if z.testzip() is not None:raise RuntimeError('Научный ZIP поврежден')
        for name,path in REMOTE_SCIENCE.items():
            inner='science_data/'+path.name
            if z.getinfo(inner).file_size!=path.stat().st_size:raise RuntimeError('ZIP потерял часть научных данных: '+name)
        if 'ML5_V10_2_REMOTE_VALIDATION.json' not in z.namelist():raise RuntimeError('Нет протокола исходников')
    final_sha=sha256(SCIENCE_ARCHIVE)
    print('Данные из приватного GitHub прочитаны и обработаны повторно: ',len(REMOTE_SCIENCE),'таблиц')
    print('Научный архив:',SCIENCE_ARCHIVE.name,round(SCIENCE_ARCHIVE.stat().st_size/2**20,2),'МиБ, SHA256',final_sha)
    print('Полный расчётный Evidence остаётся в GitHub; на компьютер скачивается только научный архив')
    if SCIENCE_ARCHIVE.stat().st_size>CFG.SCIENTIFIC_ARCHIVE_MAX_MIB*2**20:
        print('ПРЕДУПРЕЖДЕНИЕ: научный архив больше заданного порога; данные не усекались')
    from IPython.display import display,FileLink
    display(FileLink(str(SCIENCE_ARCHIVE)))
    try:
        from google.colab import files
        files.download(str(SCIENCE_ARCHIVE))
    except Exception as exc:
        print('Автоскачивание не сработало; ссылка доступна выше:',type(exc).__name__)
else:
    print('SMOKE завершён без научной публикации. Для реальных результатов выполните CFG.MODE=FULL.')
mark_stage('Скачиваемый архив');mark_stage('Финал')
if CFG.MODE=='FULL':
    if set(TASK_STAGES)!=DONE_STAGES:raise RuntimeError('Run all завершился с пропущенными этапами: '+str(set(TASK_STAGES)-DONE_STAGES))
    print('ONE_RUN_ALL_COMPLETED: 31/31; публикация, GitHub readback, научный синтез и ZIP выполнены в одном сеансе; внешняя научная рецензия не заявлена')
MAIN_PROGRESS.close()


---
**Статус:** полный научный `FULL PASS` не присваивается автоматически. При успешном `FULL Run all` публикуются расчётные данные, они повторно считываются по GitHub API и проверяются; на компьютер скачивается научный пакет. Источниково недоступные исследования остаются `NOT_FEASIBLE`, а не подменяются суррогатами.